# Final Full-PDF Unlearning A/B Evaluation Pipeline

This notebook implements the finalized evaluation protocol for the **Unlearning in federal disaster policy** project.

It is designed to:

1. extract paragraph-level text from all five source PDFs;
2. audit extraction quality and stop rather than silently misalign gold passages;
3. match the finalized 76-row `GPT Test` to the extracted corpus;
4. remove exact duplicates, extraction-created parent/child duplicates, and revised-codebook example leakage;
5. classify every eligible passage with the same three LLMs used previously;
6. run the controlled prompt A/B design:
   - direct target passage only, without a codebook;
   - revised codebook without its `Examples` column;
   - revised codebook with its `Examples` column;
   - each codebook condition with and without the checklist;
7. evaluate default and threshold-adjusted predictions against:
   - the full extracted corpus, where genuinely unmatched passages are assumed `No`;
   - finalized **new gold**;
   - preserved **old gold**;
   - Anmol, Prerana, and Kyle labels separately;
   - pooled coder decisions;
   - every scope again after removing rows labeled by Kyle alone;
8. report accuracy, recall, F1, AUROC, coverage, and confusion counts by model, document, labeler, and their combinations;
9. explore simple probability-threshold and model-weight optimization.

> **Important methodological distinction:** “all paragraphs” uses an assumed-negative reference for passages absent from the finalized human test set. It is reported separately from human-labeled test-set metrics and is never presented as equivalent to adjudicated gold.

## Recommended execution order

Run the notebook in order. The expensive API stage is disabled by default.

1. Upload or place the finalized workbook and five clean source PDFs in one of the configured input folders.
2. Run through **Pre-API audit and hard-stop checks**.
3. Review `manual_review_queue.xlsx` and `gold_match_audit.xlsx`.
4. Resolve any hard-stop items through the override CSV or extraction configuration.
5. Set `RUN_LLM_CALLS = True`, run the provider preflight, inspect it, and only then run the complete grid.
6. Run evaluation and post-processing.

No prediction is silently changed to `No` when an API request fails. Failed, missing, or schema-invalid predictions remain missing and reduce the reported coverage.

In [1]:
# Optional dependency installation for Google Colab or a fresh environment.
# Restart the runtime only if pip explicitly asks you to.
%pip install -q -U \
    rapidfuzz \
    pymupdf pdfplumber openpyxl xlsxwriter pyarrow tqdm \
    pydantic python-dotenv \
    openai anthropic google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 

In [52]:
from __future__ import annotations

import ast
import csv
import dataclasses
import fnmatch
import hashlib
import itertools
import json
import math
import os
import random
import re
import shutil
import statistics
import sys
import time
import traceback
import unicodedata
import warnings

from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Iterator, Literal, Mapping, Optional, Sequence

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import pdfplumber

from pydantic import BaseModel, ConfigDict, ValidationError
from rapidfuzz import fuzz
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 20260814
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("PyMuPDF:", fitz.version[0])
print("pandas:", pd.__version__)

Python: 3.12.13
PyMuPDF: 1.28.2
pandas: 2.2.2


In [53]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
# -----------------------------
# Project paths and run switches
# -----------------------------

# In Colab, set PROJECT_ROOT to a folder in Drive if you want checkpoints to persist.
PROJECT_ROOT = Path(os.getenv("UNLEARNING_PROJECT_ROOT", "/content/unlearning_final_pipeline"))
if not Path("/content").exists():  # local/Jupyter fallback
    PROJECT_ROOT = Path(os.getenv("UNLEARNING_PROJECT_ROOT", "./unlearning_final_pipeline"))

INPUT_DIRS = [
    PROJECT_ROOT / "inputs",
    Path("/content"),
    Path.cwd(),
    Path("/mnt/data"),
]
if os.getenv("UNLEARNING_SEARCH_MYDRIVE", "0") == "1":
    INPUT_DIRS.append(Path("/content/drive/MyDrive"))
INPUT_DIRS = [p for p in INPUT_DIRS if p.exists()]

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
AUDIT_DIR = OUTPUT_ROOT / "audit"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
RESULTS_DIR = OUTPUT_ROOT / "results"
for directory in [PROJECT_ROOT, OUTPUT_ROOT, AUDIT_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Explicit paths always override automatic discovery.
FINALIZED_WORKBOOK_PATH: Optional[Path] = None
PDF_PATH_OVERRIDES: dict[str, Optional[Path]] = {
    "ERP Report": None,
    "GAO": None,
    "EPA": None,
    "Hurricane Katrina": None,
    "PAR": None,
}

# Stage switches
RUN_EXTRACTION_STAGE = False
RUN_LLM_CALLS = True              # Safety default: no paid calls.
RUN_PROVIDER_PREFLIGHT = True       # Used only when RUN_LLM_CALLS=True.
RUN_COMPLETE_LLM_GRID = True       # Turn on only after reviewing preflight.
REBUILD_EXTRACTION = True
REBUILD_MATCHING = True
RESUME_FROM_JSONL = True

# Optional development limits. Keep both None for final results.
ROW_LIMIT: Optional[int] = None
DOCUMENT_LIMIT: Optional[list[str]] = None

# Hard-stop policy
EXPECTED_FINAL_GOLD_ROWS = 76
REQUIRE_ALL_GOLD_MATCHED = True
REQUIRE_NO_AMBIGUOUS_GOLD_MATCHES = True
REQUIRE_NO_GOLD_EXAMPLE_LEAKAGE = True
REQUIRE_NO_RESIDUAL_EXACT_OR_PARENT_CHILD_DUPLICATES = True

# Extraction settings
HEADER_FOOTER_REPEAT_MIN_FRACTION = 0.40
HEADER_ZONE_FRACTION = 0.12
FOOTER_ZONE_FRACTION = 0.12
MIN_PROSE_WORDS = 6
MAX_PARAGRAPH_CHARS_FLAG = 5000
MIN_PAGE_TEXT_CHARS = 80
DUAL_ENGINE_PAGE_SIMILARITY_WARN = 0.82
ALLOW_OCR_FALLBACK = False  # OCR is deliberately not automatic; scanned pages are flagged.

# Matching settings
GOLD_WINDOW_MAX_PARAGRAPHS = 6
GOLD_WINDOW_MAX_CHARS = 7000
GOLD_AUTO_ACCEPT_SCORE = 0.90
GOLD_AUTO_ACCEPT_MARGIN = 0.025
GOLD_EXACT_SCORE = 1.0
GOLD_CONTAINMENT_SCORE = 0.985
GOLD_FUZZY_REVIEW_SCORE = 0.84
MAX_MATCH_CANDIDATES_PER_GOLD = 15

# Duplicate/leakage settings
NEAR_DUPLICATE_THRESHOLD = 97.0
PARENT_CHILD_MIN_SHORT_CHARS = 90
PARENT_CHILD_MIN_RATIO = 1.18
FUZZY_EXAMPLE_REVIEW_THRESHOLD = 92.0
REMOVE_CODEBOOK_EXAMPLE_EXACT_OR_CONTAINMENT = True
AUTO_REMOVE_FUZZY_EXAMPLE_MATCHES = False
DEDUPE_ACROSS_DOCUMENTS = False

# API settings
MAX_API_RETRIES = 5
MAX_SCHEMA_RETRIES = 2
BASE_RETRY_SECONDS = 2.0
REQUEST_SLEEP_SECONDS = 0.15
MAX_OUTPUT_TOKENS = 900
PREFLIGHT_ROWS_PER_CONFIGURATION = 3

# Evaluation/post-processing settings
DEFAULT_THRESHOLD = 0.50
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
ENSEMBLE_WEIGHT_STEP = 0.05
PRIMARY_OPTIMIZATION_SCOPE = "new_gold_test_only"
PRIMARY_OPTIMIZATION_METRIC = "f1"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Input search folders:")
for p in INPUT_DIRS:
    print(" -", p)
print("Outputs:", OUTPUT_ROOT)

PROJECT_ROOT: /content/unlearning_final_pipeline
Input search folders:
 - /content
 - /content
Outputs: /content/unlearning_final_pipeline/outputs


## 1. Resolve and validate inputs

The resolver strongly prefers clean source PDFs. Files whose names contain terms such as `annotated`, `model_review`, or `corrected` are rejected unless supplied through `PDF_PATH_OVERRIDES`. This prevents predictions from being contaminated by prior model annotations.

In [55]:
# -----------------------------
# File identity and discovery
# -----------------------------

WORKBOOK_ALIASES = [
    "Unlearning_Curated_GPT_Test_EPA_Reevaluation.xlsx",
    "Unlearning Curated GPT Test EPA Reevaluation.xlsx",
]

PDF_SPECS: dict[str, dict[str, Any]] = {
    "ERP Report": {
        "canonical_filename": "erpreport.pdf",
        "patterns": ["erpreport.pdf", "erp report.pdf", "*erp*report*.pdf"],
        "expected_source_values": {"erpreport.pdf"},
    },
    "GAO": {
        "canonical_filename": "gao-06-442t.pdf",
        "patterns": ["gao-06-442t.pdf", "gao.pdf", "*06-442t*.pdf"],
        "expected_source_values": {"gao-06-442t.pdf"},
    },
    "EPA": {
        "canonical_filename": "EPA 20060914-2006-p-00033.pdf",
        "patterns": [
            "epa.pdf",
            "epa 20060914-2006-p-00033.pdf",
            "epa*2006-p-00033*.pdf",
            "*2006-p-00033*.pdf",
        ],
        "expected_source_values": {"epa", "EPA"},
    },
    "Hurricane Katrina": {
        "canonical_filename": "Post-Katrina preparedness.pdf",
        "patterns": ["Post-Katrina preparedness.pdf", "hurricane katrina.pdf", "hurricane_katrina.pdf"],
        "expected_source_values": {"Post-Katrina preparedness.pdf"},
    },
    "PAR": {
        "canonical_filename": (
            "Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020  The Need for Strategic Management of.pdf"
        ),
        "patterns": [
            "par.pdf",
            "public administration review*what if hurricane katrina hit in 2020*.pdf",
            "*mcguire*strategic management*.pdf",
        ],
        "expected_source_values": {
            "Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of (1).pdf",
        },
    },
}

SUSPECT_SOURCE_TOKENS = {
    "annotated",
    "unanimous_model_review",
    "model_review",
    "review_corrected",
    "corrected",
    "highlighted",
    "_v2",
    "with comments",
}

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def iter_candidate_files(suffix: str) -> Iterator[Path]:
    seen: set[Path] = set()
    for root in INPUT_DIRS:
        if not root.exists():
            continue
        try:
            candidates = root.rglob(f"*{suffix}")
        except Exception:
            continue
        for path in candidates:
            try:
                resolved = path.resolve()
            except Exception:
                resolved = path
            if resolved not in seen and path.is_file():
                seen.add(resolved)
                yield path

def is_suspect_pdf_name(path: Path) -> bool:
    name = path.name.lower().replace("-", "_").replace(" ", "_")
    return any(token in name for token in SUSPECT_SOURCE_TOKENS)

def resolve_finalized_workbook() -> Path:
    if FINALIZED_WORKBOOK_PATH is not None:
        path = Path(FINALIZED_WORKBOOK_PATH)
        if not path.exists():
            raise FileNotFoundError(f"FINALIZED_WORKBOOK_PATH does not exist: {path}")
        return path

    candidates = list(iter_candidate_files(".xlsx"))
    for alias in WORKBOOK_ALIASES:
        exact = [p for p in candidates if p.name.lower() == alias.lower()]
        if exact:
            return sorted(exact, key=lambda p: len(str(p)))[0]

    semantic = [
        p for p in candidates
        if "curated" in p.name.lower()
        and "epa" in p.name.lower()
        and "reevaluation" in p.name.lower()
    ]
    if len(semantic) == 1:
        return semantic[0]
    if not semantic:
        raise FileNotFoundError(
            "Could not find the finalized GPT Test workbook. "
            "Set FINALIZED_WORKBOOK_PATH explicitly."
        )
    raise RuntimeError(
        "Multiple plausible finalized workbooks were found. Set FINALIZED_WORKBOOK_PATH:\n"
        + "\n".join(f" - {p}" for p in semantic)
    )

def resolve_pdf(document: str) -> Path:
    override = PDF_PATH_OVERRIDES.get(document)
    if override is not None:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f"PDF override for {document} does not exist: {path}")
        return path

    spec = PDF_SPECS[document]
    all_pdfs = list(iter_candidate_files(".pdf"))
    matches: list[Path] = []
    for path in all_pdfs:
        lower_name = path.name.lower()
        if any(fnmatch.fnmatch(lower_name, pattern.lower()) for pattern in spec["patterns"]):
            if not is_suspect_pdf_name(path):
                matches.append(path)

    # Exact canonical filename gets first priority.
    exact = [
        p for p in matches
        if p.name.lower() == spec["canonical_filename"].lower()
    ]
    if len(exact) == 1:
        return exact[0]
    if len(exact) > 1:
        exact = sorted(exact, key=lambda p: (len(str(p)), str(p).lower()))
        print(f"Warning: multiple exact copies for {document}; using {exact[0]}")
        return exact[0]

    if len(matches) == 1:
        return matches[0]
    if not matches:
        suspect_matches = []
        for path in all_pdfs:
            lower_name = path.name.lower()
            if any(fnmatch.fnmatch(lower_name, pattern.lower()) for pattern in spec["patterns"]):
                suspect_matches.append(path)
        extra = ""
        if suspect_matches:
            extra = (
                "\nOnly suspect annotated/review copies were found:\n"
                + "\n".join(f" - {p}" for p in suspect_matches)
            )
        raise FileNotFoundError(
            f"Could not find a clean source PDF for {document}. "
            f"Expected approximately: {spec['canonical_filename']}. "
            f"Set PDF_PATH_OVERRIDES[{document!r}] explicitly.{extra}"
        )

    raise RuntimeError(
        f"Multiple clean PDF candidates were found for {document}; set an override:\n"
        + "\n".join(f" - {p}" for p in sorted(matches))
    )

def validate_pdf_file(path: Path) -> dict[str, Any]:
    if path.suffix.lower() != ".pdf":
        raise ValueError(f"Not a PDF: {path}")
    doc = fitz.open(path)
    try:
        if doc.is_encrypted and not doc.authenticate(""):
            raise RuntimeError(f"Encrypted PDF cannot be opened without a password: {path}")
        page_count = doc.page_count
        char_counts = [len(page.get_text("text") or "") for page in doc]
        return {
            "path": str(path),
            "filename": path.name,
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
            "page_count": page_count,
            "total_extracted_chars_preview": int(sum(char_counts)),
            "pages_below_min_text_chars": int(sum(c < MIN_PAGE_TEXT_CHARS for c in char_counts)),
        }
    finally:
        doc.close()

def resolve_all_inputs() -> tuple[Path, dict[str, Path], pd.DataFrame]:
    workbook = resolve_finalized_workbook()
    pdf_paths = {document: resolve_pdf(document) for document in PDF_SPECS}
    manifest_rows = [{
        "artifact_type": "finalized_workbook",
        "document": "",
        "path": str(workbook),
        "filename": workbook.name,
        "sha256": sha256_file(workbook),
        "bytes": workbook.stat().st_size,
        "page_count": np.nan,
        "total_extracted_chars_preview": np.nan,
        "pages_below_min_text_chars": np.nan,
    }]
    for document, path in pdf_paths.items():
        row = validate_pdf_file(path)
        row.update({"artifact_type": "source_pdf", "document": document})
        manifest_rows.append(row)
    return workbook, pdf_paths, pd.DataFrame(manifest_rows)

# The resolution call is deferred until extraction is run, so the notebook can be
# opened and inspected before the five PDFs are uploaded.

In [56]:
# -----------------------------
# Text normalization
# -----------------------------

UNICODE_REPLACEMENTS = {
    "\u00ad": "",      # soft hyphen
    "\u200b": "",
    "\ufeff": "",
    "\u2018": "'",
    "\u2019": "'",
    "\u201c": '"',
    "\u201d": '"',
    "\u2013": "-",
    "\u2014": " - ",
    "\u2212": "-",
    "\ufb01": "fi",
    "\ufb02": "fl",
    "\u2022": " • ",
    "\u25cf": " • ",
    "\ufffd": "�",
}

COMMON_PDF_ARTIFACT_REPAIRS = [
    (re.compile(r"\bTh e\b", flags=re.I), "The"),
    (re.compile(r"\bth e\b", flags=re.I), "the"),
    (re.compile(r"\boffi ce\b", flags=re.I), "office"),
    (re.compile(r"\bdiffi cult\b", flags=re.I), "difficult"),
    (re.compile(r"\beff ect", flags=re.I), "effect"),
    (re.compile(r"\bgov ernment\b", flags=re.I), "government"),
]

def normalize_space(value: Any) -> str:
    if value is None:
        return ""
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and bool(missing):
            return ""
    except Exception:
        pass
    text = str(value).replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()

def repair_unicode(value: Any) -> str:
    text = normalize_space(value)
    text = unicodedata.normalize("NFKC", text)
    for source, target in UNICODE_REPLACEMENTS.items():
        text = text.replace(source, target)
    for pattern, target in COMMON_PDF_ARTIFACT_REPAIRS:
        text = pattern.sub(target, text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

def normalize_for_match(value: Any) -> str:
    text = repair_unicode(value).lower()
    text = text.replace("•", " ")
    text = re.sub(r"(?<=\w)-\s+(?=\w)", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9%$]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def compact_for_match(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "", normalize_for_match(value))

def normalize_yes_no(value: Any, allow_blank: bool = True) -> Optional[int]:
    if value is None:
        return None if allow_blank else 0
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and bool(missing):
            return None if allow_blank else 0
    except Exception:
        pass
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    if isinstance(value, (int, np.integer)):
        if int(value) in {0, 1}:
            return int(value)
    if isinstance(value, float) and value in {0.0, 1.0}:
        return int(value)
    text = normalize_for_match(value)
    if text in {"yes", "y", "true", "1", "unlearning", "positive"}:
        return 1
    if text in {"no", "n", "false", "0", "not unlearning", "negative"}:
        return 0
    if allow_blank and not text:
        return None
    raise ValueError(f"Cannot normalize Yes/No value: {value!r}")

def safe_join_flags(*values: Any) -> str:
    flags: list[str] = []
    for value in values:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            continue
        if isinstance(value, (list, tuple, set)):
            items = value
        else:
            items = [value]
        for item in items:
            for part in str(item).split(" | "):
                part = part.strip()
                if part and part not in flags:
                    flags.append(part)
    return " | ".join(flags)

def token_set(value: Any) -> set[str]:
    return set(normalize_for_match(value).split())

def text_similarity(a: Any, b: Any) -> float:
    aa, bb = normalize_for_match(a), normalize_for_match(b)
    if not aa or not bb:
        return 0.0
    return fuzz.ratio(aa, bb) / 100.0

def containment_similarity(a: Any, b: Any) -> float:
    aa, bb = compact_for_match(a), compact_for_match(b)
    if not aa or not bb:
        return 0.0
    short, long = sorted([aa, bb], key=len)
    if short in long:
        return 1.0
    return fuzz.partial_ratio(short, long) / 100.0

In [57]:
# -----------------------------
# Finalized workbook loading
# -----------------------------

GOLD_COLUMN_ORDER = [
    "Passage_ID",
    "Original_Number",
    "Reference",
    "Text_Content",
    "Document",
    "Source_File",
    "old_gold_unlearning",
    "old_gold_binary",
    "new_gold_unlearning",
    "new_gold_binary",
    "anmol_unlearning",
    "prerana_unlearning",
    "kyle_unlearning",
    "Original_Labeler",
    "Selected_Codes",
    "Adjudication_Rationale",
    "Latest_Label_Source",
    "Rationale_Source",
    "Review_Flag",
    "Review_Issue",
    "Gold_Change",
    "Leakage_Check",
]

CODEBOOK_COLUMN_ORDER = [
    "Code",
    "Definition",
    "Detection Logic",
    "Examples",
    "Positive Clarification",
    "Negative Clarification",
]

def detect_codebook_header(raw: pd.DataFrame) -> int:
    for idx in range(min(12, len(raw))):
        values = {normalize_for_match(v) for v in raw.iloc[idx].tolist()}
        if "code" in values and "definition" in values:
            return idx
    raise ValueError("Could not locate the Code/Definition header row in Codebook (revised).")

def load_finalized_workbook(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    sheet_names = pd.ExcelFile(path).sheet_names
    expected = {"GPT Test", "Codebook (revised)"}
    missing = expected - set(sheet_names)
    if missing:
        raise ValueError(f"Workbook is missing required sheets: {sorted(missing)}")

    gold_raw = pd.read_excel(path, sheet_name="GPT Test")
    missing_gold = [c for c in GOLD_COLUMN_ORDER if c not in gold_raw.columns]
    if missing_gold:
        raise ValueError(f"GPT Test is missing columns: {missing_gold}")
    gold = gold_raw[GOLD_COLUMN_ORDER].copy()

    if len(gold) != EXPECTED_FINAL_GOLD_ROWS:
        raise AssertionError(
            f"Expected {EXPECTED_FINAL_GOLD_ROWS} finalized gold rows, found {len(gold)}."
        )
    if gold["Passage_ID"].isna().any() or gold["Passage_ID"].duplicated().any():
        raise AssertionError("Passage_ID must be nonblank and unique.")
    if gold["Text_Content"].fillna("").str.strip().eq("").any():
        raise AssertionError("Every gold row must have Text_Content.")

    expected_documents = set(PDF_SPECS)
    actual_documents = set(gold["Document"].dropna().astype(str))
    if actual_documents != expected_documents:
        raise AssertionError(
            f"Unexpected document set. Expected {sorted(expected_documents)}, "
            f"found {sorted(actual_documents)}"
        )

    for label_col, binary_col in [
        ("old_gold_unlearning", "old_gold_binary"),
        ("new_gold_unlearning", "new_gold_binary"),
    ]:
        normalized = gold[label_col].map(lambda x: normalize_yes_no(x, allow_blank=False))
        numeric = pd.to_numeric(gold[binary_col], errors="coerce")
        bad = numeric.isna() | (normalized.astype(int) != numeric.astype(int))
        if bad.any():
            raise AssertionError(
                f"{label_col} and {binary_col} disagree on rows: "
                f"{gold.loc[bad, 'Passage_ID'].tolist()}"
            )
        gold[binary_col] = numeric.astype(int)

    for coder in ["anmol", "prerana", "kyle"]:
        source_col = f"{coder}_unlearning"
        binary_col = f"{coder}_binary"
        gold[binary_col] = gold[source_col].map(normalize_yes_no).astype("Int64")

    allowed_original_labelers = {"anmol", "prerana", "kyle"}
    original = gold["Original_Labeler"].dropna().astype(str).str.strip().str.lower()
    invalid = sorted(set(original) - allowed_original_labelers)
    if invalid:
        raise AssertionError(f"Unexpected Original_Labeler values: {invalid}")
    gold["Original_Labeler"] = (
        gold["Original_Labeler"].astype("string").str.strip().str.lower()
    )

    gold["kyle_only_row"] = (
        gold["kyle_binary"].notna()
        & gold["anmol_binary"].isna()
        & gold["prerana_binary"].isna()
    )
    gold["gold_text_norm"] = gold["Text_Content"].map(normalize_for_match)
    gold["gold_text_compact"] = gold["Text_Content"].map(compact_for_match)
    if gold["gold_text_compact"].duplicated().any():
        dup = gold.loc[
            gold["gold_text_compact"].duplicated(keep=False),
            ["Passage_ID", "Document", "Text_Content"],
        ]
        raise AssertionError(
            "The finalized workbook unexpectedly contains normalized exact duplicates:\n"
            + dup.to_string(index=False)
        )

    raw_codebook = pd.read_excel(path, sheet_name="Codebook (revised)", header=None)
    header_idx = detect_codebook_header(raw_codebook)
    codebook = raw_codebook.iloc[header_idx + 1:].copy()
    codebook.columns = [normalize_space(v) for v in raw_codebook.iloc[header_idx]]
    missing_cb = [c for c in CODEBOOK_COLUMN_ORDER if c not in codebook.columns]
    if missing_cb:
        raise ValueError(f"Codebook is missing expected columns: {missing_cb}")
    codebook = codebook[CODEBOOK_COLUMN_ORDER].copy()
    codebook = codebook.loc[
        codebook["Code"].notna() & codebook["Code"].astype(str).str.strip().ne("")
    ].reset_index(drop=True)

    return gold, codebook

def workbook_summary(gold: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for document, group in gold.groupby("Document", sort=False):
        rows.append({
            "Document": document,
            "N": len(group),
            "old_yes": int(group["old_gold_binary"].sum()),
            "old_no": int((1 - group["old_gold_binary"]).sum()),
            "new_yes": int(group["new_gold_binary"].sum()),
            "new_no": int((1 - group["new_gold_binary"]).sum()),
            "kyle_only_rows": int(group["kyle_only_row"].sum()),
            "anmol_labeled": int(group["anmol_binary"].notna().sum()),
            "prerana_labeled": int(group["prerana_binary"].notna().sum()),
            "kyle_labeled": int(group["kyle_binary"].notna().sum()),
        })
    return pd.DataFrame(rows)

# Workbook-only validation can run before the PDFs are uploaded.
WORKBOOK_PATH = resolve_finalized_workbook()
GOLD_DF, CODEBOOK_DF = load_finalized_workbook(WORKBOOK_PATH)
display(workbook_summary(GOLD_DF))
print("Workbook:", WORKBOOK_PATH)
print("Workbook SHA256:", sha256_file(WORKBOOK_PATH))
print("Gold rows:", len(GOLD_DF))
print("Codebook rows:", len(CODEBOOK_DF))

,Document,N,old_yes,old_no,new_yes,new_no,kyle_only_rows,anmol_labeled,prerana_labeled,kyle_labeled
0,ERP Report,28,22,6,22,6,0,28,0,28
1,GAO,7,6,1,6,1,4,3,0,6
2,EPA,22,13,9,5,17,4,6,14,8
3,Hurricane Katrina,11,5,6,5,6,0,9,0,9
4,PAR,8,8,0,8,0,0,7,4,0


Workbook: /content/drive/MyDrive/Unlearning_Project/Untitled folder/Unlearning_Curated_GPT_Test_EPA_Reevaluation.xlsx
Workbook SHA256: a21ef42ea3621d5f4fa706616c0fe7f64774ab3cbe6f4655f5349c8d05d39820
Gold rows: 76
Codebook rows: 14


## 2. Paragraph extraction with page-level and paragraph-level audit

The extractor uses PyMuPDF for coordinate-aware reading order and pdfplumber as an independent page-text cross-check. It does **not** silently OCR a low-text page. Such pages receive an `ocr_recommended` flag and enter the review queue.

In [58]:
# -----------------------------
# PDF line extraction
# -----------------------------

STANDALONE_PAGE_NUMBER_RE = re.compile(
    r"^(?:page\s+)?(?:[ivxlcdm]+|\d+)(?:\s+of\s+\d+)?$", re.I
)
BULLET_RE = re.compile(r"^\s*(?:[•●▪◦\-–—]|\(?[a-z0-9]{1,3}[\).])\s+")
TERMINAL_RE = re.compile(r"[.!?;:][\"')\]]*$")
REFERENCE_ENTRY_RE = re.compile(
    r"^(?:[A-Z][A-Za-z'’\-]+,\s+(?:[A-Z]\.?\s*)+|\u2014{2,}|———\.?)"
)
CONTACT_NOISE_RE = re.compile(
    r"(?:www\.|https?://|@|e-mail:|email:|fax:|tdd:|to order|"
    r"printed on recycled paper|congressional relations|public affairs|"
    r"obtaining copies|report fraud|room \d{3,}|washington,\s*d\.?c\.?)",
    re.I,
)
PUBLISHER_FOOTER_RE = re.compile(
    r"(?:downloaded from|wiley online library|terms and conditions|"
    r"creative commons license|public administration review\s*[•|·]|"
    r"\d{8,},\s*20\d{2},\s*s\d)",
    re.I,
)
CONTENTS_RE = re.compile(r"^(?:table of contents|contents)$", re.I)

def looks_like_heading_text(text: str) -> bool:
    text = repair_unicode(text)
    words = text.split()
    if not text or len(words) > 18:
        return False
    if TERMINAL_RE.search(text):
        return False
    alpha = [c for c in text if c.isalpha()]
    if not alpha:
        return False
    uppercase_ratio = sum(c.isupper() for c in alpha) / len(alpha)
    title_like = sum(w[:1].isupper() for w in words if w[:1].isalpha()) >= max(1, len(words) - 1)
    return uppercase_ratio > 0.72 or title_like

def line_signature(text: str) -> str:
    text = normalize_for_match(text)
    text = re.sub(r"\b\d+\b", "<num>", text)
    return text[:180]

def page_text_similarity(a: str, b: str) -> float:
    aa, bb = normalize_for_match(a), normalize_for_match(b)
    if not aa and not bb:
        return 1.0
    if not aa or not bb:
        return 0.0
    return fuzz.ratio(aa, bb) / 100.0

def extract_page_lines_pymupdf(page: fitz.Page, document: str, pdf_path: Path) -> list[dict[str, Any]]:
    payload = page.get_text("dict", sort=False)
    width = float(page.rect.width)
    height = float(page.rect.height)
    lines: list[dict[str, Any]] = []

    for block_index, block in enumerate(payload.get("blocks", [])):
        if block.get("type") != 0:
            continue
        for line_index, line in enumerate(block.get("lines", [])):
            spans = line.get("spans", [])
            if not spans:
                continue
            text_parts = [repair_unicode(span.get("text", "")) for span in spans]
            text = normalize_space(" ".join(part for part in text_parts if part))
            if not text:
                continue
            x0, y0, x1, y1 = map(float, line.get("bbox", block.get("bbox", (0, 0, 0, 0))))
            sizes = [float(s.get("size", 0.0)) for s in spans if s.get("size")]
            fonts = [str(s.get("font", "")) for s in spans]
            lines.append({
                "document": document,
                "source_file": pdf_path.name,
                "pdf_path": str(pdf_path),
                "pdf_page_index": page.number,
                "pdf_page_number": page.number + 1,
                "page_width": width,
                "page_height": height,
                "block_index": block_index,
                "line_index_in_block": line_index,
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "width": max(0.0, x1 - x0),
                "height": max(0.0, y1 - y0),
                "font_size_median": float(np.median(sizes)) if sizes else np.nan,
                "font_names": " | ".join(sorted(set(fonts))),
                "raw_text": text,
                "line_signature": line_signature(text),
                "top_zone": y0 <= height * HEADER_ZONE_FRACTION,
                "bottom_zone": y1 >= height * (1 - FOOTER_ZONE_FRACTION),
            })
    return lines

def extract_page_text_pdfplumber(pdf: pdfplumber.PDF, page_index: int) -> str:
    try:
        page = pdf.pages[page_index]
        return repair_unicode(page.extract_text(layout=True, x_tolerance=2, y_tolerance=3) or "")
    except Exception as exc:
        return f"__PDFPLUMBER_ERROR__ {type(exc).__name__}: {exc}"

def detect_repeated_marginal_lines(lines_df: pd.DataFrame, page_count: int) -> set[tuple[str, str]]:
    if lines_df.empty:
        return set()
    marginal = lines_df.loc[lines_df["top_zone"] | lines_df["bottom_zone"]].copy()
    marginal = marginal.loc[marginal["line_signature"].str.len().between(3, 180)]
    counts = (
        marginal.groupby(["document", "line_signature"])["pdf_page_index"]
        .nunique()
        .reset_index(name="page_frequency")
    )
    minimum = max(2, math.ceil(page_count * HEADER_FOOTER_REPEAT_MIN_FRACTION))
    repeated = counts.loc[counts["page_frequency"] >= minimum]
    return set(map(tuple, repeated[["document", "line_signature"]].itertuples(index=False, name=None)))

def detect_two_column_page(page_lines: pd.DataFrame) -> bool:
    if page_lines.empty:
        return False
    width = float(page_lines["page_width"].iloc[0])
    midpoint = width / 2
    body = page_lines.loc[
        ~(page_lines["top_zone"] | page_lines["bottom_zone"])
        & (page_lines["width"] < width * 0.72)
    ]
    if len(body) < 10:
        return False
    left = body.loc[body["x1"] <= midpoint * 1.06]
    right = body.loc[body["x0"] >= midpoint * 0.94]
    crossing = body.loc[(body["x0"] < midpoint) & (body["x1"] > midpoint)]
    return len(left) >= 4 and len(right) >= 4 and len(crossing) <= max(2, int(len(body) * 0.15))

def order_page_lines(page_lines: pd.DataFrame, two_column: bool) -> pd.DataFrame:
    if page_lines.empty:
        return page_lines.copy()
    work = page_lines.copy()
    width = float(work["page_width"].iloc[0])
    midpoint = width / 2

    if not two_column:
        return work.sort_values(["y0", "x0", "block_index", "line_index_in_block"]).reset_index(drop=True)

    work["spanning"] = (
        (work["x0"] < midpoint * 0.90) & (work["x1"] > midpoint * 1.10)
    ) | (work["width"] >= width * 0.72)
    work["column"] = np.where(
        work["spanning"],
        "span",
        np.where(work["x0"] < midpoint, "left", "right"),
    )

    spans = work.loc[work["spanning"]].sort_values("y0")
    boundaries = [-np.inf] + spans["y0"].tolist() + [np.inf]
    ordered_indices: list[int] = []

    for band_index in range(len(boundaries) - 1):
        low, high = boundaries[band_index], boundaries[band_index + 1]
        band = work.loc[
            (~work["spanning"]) & (work["y0"] >= low) & (work["y0"] < high)
        ]
        left = band.loc[band["column"] == "left"].sort_values(["y0", "x0"])
        right = band.loc[band["column"] == "right"].sort_values(["y0", "x0"])
        ordered_indices.extend(left.index.tolist())
        ordered_indices.extend(right.index.tolist())
        if band_index < len(spans):
            ordered_indices.append(int(spans.index[band_index]))

    # Safety: retain any line missed by the band logic.
    ordered_indices.extend(i for i in work.index if i not in set(ordered_indices))
    ordered = work.loc[ordered_indices].copy()
    ordered["reading_order"] = np.arange(len(ordered))
    return ordered.reset_index(drop=True)

def join_pdf_lines(line_texts: Sequence[str]) -> str:
    output = ""
    for raw in line_texts:
        text = repair_unicode(raw)
        if not text:
            continue
        if not output:
            output = text
            continue
        if re.search(r"[A-Za-z]-$", output) and re.match(r"^[a-z]", text):
            output = output[:-1] + text
        elif output.endswith(("/", "–", "—")):
            output += text
        else:
            output += " " + text
    return re.sub(r"\s+", " ", output).strip()

In [59]:
# -----------------------------
# Paragraph segmentation
# -----------------------------

def should_start_new_paragraph(
    previous: pd.Series,
    current: pd.Series,
    median_line_height: float,
) -> tuple[bool, str]:
    prev_text = repair_unicode(previous["raw_text"])
    curr_text = repair_unicode(current["raw_text"])

    if previous.get("column") != current.get("column"):
        return True, "column_change"
    if previous.get("spanning", False) or current.get("spanning", False):
        return True, "spanning_line_boundary"
    if looks_like_heading_text(prev_text) or looks_like_heading_text(curr_text):
        return True, "heading_boundary"
    if BULLET_RE.match(curr_text):
        return True, "bullet_or_numbered_item"

    vertical_gap = float(current["y0"]) - float(previous["y1"])
    if vertical_gap > max(4.0, median_line_height * 0.90):
        return True, "vertical_gap"

    indent_delta = float(current["x0"]) - float(previous["x0"])
    if indent_delta > max(12.0, median_line_height * 0.9) and TERMINAL_RE.search(prev_text):
        return True, "first_line_indent"

    if (
        TERMINAL_RE.search(prev_text)
        and re.match(r"^[A-Z“\"']", curr_text)
        and vertical_gap > max(1.5, median_line_height * 0.35)
    ):
        return True, "sentence_end_plus_gap"

    return False, ""

def paragraph_quality_flags(text: str, page_flags: str = "") -> str:
    flags: list[str] = []
    words = text.split()
    if len(words) < MIN_PROSE_WORDS:
        flags.append("short_text")
    if len(text) > MAX_PARAGRAPH_CHARS_FLAG:
        flags.append("very_long_text")
    if text and text[:1].islower():
        flags.append("starts_lowercase_possible_fragment")
    if len(words) >= MIN_PROSE_WORDS and not TERMINAL_RE.search(text):
        flags.append("no_terminal_punctuation")
    if "�" in text or re.search(r"[^\x00-\x7F]{4,}", text):
        flags.append("unicode_or_glyph_artifact")
    if re.search(r"\b(?:\w\s+){4,}\w\b", text):
        flags.append("possible_character_spacing_artifact")
    if re.search(r"\s{3,}", text):
        flags.append("unusual_whitespace")
    if page_flags:
        flags.append(page_flags)
    return safe_join_flags(flags)

def paragraph_exclusion_reason(text: str, in_reference_section: bool) -> str:
    clean = repair_unicode(text)
    reasons: list[str] = []
    if STANDALONE_PAGE_NUMBER_RE.fullmatch(clean):
        reasons.append("standalone_page_number")
    if PUBLISHER_FOOTER_RE.search(clean):
        reasons.append("publisher_or_download_footer")
    if CONTACT_NOISE_RE.search(clean) and len(clean.split()) < 70:
        reasons.append("contact_or_ordering_material")
    if in_reference_section:
        reasons.append("reference_section")
    elif REFERENCE_ENTRY_RE.match(clean) and len(clean.split()) < 80:
        reasons.append("reference_entry")
    if CONTENTS_RE.fullmatch(clean):
        reasons.append("contents_heading")
    return safe_join_flags(reasons)

def segment_page_paragraphs(
    ordered_lines: pd.DataFrame,
    page_audit_row: Mapping[str, Any],
) -> list[dict[str, Any]]:
    if ordered_lines.empty:
        return []
    work = ordered_lines.copy()
    median_height = float(work["height"].replace(0, np.nan).median())
    if not np.isfinite(median_height):
        median_height = 10.0

    groups: list[list[int]] = []
    boundary_reasons: list[str] = []
    current_group = [0]
    for idx in range(1, len(work)):
        new_para, reason = should_start_new_paragraph(
            work.iloc[idx - 1], work.iloc[idx], median_height
        )
        if new_para:
            groups.append(current_group)
            boundary_reasons.append(reason)
            current_group = [idx]
        else:
            current_group.append(idx)
    groups.append(current_group)
    boundary_reasons.append("page_end")

    paragraphs: list[dict[str, Any]] = []
    for order, (indices, boundary_reason) in enumerate(zip(groups, boundary_reasons), start=1):
        subset = work.iloc[indices]
        text = join_pdf_lines(subset["raw_text"].tolist())
        if not text:
            continue
        paragraphs.append({
            "document": subset["document"].iloc[0],
            "source_file": subset["source_file"].iloc[0],
            "pdf_path": subset["pdf_path"].iloc[0],
            "pdf_page_start": int(subset["pdf_page_number"].min()),
            "pdf_page_end": int(subset["pdf_page_number"].max()),
            "paragraph_order_on_page": order,
            "line_count": len(subset),
            "first_block_index": int(subset["block_index"].iloc[0]),
            "last_block_index": int(subset["block_index"].iloc[-1]),
            "first_line_index": int(subset["line_index_in_block"].iloc[0]),
            "last_line_index": int(subset["line_index_in_block"].iloc[-1]),
            "x0": float(subset["x0"].min()),
            "y0": float(subset["y0"].min()),
            "x1": float(subset["x1"].max()),
            "y1": float(subset["y1"].max()),
            "column": safe_join_flags(subset.get("column", pd.Series(["single"])).tolist()),
            "two_column_page": bool(page_audit_row.get("two_column_detected", False)),
            "text_raw_extracted": text,
            "segmentation_boundary_after": boundary_reason,
            "source_line_text": "\n".join(subset["raw_text"].tolist()),
            "source_line_ids": " | ".join(
                f"p{int(r.pdf_page_number)}:b{int(r.block_index)}:l{int(r.line_index_in_block)}"
                for r in subset.itertuples()
            ),
        })
    return paragraphs

def can_merge_across_pages(previous: Mapping[str, Any], current: Mapping[str, Any]) -> tuple[bool, str]:
    prev_text = repair_unicode(previous["text_raw_extracted"])
    curr_text = repair_unicode(current["text_raw_extracted"])
    if not prev_text or not curr_text:
        return False, ""
    if int(current["pdf_page_start"]) != int(previous["pdf_page_end"]) + 1:
        return False, ""
    if looks_like_heading_text(prev_text) or looks_like_heading_text(curr_text):
        return False, ""
    if BULLET_RE.match(curr_text):
        return False, ""
    if STANDALONE_PAGE_NUMBER_RE.fullmatch(curr_text):
        return False, ""

    if re.search(r"[A-Za-z]-$", prev_text) and re.match(r"^[a-z]", curr_text):
        return True, "cross_page_hyphen_continuation"
    if not TERMINAL_RE.search(prev_text) and re.match(r"^[a-z(\[\"'“]", curr_text):
        return True, "cross_page_lowercase_continuation"
    return False, ""

def merge_cross_page_fragments(paragraphs: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if paragraphs.empty:
        return paragraphs.copy(), pd.DataFrame()
    rows = paragraphs.sort_values(
        ["document", "pdf_page_start", "paragraph_order_on_page"]
    ).to_dict("records")
    output: list[dict[str, Any]] = []
    audit: list[dict[str, Any]] = []

    for row in rows:
        if output and output[-1]["document"] == row["document"]:
            merge, reason = can_merge_across_pages(output[-1], row)
            if merge:
                previous = output.pop()
                merged = previous.copy()
                before_id = previous.get("provisional_paragraph_id", "")
                current_id = row.get("provisional_paragraph_id", "")
                merged["pdf_page_end"] = row["pdf_page_end"]
                merged["text_raw_extracted"] = join_pdf_lines([
                    previous["text_raw_extracted"],
                    row["text_raw_extracted"],
                ])
                merged["source_line_text"] = (
                    previous["source_line_text"] + "\n" + row["source_line_text"]
                )
                merged["source_line_ids"] = safe_join_flags(
                    previous["source_line_ids"], row["source_line_ids"]
                )
                merged["line_count"] = int(previous["line_count"]) + int(row["line_count"])
                merged["cross_page_merge"] = True
                merged["extraction_flags"] = safe_join_flags(
                    previous.get("extraction_flags", ""),
                    row.get("extraction_flags", ""),
                    reason,
                )
                output.append(merged)
                audit.append({
                    "document": row["document"],
                    "left_provisional_id": before_id,
                    "right_provisional_id": current_id,
                    "merge_reason": reason,
                    "merged_text": merged["text_raw_extracted"],
                })
                continue
        row["cross_page_merge"] = False
        output.append(row)

    merged_df = pd.DataFrame(output)
    return merged_df, pd.DataFrame(audit)

In [60]:
# -----------------------------
# Full-document extraction
# -----------------------------

def extract_document(document: str, pdf_path: Path) -> dict[str, pd.DataFrame]:
    doc = fitz.open(pdf_path)
    plumber = pdfplumber.open(pdf_path)
    try:
        all_lines: list[dict[str, Any]] = []
        page_engine_rows: list[dict[str, Any]] = []

        for page_index in range(doc.page_count):
            page = doc[page_index]
            page_lines = extract_page_lines_pymupdf(page, document, pdf_path)
            all_lines.extend(page_lines)
            pymupdf_text = "\n".join(line["raw_text"] for line in page_lines)
            plumber_text = extract_page_text_pdfplumber(plumber, page_index)
            plumber_error = plumber_text.startswith("__PDFPLUMBER_ERROR__")
            page_engine_rows.append({
                "document": document,
                "source_file": pdf_path.name,
                "pdf_page_index": page_index,
                "pdf_page_number": page_index + 1,
                "page_width": float(page.rect.width),
                "page_height": float(page.rect.height),
                "pymupdf_chars": len(pymupdf_text),
                "pdfplumber_chars": 0 if plumber_error else len(plumber_text),
                "dual_engine_similarity": (
                    np.nan if plumber_error else page_text_similarity(pymupdf_text, plumber_text)
                ),
                "pdfplumber_error": plumber_text if plumber_error else "",
                "pymupdf_text_preview": pymupdf_text[:500],
                "pdfplumber_text_preview": "" if plumber_error else plumber_text[:500],
            })

        lines_df = pd.DataFrame(all_lines)
        page_audit = pd.DataFrame(page_engine_rows)
        repeated = detect_repeated_marginal_lines(lines_df, doc.page_count)
        if not lines_df.empty:
            lines_df["repeated_header_footer"] = [
                (doc_name, signature) in repeated
                for doc_name, signature in zip(lines_df["document"], lines_df["line_signature"])
            ]
            lines_df["standalone_page_number"] = lines_df["raw_text"].map(
                lambda x: bool(STANDALONE_PAGE_NUMBER_RE.fullmatch(repair_unicode(x)))
            )
            lines_df["line_excluded"] = (
                lines_df["repeated_header_footer"] | lines_df["standalone_page_number"]
            )
        else:
            lines_df["repeated_header_footer"] = pd.Series(dtype=bool)
            lines_df["standalone_page_number"] = pd.Series(dtype=bool)
            lines_df["line_excluded"] = pd.Series(dtype=bool)

        removed_lines = lines_df.loc[lines_df["line_excluded"]].copy()
        body_lines = lines_df.loc[~lines_df["line_excluded"]].copy()

        paragraph_rows: list[dict[str, Any]] = []
        updated_page_rows: list[dict[str, Any]] = []
        for page_row in page_engine_rows:
            page_no = page_row["pdf_page_number"]
            page_lines = body_lines.loc[body_lines["pdf_page_number"] == page_no].copy()
            two_col = detect_two_column_page(page_lines)
            ordered = order_page_lines(page_lines, two_col)
            page_flags: list[str] = []
            if page_row["pymupdf_chars"] < MIN_PAGE_TEXT_CHARS:
                page_flags.append("low_text_page_ocr_recommended")
            if (
                np.isfinite(page_row["dual_engine_similarity"])
                and page_row["dual_engine_similarity"] < DUAL_ENGINE_PAGE_SIMILARITY_WARN
            ):
                page_flags.append("dual_engine_text_disagreement")
            if two_col:
                page_flags.append("two_column_layout")
            if page_row["pdfplumber_error"]:
                page_flags.append("pdfplumber_error")

            page_row = dict(page_row)
            page_row["two_column_detected"] = two_col
            page_row["page_flags"] = safe_join_flags(page_flags)
            page_row["retained_line_count"] = len(ordered)
            page_row["removed_marginal_line_count"] = int(
                removed_lines["pdf_page_number"].eq(page_no).sum()
            )
            updated_page_rows.append(page_row)
            paragraph_rows.extend(segment_page_paragraphs(ordered, page_row))

        paragraphs = pd.DataFrame(paragraph_rows)
        page_audit = pd.DataFrame(updated_page_rows)

        if paragraphs.empty:
            return {
                "lines": lines_df,
                "removed_lines": removed_lines,
                "page_audit": page_audit,
                "paragraphs": paragraphs,
                "cross_page_merge_audit": pd.DataFrame(),
            }

        paragraphs = paragraphs.sort_values(
            ["pdf_page_start", "paragraph_order_on_page"]
        ).reset_index(drop=True)
        paragraphs["provisional_paragraph_id"] = [
            f"{document.replace(' ', '_')}-raw-{i:05d}" for i in range(1, len(paragraphs) + 1)
        ]

        page_flag_map = page_audit.set_index("pdf_page_number")["page_flags"].to_dict()
        paragraphs["extraction_flags"] = [
            paragraph_quality_flags(
                text,
                safe_join_flags(
                    page_flag_map.get(start, ""),
                    *[
                        page_flag_map.get(page, "")
                        for page in range(int(start) + 1, int(end) + 1)
                    ],
                ),
            )
            for text, start, end in zip(
                paragraphs["text_raw_extracted"],
                paragraphs["pdf_page_start"],
                paragraphs["pdf_page_end"],
            )
        ]

        paragraphs, cross_page_audit = merge_cross_page_fragments(paragraphs)
        paragraphs = paragraphs.reset_index(drop=True)
        paragraphs["base_paragraph_id"] = [
            f"{document.replace(' ', '_')}-P{i:05d}" for i in range(1, len(paragraphs) + 1)
        ]
        paragraphs["document_paragraph_order"] = np.arange(1, len(paragraphs) + 1)

        # Detect reference sections after cross-page merging.
        in_references = False
        exclusion_reasons: list[str] = []
        for text in paragraphs["text_raw_extracted"]:
            normalized = normalize_for_match(text)
            if normalized in {"references", "bibliography"}:
                in_references = True
            exclusion_reasons.append(paragraph_exclusion_reason(text, in_references))
        paragraphs["exclusion_reason"] = exclusion_reasons
        paragraphs["is_heading_like"] = paragraphs["text_raw_extracted"].map(looks_like_heading_text)
        paragraphs["word_count"] = paragraphs["text_raw_extracted"].str.split().str.len()
        paragraphs["text_norm"] = paragraphs["text_raw_extracted"].map(normalize_for_match)
        paragraphs["text_compact"] = paragraphs["text_raw_extracted"].map(compact_for_match)
        paragraphs["text_sha256"] = paragraphs["text_norm"].map(sha256_text)

        # Match eligibility is broad so short bullet fragments can participate in gold windows.
        paragraphs["match_eligible"] = (
            paragraphs["exclusion_reason"].eq("")
            & paragraphs["text_norm"].ne("")
        )
        # Analysis eligibility is narrower. A gold match later overrides the short/heading rule.
        paragraphs["analysis_eligible_pre_gold"] = (
            paragraphs["match_eligible"]
            & (paragraphs["word_count"] >= MIN_PROSE_WORDS)
            & ~paragraphs["is_heading_like"]
        )
        return {
            "lines": lines_df,
            "removed_lines": removed_lines,
            "page_audit": page_audit,
            "paragraphs": paragraphs,
            "cross_page_merge_audit": cross_page_audit,
        }
    finally:
        plumber.close()
        doc.close()

def extract_all_documents(pdf_paths: Mapping[str, Path]) -> dict[str, pd.DataFrame]:
    collections: dict[str, list[pd.DataFrame]] = defaultdict(list)
    for document, path in pdf_paths.items():
        if DOCUMENT_LIMIT and document not in DOCUMENT_LIMIT:
            continue
        print(f"Extracting {document}: {path.name}")
        extracted = extract_document(document, path)
        for key, frame in extracted.items():
            if frame is not None and not frame.empty:
                collections[key].append(frame)
    outputs: dict[str, pd.DataFrame] = {}
    for key in [
        "lines",
        "removed_lines",
        "page_audit",
        "paragraphs",
        "cross_page_merge_audit",
    ]:
        outputs[key] = (
            pd.concat(collections[key], ignore_index=True)
            if collections.get(key)
            else pd.DataFrame()
        )
    return outputs

def extraction_summary(paragraphs: pd.DataFrame, page_audit: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for document in PDF_SPECS:
        p = paragraphs.loc[paragraphs["document"].eq(document)]
        pages = page_audit.loc[page_audit["document"].eq(document)]
        rows.append({
            "document": document,
            "pages": int(len(pages)),
            "raw_paragraphs": int(len(p)),
            "match_eligible": int(p.get("match_eligible", pd.Series(dtype=bool)).sum()),
            "analysis_eligible_pre_gold": int(
                p.get("analysis_eligible_pre_gold", pd.Series(dtype=bool)).sum()
            ),
            "flagged_paragraphs": int(p.get("extraction_flags", "").astype(str).ne("").sum()),
            "excluded_noise_or_references": int(
                p.get("exclusion_reason", "").astype(str).ne("").sum()
            ),
            "low_text_pages": int(
                pages.get("page_flags", "").astype(str).str.contains(
                    "low_text_page_ocr_recommended", na=False
                ).sum()
            ),
            "dual_engine_disagreement_pages": int(
                pages.get("page_flags", "").astype(str).str.contains(
                    "dual_engine_text_disagreement", na=False
                ).sum()
            ),
            "two_column_pages": int(
                pages.get("two_column_detected", pd.Series(dtype=bool)).sum()
            ),
        })
    return pd.DataFrame(rows)

In [61]:
# Run extraction and save immediate audit artifacts.
if RUN_EXTRACTION_STAGE:
    WORKBOOK_PATH, PDF_PATHS, INPUT_MANIFEST_DF = resolve_all_inputs()
    GOLD_DF, CODEBOOK_DF = load_finalized_workbook(WORKBOOK_PATH)

    EXTRACTION = extract_all_documents(PDF_PATHS)
    LINES_DF = EXTRACTION["lines"]
    REMOVED_LINES_DF = EXTRACTION["removed_lines"]
    PAGE_AUDIT_DF = EXTRACTION["page_audit"]
    BASE_PARAGRAPHS_DF = EXTRACTION["paragraphs"]
    CROSS_PAGE_MERGE_AUDIT_DF = EXTRACTION["cross_page_merge_audit"]

    display(INPUT_MANIFEST_DF)
    display(extraction_summary(BASE_PARAGRAPHS_DF, PAGE_AUDIT_DF))

    INPUT_MANIFEST_DF.to_csv(AUDIT_DIR / "input_manifest.csv", index=False)
    PAGE_AUDIT_DF.to_csv(AUDIT_DIR / "page_extraction_audit.csv", index=False)
    BASE_PARAGRAPHS_DF.to_csv(AUDIT_DIR / "base_paragraph_extraction.csv", index=False)
    REMOVED_LINES_DF.to_csv(AUDIT_DIR / "removed_repeated_headers_footers.csv", index=False)
    CROSS_PAGE_MERGE_AUDIT_DF.to_csv(
        AUDIT_DIR / "cross_page_merge_audit.csv", index=False
    )

    if PAGE_AUDIT_DF["page_flags"].str.contains(
        "low_text_page_ocr_recommended", na=False
    ).any():
        print(
            "WARNING: one or more pages have very little extractable text. "
            "Review page_extraction_audit.csv before continuing."
        )
else:
    print("RUN_EXTRACTION_STAGE=False. Load previously exported audit CSVs before later stages.")

RUN_EXTRACTION_STAGE=False. Load previously exported audit CSVs before later stages.


## 3. Match finalized gold rows to extracted PDF text

Gold matching is intentionally stricter than ordinary fuzzy search. It supports consecutive paragraph windows because PDF layout engines may split one human-coded passage into several fragments. The assignment is one-to-one and component-disjoint: the same extracted fragment cannot silently support two different gold rows.

When matching is incomplete or ambiguous, the notebook exports ranked candidates and stops before API calls.

In [62]:
# -----------------------------
# Gold matching candidates
# -----------------------------

MANUAL_GOLD_OVERRIDE_PATH = AUDIT_DIR / "manual_gold_match_overrides.csv"

def parse_reference_pages(value: Any) -> list[int]:
    text = normalize_space(value)
    pages: list[int] = []
    for left, right in re.findall(r"(\d+)(?:\s*[-–]\s*(\d+))?", text):
        start = int(left)
        end = int(right) if right else start
        if end >= start and end - start <= 10:
            pages.extend(range(start, end + 1))
    return sorted(set(pages))

def make_gold_window_candidates(base: pd.DataFrame) -> pd.DataFrame:
    required = {
        "document", "base_paragraph_id", "document_paragraph_order",
        "pdf_page_start", "pdf_page_end", "text_raw_extracted", "match_eligible"
    }
    missing = required - set(base.columns)
    if missing:
        raise ValueError(f"Base paragraph table missing columns: {sorted(missing)}")

    rows: list[dict[str, Any]] = []
    for document, group in base.loc[base["match_eligible"]].groupby("document", sort=False):
        group = group.sort_values("document_paragraph_order").reset_index(drop=True)
        records = group.to_dict("records")
        for start in range(len(records)):
            texts: list[str] = []
            components: list[str] = []
            page_start = int(records[start]["pdf_page_start"])
            page_end = int(records[start]["pdf_page_end"])
            for end in range(start, min(len(records), start + GOLD_WINDOW_MAX_PARAGRAPHS)):
                record = records[end]
                # Windows must be consecutive in extracted document order.
                if end > start:
                    previous_order = int(records[end - 1]["document_paragraph_order"])
                    if int(record["document_paragraph_order"]) != previous_order + 1:
                        break
                page_end = max(page_end, int(record["pdf_page_end"]))
                if page_end - page_start > 2:
                    break
                texts.append(record["text_raw_extracted"])
                components.append(record["base_paragraph_id"])
                candidate_text = join_pdf_lines(texts)
                if len(candidate_text) > GOLD_WINDOW_MAX_CHARS:
                    break
                candidate_id = (
                    components[0] if len(components) == 1
                    else f"WINDOW::{components[0]}::{components[-1]}"
                )
                rows.append({
                    "candidate_id": candidate_id,
                    "document": document,
                    "component_ids": tuple(components),
                    "component_ids_text": " | ".join(components),
                    "n_components": len(components),
                    "candidate_text": candidate_text,
                    "candidate_text_norm": normalize_for_match(candidate_text),
                    "candidate_text_compact": compact_for_match(candidate_text),
                    "pdf_page_start": page_start,
                    "pdf_page_end": page_end,
                    "first_document_order": int(records[start]["document_paragraph_order"]),
                    "last_document_order": int(record["document_paragraph_order"]),
                })
    candidates = pd.DataFrame(rows)
    if candidates.empty:
        return candidates

    # Repeated text can legitimately occur in an executive summary and again in the
    # body. Keep every occurrence so page references can disambiguate them.
    candidates["candidate_length"] = candidates["candidate_text_compact"].str.len()
    candidates = candidates.sort_values(
        ["document", "first_document_order", "n_components"]
    ).reset_index(drop=True)
    candidates["candidate_text_duplicate"] = candidates.duplicated(
        ["document", "candidate_text_compact"], keep=False
    )
    return candidates

def raw_match_score(gold_text: str, candidate_text: str) -> tuple[float, str, dict[str, float]]:
    gold_norm = normalize_for_match(gold_text)
    cand_norm = normalize_for_match(candidate_text)
    gold_compact = compact_for_match(gold_text)
    cand_compact = compact_for_match(candidate_text)

    if not gold_norm or not cand_norm:
        return 0.0, "empty", {}

    if gold_norm == cand_norm or gold_compact == cand_compact:
        return GOLD_EXACT_SCORE, "exact", {
            "ratio": 1.0, "token_set": 1.0, "partial": 1.0, "containment": 1.0
        }

    short, long = sorted([gold_compact, cand_compact], key=len)
    length_ratio = len(long) / max(1, len(short))
    if (
        len(short) >= 60
        and short in long
        and length_ratio <= 2.5
    ):
        penalty = min(0.025, abs(math.log(length_ratio)) * 0.012)
        return GOLD_CONTAINMENT_SCORE - penalty, "containment", {
            "ratio": fuzz.ratio(gold_norm, cand_norm) / 100,
            "token_set": fuzz.token_set_ratio(gold_norm, cand_norm) / 100,
            "partial": fuzz.partial_ratio(gold_norm, cand_norm) / 100,
            "containment": 1.0,
        }

    ratio = fuzz.ratio(gold_norm, cand_norm) / 100
    token_sort = fuzz.token_sort_ratio(gold_norm, cand_norm) / 100
    token_set_ratio = fuzz.token_set_ratio(gold_norm, cand_norm) / 100
    partial = fuzz.partial_ratio(gold_norm, cand_norm) / 100
    jaccard = len(token_set(gold_norm) & token_set(cand_norm)) / max(
        1, len(token_set(gold_norm) | token_set(cand_norm))
    )
    score = (
        0.34 * ratio
        + 0.21 * token_sort
        + 0.20 * token_set_ratio
        + 0.15 * partial
        + 0.10 * jaccard
    )
    return float(score), "fuzzy", {
        "ratio": ratio,
        "token_sort": token_sort,
        "token_set": token_set_ratio,
        "partial": partial,
        "jaccard": jaccard,
        "containment": 0.0,
    }

def infer_document_page_offsets(
    gold: pd.DataFrame,
    candidates: pd.DataFrame,
) -> dict[str, int]:
    offsets: dict[str, int] = {}
    for document, gold_group in gold.groupby("Document", sort=False):
        cand_group = candidates.loc[candidates["document"].eq(document)]
        observed: list[int] = []
        compact_to_rows = defaultdict(list)
        for row in cand_group.itertuples():
            compact_to_rows[row.candidate_text_compact].append(row)
        for gold_row in gold_group.itertuples():
            pages = parse_reference_pages(gold_row.Reference)
            if not pages:
                continue
            exact = compact_to_rows.get(gold_row.gold_text_compact, [])
            if len(exact) == 1:
                observed.append(int(exact[0].pdf_page_start) - int(pages[0]))
        if observed:
            offsets[document] = int(round(statistics.median(observed)))
        else:
            offsets[document] = 0
    return offsets

def build_gold_candidate_scores(
    gold: pd.DataFrame,
    candidates: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, int]]:
    page_offsets = infer_document_page_offsets(gold, candidates)
    rows: list[dict[str, Any]] = []
    for gold_row in tqdm(gold.itertuples(), total=len(gold), desc="Scoring gold candidates"):
        group = candidates.loc[candidates["document"].eq(gold_row.Document)]
        reference_pages = parse_reference_pages(gold_row.Reference)
        expected_pages = [
            p + page_offsets.get(gold_row.Document, 0) for p in reference_pages
        ]

        scored: list[dict[str, Any]] = []
        for candidate in group.itertuples():
            score, match_type, components = raw_match_score(
                gold_row.Text_Content, candidate.candidate_text
            )
            page_hit = bool(expected_pages) and any(
                candidate.pdf_page_start <= page <= candidate.pdf_page_end
                for page in expected_pages
            )
            page_distance = (
                min(
                    min(abs(page - candidate.pdf_page_start), abs(page - candidate.pdf_page_end))
                    for page in expected_pages
                )
                if expected_pages else np.nan
            )
            # Page is only a soft tie-breaker because internal printed page numbers and
            # PDF page indices may differ.
            score_with_page = min(1.0, score + (0.006 if page_hit else 0.0))
            scored.append({
                "Passage_ID": gold_row.Passage_ID,
                "Document": gold_row.Document,
                "Reference": gold_row.Reference,
                "gold_text": gold_row.Text_Content,
                "candidate_id": candidate.candidate_id,
                "component_ids": candidate.component_ids,
                "component_ids_text": candidate.component_ids_text,
                "n_components": candidate.n_components,
                "candidate_text": candidate.candidate_text,
                "pdf_page_start": candidate.pdf_page_start,
                "pdf_page_end": candidate.pdf_page_end,
                "raw_score": score,
                "score": score_with_page,
                "match_type": match_type,
                "page_hit": page_hit,
                "page_distance": page_distance,
                **{f"similarity_{k}": v for k, v in components.items()},
            })
        scored.sort(
            key=lambda x: (
                x["score"],
                x["match_type"] == "exact",
                x["page_hit"],
                -x["n_components"],
            ),
            reverse=True,
        )
        for rank, row in enumerate(scored[:MAX_MATCH_CANDIDATES_PER_GOLD], start=1):
            row["candidate_rank"] = rank
            rows.append(row)
    return pd.DataFrame(rows), page_offsets

def initialize_manual_override_file() -> None:
    if MANUAL_GOLD_OVERRIDE_PATH.exists():
        return
    pd.DataFrame(columns=[
        "Passage_ID",
        "candidate_id",
        "action",  # use | leave_unmatched
        "notes",
    ]).to_csv(MANUAL_GOLD_OVERRIDE_PATH, index=False)

def read_manual_overrides(candidates: pd.DataFrame) -> pd.DataFrame:
    initialize_manual_override_file()
    overrides = pd.read_csv(MANUAL_GOLD_OVERRIDE_PATH, dtype=str).fillna("")
    if overrides.empty:
        return overrides
    valid_actions = {"use", "leave_unmatched"}
    bad_actions = sorted(set(overrides["action"]) - valid_actions)
    if bad_actions:
        raise ValueError(f"Invalid manual override actions: {bad_actions}")
    duplicate_ids = overrides["Passage_ID"].duplicated(keep=False)
    if duplicate_ids.any():
        raise ValueError(
            "Only one override row is allowed per Passage_ID:\n"
            + overrides.loc[duplicate_ids].to_string(index=False)
        )
    valid_candidates = set(candidates["candidate_id"])
    bad_candidates = overrides.loc[
        overrides["action"].eq("use") & ~overrides["candidate_id"].isin(valid_candidates)
    ]
    if not bad_candidates.empty:
        raise ValueError(
            "Manual override candidate_id values not found:\n"
            + bad_candidates.to_string(index=False)
        )
    return overrides

def select_disjoint_gold_matches(
    gold: pd.DataFrame,
    ranked: pd.DataFrame,
    candidates: pd.DataFrame,
) -> pd.DataFrame:
    overrides = read_manual_overrides(candidates)
    override_map = overrides.set_index("Passage_ID").to_dict("index") if not overrides.empty else {}
    candidate_lookup = candidates.set_index("candidate_id").to_dict("index")

    selections: dict[str, dict[str, Any]] = {}
    reserved_components: set[str] = set()

    # Apply explicit overrides first.
    for gold_row in gold.itertuples():
        override = override_map.get(gold_row.Passage_ID)
        if not override:
            continue
        if override["action"] == "leave_unmatched":
            selections[gold_row.Passage_ID] = {
                "Passage_ID": gold_row.Passage_ID,
                "selected_candidate_id": "",
                "match_status": "manual_unmatched",
                "selected_score": np.nan,
                "selected_match_type": "",
                "selected_component_ids": tuple(),
                "manual_override": True,
                "manual_notes": override.get("notes", ""),
                "component_conflict": False,
            }
            continue
        candidate = candidate_lookup[override["candidate_id"]]
        components = tuple(candidate["component_ids"])
        conflict = bool(reserved_components.intersection(components))
        selections[gold_row.Passage_ID] = {
            "Passage_ID": gold_row.Passage_ID,
            "selected_candidate_id": override["candidate_id"],
            "match_status": "manual",
            "selected_score": 1.0,
            "selected_match_type": "manual",
            "selected_component_ids": components,
            "manual_override": True,
            "manual_notes": override.get("notes", ""),
            "component_conflict": conflict,
        }
        reserved_components.update(components)

    # Prioritize strongest and most decisive rows.
    best_rows = []
    for passage_id, group in ranked.groupby("Passage_ID", sort=False):
        group = group.sort_values(["score", "candidate_rank"], ascending=[False, True])
        best = group.iloc[0]
        second_score = float(group.iloc[1]["score"]) if len(group) > 1 else 0.0
        best_rows.append({
            "Passage_ID": passage_id,
            "best_score": float(best["score"]),
            "best_type": best["match_type"],
            "margin": float(best["score"]) - second_score,
        })
    priority = pd.DataFrame(best_rows)
    type_priority = {"exact": 3, "containment": 2, "fuzzy": 1}
    priority["type_priority"] = priority["best_type"].map(type_priority).fillna(0)
    priority = priority.sort_values(
        ["type_priority", "best_score", "margin"],
        ascending=[False, False, False],
    )

    for passage_id in priority["Passage_ID"]:
        if passage_id in selections:
            continue
        group = ranked.loc[ranked["Passage_ID"].eq(passage_id)].sort_values(
            ["score", "candidate_rank"], ascending=[False, True]
        )
        chosen = None
        had_conflicting_candidate = False
        for candidate_row in group.itertuples():
            components = tuple(candidate_row.component_ids)
            if reserved_components.intersection(components):
                had_conflicting_candidate = True
                continue
            chosen = candidate_row
            break

        if chosen is None:
            selections[passage_id] = {
                "Passage_ID": passage_id,
                "selected_candidate_id": "",
                "match_status": "unmatched_component_conflict",
                "selected_score": np.nan,
                "selected_match_type": "",
                "selected_component_ids": tuple(),
                "manual_override": False,
                "manual_notes": "",
                "component_conflict": had_conflicting_candidate,
            }
            continue

        reserved_components.update(tuple(chosen.component_ids))
        second_available_scores = [
            float(r.score) for r in group.itertuples()
            if r.candidate_id != chosen.candidate_id
        ]
        second = max(second_available_scores) if second_available_scores else 0.0
        selections[passage_id] = {
            "Passage_ID": passage_id,
            "selected_candidate_id": chosen.candidate_id,
            "match_status": "auto",
            "selected_score": float(chosen.score),
            "selected_match_type": chosen.match_type,
            "selected_component_ids": tuple(chosen.component_ids),
            "manual_override": False,
            "manual_notes": "",
            "component_conflict": had_conflicting_candidate,
            "selected_margin": float(chosen.score) - second,
        }

    selected = pd.DataFrame(selections.values())
    selected = gold[["Passage_ID", "Document", "Reference", "Text_Content"]].merge(
        selected, on="Passage_ID", how="left", validate="one_to_one"
    )
    selected["selected_margin"] = pd.to_numeric(
        selected.get("selected_margin"), errors="coerce"
    )
    selected["unmatched"] = selected["selected_candidate_id"].fillna("").eq("")
    selected["ambiguous"] = (
        selected["unmatched"]
        | selected["component_conflict"].fillna(False)
        | (
            ~selected["manual_override"].fillna(False)
            & selected["selected_match_type"].ne("exact")
            & (
                selected["selected_score"].fillna(0).lt(GOLD_AUTO_ACCEPT_SCORE)
                | selected["selected_margin"].fillna(0).lt(GOLD_AUTO_ACCEPT_MARGIN)
            )
        )
    )

    selected = selected.merge(
        candidates.add_prefix("candidate_"),
        left_on="selected_candidate_id",
        right_on="candidate_candidate_id",
        how="left",
        validate="many_to_one",
    )
    return selected

def gold_match_summary(matches: pd.DataFrame) -> pd.DataFrame:
    return (
        matches.groupby(["Document", "selected_match_type"], dropna=False)
        .agg(
            n=("Passage_ID", "size"),
            unmatched=("unmatched", "sum"),
            ambiguous=("ambiguous", "sum"),
            minimum_score=("selected_score", "min"),
            median_score=("selected_score", "median"),
        )
        .reset_index()
    )

In [63]:
# -----------------------------
# Execute matching and create a review queue
# -----------------------------

def run_gold_matching(
    gold: pd.DataFrame,
    base_paragraphs: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, int]]:
    candidates = make_gold_window_candidates(base_paragraphs)
    ranked, page_offsets = build_gold_candidate_scores(gold, candidates)
    matches = select_disjoint_gold_matches(gold, ranked, candidates)
    return candidates, ranked, matches, page_offsets

def create_match_review_queue(
    matches: pd.DataFrame,
    ranked: pd.DataFrame,
) -> pd.DataFrame:
    problem_ids = set(
        matches.loc[matches["unmatched"] | matches["ambiguous"], "Passage_ID"]
    )
    if not problem_ids:
        return pd.DataFrame()
    details = ranked.loc[ranked["Passage_ID"].isin(problem_ids)].copy()
    selected_map = matches.set_index("Passage_ID")["selected_candidate_id"].to_dict()
    details["currently_selected"] = [
        candidate_id == selected_map.get(passage_id, "")
        for passage_id, candidate_id in zip(details["Passage_ID"], details["candidate_id"])
    ]
    return details.sort_values(["Passage_ID", "candidate_rank"])

if RUN_EXTRACTION_STAGE:
    GOLD_CANDIDATES_DF, GOLD_RANKED_CANDIDATES_DF, GOLD_MATCH_DF, PAGE_OFFSETS = (
        run_gold_matching(GOLD_DF, BASE_PARAGRAPHS_DF)
    )
    MATCH_REVIEW_QUEUE_DF = create_match_review_queue(
        GOLD_MATCH_DF, GOLD_RANKED_CANDIDATES_DF
    )

    display(gold_match_summary(GOLD_MATCH_DF))
    print("Inferred page offsets:", PAGE_OFFSETS)
    print("Unmatched:", int(GOLD_MATCH_DF["unmatched"].sum()))
    print("Ambiguous:", int(GOLD_MATCH_DF["ambiguous"].sum()))

    GOLD_RANKED_CANDIDATES_DF.to_csv(
        AUDIT_DIR / "gold_match_ranked_candidates.csv", index=False
    )
    GOLD_MATCH_DF.to_csv(AUDIT_DIR / "gold_match_audit.csv", index=False)
    MATCH_REVIEW_QUEUE_DF.to_csv(
        AUDIT_DIR / "gold_match_manual_review_queue.csv", index=False
    )

    # Hard-stop is intentionally after audit files are written.
    hard_stop_messages = []
    if REQUIRE_ALL_GOLD_MATCHED and GOLD_MATCH_DF["unmatched"].any():
        hard_stop_messages.append(
            f"{int(GOLD_MATCH_DF['unmatched'].sum())} gold rows are unmatched."
        )
    if REQUIRE_NO_AMBIGUOUS_GOLD_MATCHES and GOLD_MATCH_DF["ambiguous"].any():
        hard_stop_messages.append(
            f"{int(GOLD_MATCH_DF['ambiguous'].sum())} gold rows are ambiguous."
        )
    if hard_stop_messages:
        raise RuntimeError(
            "Gold-to-PDF matching did not pass the hard-stop gate. "
            + " ".join(hard_stop_messages)
            + f" Review {AUDIT_DIR / 'gold_match_manual_review_queue.csv'} and edit "
            + f"{MANUAL_GOLD_OVERRIDE_PATH} before continuing."
        )

## 4. Construct canonical passages, remove extraction duplicates, and audit codebook leakage

Gold-matched windows replace their constituent extraction fragments. Remaining passages are deduplicated within their source document. Near-duplicates are flagged for review by default rather than silently removed.

In [64]:
# -----------------------------
# Canonical passage construction
# -----------------------------

AUTO_REMOVE_NEAR_DUPLICATES = False

def aggregate_component_rows(
    component_ids: Sequence[str],
    base_lookup: Mapping[str, Mapping[str, Any]],
    candidate_text: str,
) -> dict[str, Any]:
    parts = [base_lookup[cid] for cid in component_ids]
    parts = sorted(parts, key=lambda r: int(r["document_paragraph_order"]))
    first, last = parts[0], parts[-1]
    return {
        "document": first["document"],
        "source_file": first["source_file"],
        "pdf_path": first["pdf_path"],
        "pdf_page_start": min(int(p["pdf_page_start"]) for p in parts),
        "pdf_page_end": max(int(p["pdf_page_end"]) for p in parts),
        "first_document_order": min(int(p["document_paragraph_order"]) for p in parts),
        "last_document_order": max(int(p["document_paragraph_order"]) for p in parts),
        "text_raw_extracted": repair_unicode(candidate_text),
        "source_line_text": "\n".join(str(p.get("source_line_text", "")) for p in parts),
        "source_line_ids": safe_join_flags(
            *[p.get("source_line_ids", "") for p in parts]
        ),
        "component_base_ids": tuple(component_ids),
        "component_base_ids_text": " | ".join(component_ids),
        "n_component_base_paragraphs": len(component_ids),
        "extraction_flags": safe_join_flags(
            *[p.get("extraction_flags", "") for p in parts],
            "gold_window_constructed" if len(component_ids) > 1 else "",
        ),
        "cross_page_merge": any(bool(p.get("cross_page_merge", False)) for p in parts),
        "analysis_eligible_pre_gold": all(
            bool(p.get("analysis_eligible_pre_gold", False)) for p in parts
        ),
        "match_eligible": True,
        "exclusion_reason": "",
    }

def construct_gold_aligned_corpus(
    base: pd.DataFrame,
    matches: pd.DataFrame,
    gold: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if matches["unmatched"].any() or matches["ambiguous"].any():
        raise RuntimeError("Cannot construct final corpus while gold matching is unresolved.")

    base_lookup = base.set_index("base_paragraph_id").to_dict("index")
    gold_lookup = gold.set_index("Passage_ID").to_dict("index")
    consumed: set[str] = set()
    canonical_rows: list[dict[str, Any]] = []
    construction_audit: list[dict[str, Any]] = []

    for match in matches.itertuples():
        component_ids = tuple(match.candidate_component_ids)
        overlap = consumed.intersection(component_ids)
        if overlap:
            raise RuntimeError(
                f"Gold component overlap for {match.Passage_ID}: {sorted(overlap)}"
            )
        consumed.update(component_ids)
        row = aggregate_component_rows(
            component_ids,
            base_lookup,
            match.candidate_candidate_text,
        )
        row.update({
            "matched_gold": True,
            "Passage_ID": match.Passage_ID,
            "gold_match_type": match.selected_match_type,
            "gold_match_score": float(match.selected_score),
            "gold_match_margin": float(match.selected_margin),
            "gold_candidate_id": match.selected_candidate_id,
            "canonical_source": "gold_aligned_extraction_window",
        })
        canonical_rows.append(row)
        construction_audit.append({
            "Passage_ID": match.Passage_ID,
            "document": match.Document,
            "candidate_id": match.selected_candidate_id,
            "component_base_ids": " | ".join(component_ids),
            "n_components": len(component_ids),
            "extracted_candidate_text": match.candidate_candidate_text,
            "gold_text": match.Text_Content,
            "normalized_exact": (
                compact_for_match(match.candidate_candidate_text)
                == compact_for_match(match.Text_Content)
            ),
            "match_score": match.selected_score,
        })

    for base_row in base.loc[base["match_eligible"]].to_dict("records"):
        if base_row["base_paragraph_id"] in consumed:
            continue
        canonical_rows.append({
            "document": base_row["document"],
            "source_file": base_row["source_file"],
            "pdf_path": base_row["pdf_path"],
            "pdf_page_start": int(base_row["pdf_page_start"]),
            "pdf_page_end": int(base_row["pdf_page_end"]),
            "first_document_order": int(base_row["document_paragraph_order"]),
            "last_document_order": int(base_row["document_paragraph_order"]),
            "text_raw_extracted": base_row["text_raw_extracted"],
            "source_line_text": base_row.get("source_line_text", ""),
            "source_line_ids": base_row.get("source_line_ids", ""),
            "component_base_ids": (base_row["base_paragraph_id"],),
            "component_base_ids_text": base_row["base_paragraph_id"],
            "n_component_base_paragraphs": 1,
            "extraction_flags": base_row.get("extraction_flags", ""),
            "cross_page_merge": bool(base_row.get("cross_page_merge", False)),
            "analysis_eligible_pre_gold": bool(
                base_row.get("analysis_eligible_pre_gold", False)
            ),
            "match_eligible": True,
            "exclusion_reason": base_row.get("exclusion_reason", ""),
            "matched_gold": False,
            "Passage_ID": pd.NA,
            "gold_match_type": "",
            "gold_match_score": np.nan,
            "gold_match_margin": np.nan,
            "gold_candidate_id": "",
            "canonical_source": "single_extracted_paragraph",
        })

    corpus = pd.DataFrame(canonical_rows)
    corpus["text_clean"] = corpus["text_raw_extracted"].map(repair_unicode)
    corpus["text_norm"] = corpus["text_clean"].map(normalize_for_match)
    corpus["text_compact"] = corpus["text_clean"].map(compact_for_match)
    corpus["word_count"] = corpus["text_clean"].str.split().str.len()
    corpus["text_sha256"] = corpus["text_norm"].map(sha256_text)
    corpus["analysis_eligible"] = (
        corpus["matched_gold"]
        | corpus["analysis_eligible_pre_gold"]
    )
    corpus = corpus.sort_values(
        ["document", "first_document_order", "last_document_order", "matched_gold"],
        ascending=[True, True, True, False],
    ).reset_index(drop=True)
    corpus["pre_dedup_corpus_id"] = [
        f"PRE-{i:06d}" for i in range(1, len(corpus) + 1)
    ]
    return corpus, pd.DataFrame(construction_audit)

class UnionFind:
    def __init__(self, items: Iterable[str]):
        self.parent = {item: item for item in items}
        self.rank = {item: 0 for item in items}

    def find(self, item: str) -> str:
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left: str, right: str) -> None:
        root_left, root_right = self.find(left), self.find(right)
        if root_left == root_right:
            return
        if self.rank[root_left] < self.rank[root_right]:
            root_left, root_right = root_right, root_left
        self.parent[root_right] = root_left
        if self.rank[root_left] == self.rank[root_right]:
            self.rank[root_left] += 1

def duplicate_scope_columns() -> list[str]:
    return [] if DEDUPE_ACROSS_DOCUMENTS else ["document"]

def find_exact_duplicate_pairs(corpus: pd.DataFrame) -> list[dict[str, Any]]:
    pairs: list[dict[str, Any]] = []
    scope_cols = duplicate_scope_columns()
    group_cols = scope_cols + ["text_compact"]
    eligible = corpus.loc[corpus["analysis_eligible"] & corpus["text_compact"].ne("")]
    for _, group in eligible.groupby(group_cols, dropna=False, sort=False):
        if len(group) < 2:
            continue
        ids = group["pre_dedup_corpus_id"].tolist()
        for left, right in itertools.combinations(ids, 2):
            pairs.append({
                "left_id": left,
                "right_id": right,
                "duplicate_type": "exact_normalized",
                "similarity": 100.0,
            })
    return pairs

def find_parent_child_pairs(corpus: pd.DataFrame) -> list[dict[str, Any]]:
    pairs: list[dict[str, Any]] = []
    scope_cols = duplicate_scope_columns()
    grouped = (
        [("", corpus)]
        if not scope_cols
        else corpus.groupby(scope_cols, dropna=False, sort=False)
    )
    for _, group in grouped:
        records = group.loc[
            group["analysis_eligible"] & group["text_compact"].ne("")
        ].sort_values("text_compact", key=lambda s: s.str.len()).to_dict("records")
        for i, short_row in enumerate(records):
            short = short_row["text_compact"]
            if len(short) < PARENT_CHILD_MIN_SHORT_CHARS:
                continue
            for long_row in records[i + 1:]:
                long = long_row["text_compact"]
                if len(long) < len(short) * PARENT_CHILD_MIN_RATIO:
                    continue
                if short in long:
                    pairs.append({
                        "left_id": short_row["pre_dedup_corpus_id"],
                        "right_id": long_row["pre_dedup_corpus_id"],
                        "duplicate_type": "parent_child_containment",
                        "similarity": 100.0 * len(short) / max(1, len(long)),
                        "short_id": short_row["pre_dedup_corpus_id"],
                        "long_id": long_row["pre_dedup_corpus_id"],
                    })
    return pairs

def find_near_duplicate_pairs(corpus: pd.DataFrame) -> list[dict[str, Any]]:
    pairs: list[dict[str, Any]] = []
    scope_cols = duplicate_scope_columns()
    grouped = (
        [("", corpus)]
        if not scope_cols
        else corpus.groupby(scope_cols, dropna=False, sort=False)
    )
    for _, group in grouped:
        records = group.loc[
            group["analysis_eligible"]
            & group["text_norm"].str.len().ge(PARENT_CHILD_MIN_SHORT_CHARS)
        ].to_dict("records")
        for left, right in itertools.combinations(records, 2):
            length_ratio = max(len(left["text_norm"]), len(right["text_norm"])) / max(
                1, min(len(left["text_norm"]), len(right["text_norm"]))
            )
            if length_ratio > 1.20:
                continue
            similarity = fuzz.ratio(left["text_norm"], right["text_norm"])
            if similarity >= NEAR_DUPLICATE_THRESHOLD:
                pairs.append({
                    "left_id": left["pre_dedup_corpus_id"],
                    "right_id": right["pre_dedup_corpus_id"],
                    "duplicate_type": "near_duplicate",
                    "similarity": float(similarity),
                })
    return pairs

def choose_duplicate_canonical(
    group: pd.DataFrame,
    relation_rows: pd.DataFrame,
) -> str:
    work = group.copy()
    # Gold-aligned units are never discarded in favor of an unlabeled extraction unit.
    work["gold_priority"] = work["matched_gold"].astype(int)
    # For parent-child extraction artifacts, prefer the parent/longer unit after gold.
    work["length_priority"] = work["text_compact"].str.len()
    # Prefer fewer extraction warnings and simpler component construction.
    work["flag_count"] = work["extraction_flags"].fillna("").map(
        lambda x: 0 if not x else len(str(x).split(" | "))
    )
    work = work.sort_values(
        [
            "gold_priority",
            "length_priority",
            "flag_count",
            "n_component_base_paragraphs",
            "first_document_order",
        ],
        ascending=[False, False, True, True, True],
    )
    return str(work.iloc[0]["pre_dedup_corpus_id"])

def resolve_duplicates(
    corpus: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    exact = pd.DataFrame(find_exact_duplicate_pairs(corpus))
    parent_child = pd.DataFrame(find_parent_child_pairs(corpus))
    near = pd.DataFrame(find_near_duplicate_pairs(corpus))
    all_relations = pd.concat(
        [
            exact.assign(auto_resolve=True) if not exact.empty else exact,
            parent_child.assign(auto_resolve=True) if not parent_child.empty else parent_child,
            near.assign(auto_resolve=AUTO_REMOVE_NEAR_DUPLICATES) if not near.empty else near,
        ],
        ignore_index=True,
    )

    uf = UnionFind(corpus["pre_dedup_corpus_id"])
    if not all_relations.empty:
        for relation in all_relations.loc[all_relations["auto_resolve"]].itertuples():
            uf.union(relation.left_id, relation.right_id)

    work = corpus.copy()
    work["duplicate_cluster"] = work["pre_dedup_corpus_id"].map(uf.find)
    canonical_map: dict[str, str] = {}
    audit_rows: list[dict[str, Any]] = []

    for cluster, group in work.groupby("duplicate_cluster", sort=False):
        if len(group) == 1:
            canonical_id = str(group.iloc[0]["pre_dedup_corpus_id"])
        else:
            relation_rows = all_relations.loc[
                all_relations["left_id"].isin(group["pre_dedup_corpus_id"])
                & all_relations["right_id"].isin(group["pre_dedup_corpus_id"])
            ]
            distinct_gold = group["Passage_ID"].dropna().astype(str).unique()
            if len(distinct_gold) > 1:
                raise RuntimeError(
                    "Duplicate resolution would combine distinct finalized gold rows: "
                    f"{distinct_gold.tolist()}"
                )
            canonical_id = choose_duplicate_canonical(group, relation_rows)
        for row_id in group["pre_dedup_corpus_id"]:
            canonical_map[str(row_id)] = canonical_id
            if str(row_id) != canonical_id:
                relevant = all_relations.loc[
                    (
                        all_relations["left_id"].eq(row_id)
                        & all_relations["right_id"].isin(group["pre_dedup_corpus_id"])
                    )
                    | (
                        all_relations["right_id"].eq(row_id)
                        & all_relations["left_id"].isin(group["pre_dedup_corpus_id"])
                    )
                ]
                audit_rows.append({
                    "removed_id": row_id,
                    "canonical_id": canonical_id,
                    "document": group.loc[
                        group["pre_dedup_corpus_id"].eq(row_id), "document"
                    ].iloc[0],
                    "duplicate_types": safe_join_flags(
                        relevant.get("duplicate_type", pd.Series(dtype=str)).tolist()
                    ),
                    "maximum_similarity": (
                        relevant["similarity"].max() if not relevant.empty else np.nan
                    ),
                    "removed_text": group.loc[
                        group["pre_dedup_corpus_id"].eq(row_id), "text_clean"
                    ].iloc[0],
                    "canonical_text": group.loc[
                        group["pre_dedup_corpus_id"].eq(canonical_id), "text_clean"
                    ].iloc[0],
                })

    work["canonical_pre_id"] = work["pre_dedup_corpus_id"].map(canonical_map)
    deduped = work.loc[
        work["pre_dedup_corpus_id"].eq(work["canonical_pre_id"])
    ].copy()
    deduped = deduped.sort_values(
        ["document", "first_document_order", "last_document_order"]
    ).reset_index(drop=True)
    deduped["corpus_passage_id"] = [
        f"CORPUS-{i:06d}" for i in range(1, len(deduped) + 1)
    ]

    near_review = near.copy()
    if not near_review.empty:
        near_review["manual_review_required"] = ~AUTO_REMOVE_NEAR_DUPLICATES
    return deduped, pd.DataFrame(audit_rows), near_review

def verify_no_exact_or_parent_child_duplicates(corpus: pd.DataFrame) -> None:
    check = corpus.copy()
    check["pre_dedup_corpus_id"] = check["corpus_passage_id"]
    exact_remaining = find_exact_duplicate_pairs(check)
    parent_remaining = find_parent_child_pairs(check)
    if exact_remaining or parent_remaining:
        raise AssertionError(
            f"Residual duplicates remain: exact={len(exact_remaining)}, "
            f"parent_child={len(parent_remaining)}"
        )

In [65]:
# -----------------------------
# Revised-codebook example leakage
# -----------------------------

def split_codebook_examples(codebook: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for cb_row in codebook.itertuples():
        cell = repair_unicode(getattr(cb_row, "Examples", ""))
        if not cell:
            continue
        candidates = [cell]
        candidates.extend(
            part.strip(" \n\t\"'“”")
            for part in re.split(r"\n\s*\n|(?<=\])\s*(?=\[)", cell)
            if part.strip()
        )
        # Quoted spans are useful when an attribution follows the example.
        candidates.extend(
            match.strip()
            for match in re.findall(r'[“"](.{70,}?)["”]', cell, flags=re.S)
        )
        seen: set[str] = set()
        for candidate in candidates:
            candidate = re.split(r"\n\s*[—-]\s*[A-Z]", candidate, maxsplit=1)[0].strip()
            norm = normalize_for_match(candidate)
            compact = compact_for_match(candidate)
            if len(compact) < 50 or compact in seen:
                continue
            seen.add(compact)
            rows.append({
                "code": getattr(cb_row, "Code"),
                "example_text": candidate,
                "example_norm": norm,
                "example_compact": compact,
                "example_id": f"EX-{len(rows)+1:03d}",
            })
    return pd.DataFrame(rows)

def audit_example_leakage(
    corpus: pd.DataFrame,
    examples: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    if examples.empty:
        return pd.DataFrame()
    for passage in tqdm(corpus.itertuples(), total=len(corpus), desc="Checking codebook leakage"):
        best: Optional[dict[str, Any]] = None
        for example in examples.itertuples():
            p_compact = passage.text_compact
            e_compact = example.example_compact
            if not p_compact or not e_compact:
                continue
            exact = p_compact == e_compact
            containment = (
                min(len(p_compact), len(e_compact)) >= 60
                and (p_compact in e_compact or e_compact in p_compact)
            )
            similarity = fuzz.ratio(passage.text_norm, example.example_norm)
            partial = fuzz.partial_ratio(passage.text_norm, example.example_norm)
            if exact:
                match_type = "exact"
                score = 100.0
            elif containment:
                match_type = "containment"
                score = 99.0
            else:
                match_type = "fuzzy"
                score = max(similarity, partial * 0.98)
            if best is None or score > best["leakage_score"]:
                best = {
                    "corpus_passage_id": passage.corpus_passage_id,
                    "document": passage.document,
                    "Passage_ID": passage.Passage_ID,
                    "matched_gold": passage.matched_gold,
                    "passage_text": passage.text_clean,
                    "example_id": example.example_id,
                    "code": example.code,
                    "example_text": example.example_text,
                    "leakage_match_type": match_type,
                    "leakage_score": float(score),
                    "ratio_similarity": float(similarity),
                    "partial_similarity": float(partial),
                }
        if best and (
            best["leakage_match_type"] in {"exact", "containment"}
            or best["leakage_score"] >= FUZZY_EXAMPLE_REVIEW_THRESHOLD
        ):
            best["auto_remove"] = (
                REMOVE_CODEBOOK_EXAMPLE_EXACT_OR_CONTAINMENT
                and best["leakage_match_type"] in {"exact", "containment"}
            ) or (
                AUTO_REMOVE_FUZZY_EXAMPLE_MATCHES
                and best["leakage_match_type"] == "fuzzy"
            )
            best["manual_review_required"] = (
                best["leakage_match_type"] == "fuzzy"
                and not AUTO_REMOVE_FUZZY_EXAMPLE_MATCHES
            )
            rows.append(best)
    return pd.DataFrame(rows)

def apply_example_leakage_removal(
    corpus: pd.DataFrame,
    leakage: pd.DataFrame,
) -> pd.DataFrame:
    work = corpus.copy()
    work["codebook_example_leakage"] = False
    work["leakage_match_type"] = ""
    work["leakage_score"] = np.nan
    work["leakage_example_id"] = ""
    work["leakage_code"] = ""
    work["leakage_removed"] = False
    work["leakage_manual_review"] = False
    if leakage.empty:
        return work

    lookup = leakage.set_index("corpus_passage_id").to_dict("index")
    for index, row in work.iterrows():
        item = lookup.get(row["corpus_passage_id"])
        if not item:
            continue
        work.at[index, "codebook_example_leakage"] = True
        work.at[index, "leakage_match_type"] = item["leakage_match_type"]
        work.at[index, "leakage_score"] = item["leakage_score"]
        work.at[index, "leakage_example_id"] = item["example_id"]
        work.at[index, "leakage_code"] = item["code"]
        work.at[index, "leakage_removed"] = bool(item["auto_remove"])
        work.at[index, "leakage_manual_review"] = bool(item["manual_review_required"])
    return work

In [66]:
# -----------------------------
# Run canonicalization, dedupe, and leakage audit
# -----------------------------

def attach_gold_metadata(corpus: pd.DataFrame, gold: pd.DataFrame) -> pd.DataFrame:
    gold_payload_cols = [
        "Passage_ID",
        "Original_Number",
        "Reference",
        "Text_Content",
        "old_gold_unlearning",
        "old_gold_binary",
        "new_gold_unlearning",
        "new_gold_binary",
        "anmol_unlearning",
        "anmol_binary",
        "prerana_unlearning",
        "prerana_binary",
        "kyle_unlearning",
        "kyle_binary",
        "Original_Labeler",
        "Selected_Codes",
        "Adjudication_Rationale",
        "Review_Flag",
        "Review_Issue",
        "Gold_Change",
        "kyle_only_row",
    ]
    payload = gold[gold_payload_cols].copy()
    merged = corpus.merge(payload, on="Passage_ID", how="left", validate="many_to_one")
    if merged.loc[merged["matched_gold"], "new_gold_binary"].isna().any():
        raise AssertionError("A gold-matched corpus row is missing gold metadata.")
    if merged.loc[~merged["matched_gold"], "new_gold_binary"].notna().any():
        raise AssertionError("An unmatched corpus row unexpectedly received gold metadata.")
    return merged

if RUN_EXTRACTION_STAGE:
    PRE_DEDUP_CORPUS_DF, GOLD_WINDOW_CONSTRUCTION_AUDIT_DF = (
        construct_gold_aligned_corpus(BASE_PARAGRAPHS_DF, GOLD_MATCH_DF, GOLD_DF)
    )
    DEDUPED_CORPUS_DF, DUPLICATE_REMOVAL_AUDIT_DF, NEAR_DUPLICATE_REVIEW_DF = (
        resolve_duplicates(PRE_DEDUP_CORPUS_DF)
    )
    if REQUIRE_NO_RESIDUAL_EXACT_OR_PARENT_CHILD_DUPLICATES:
        verify_no_exact_or_parent_child_duplicates(DEDUPED_CORPUS_DF)

    CODEBOOK_EXAMPLES_DF = split_codebook_examples(CODEBOOK_DF)
    EXAMPLE_LEAKAGE_AUDIT_DF = audit_example_leakage(
        DEDUPED_CORPUS_DF, CODEBOOK_EXAMPLES_DF
    )
    CORPUS_WITH_LEAKAGE_DF = apply_example_leakage_removal(
        DEDUPED_CORPUS_DF, EXAMPLE_LEAKAGE_AUDIT_DF
    )

    leaked_gold = CORPUS_WITH_LEAKAGE_DF.loc[
        CORPUS_WITH_LEAKAGE_DF["matched_gold"]
        & CORPUS_WITH_LEAKAGE_DF["leakage_removed"]
    ]
    if REQUIRE_NO_GOLD_EXAMPLE_LEAKAGE and not leaked_gold.empty:
        leaked_gold.to_csv(AUDIT_DIR / "HARD_STOP_gold_example_leakage.csv", index=False)
        raise RuntimeError(
            "A finalized gold passage matches a revised-codebook example. "
            "Review HARD_STOP_gold_example_leakage.csv."
        )

    FINAL_CORPUS_DF = CORPUS_WITH_LEAKAGE_DF.loc[
        CORPUS_WITH_LEAKAGE_DF["analysis_eligible"]
        & ~CORPUS_WITH_LEAKAGE_DF["leakage_removed"]
    ].copy()
    FINAL_CORPUS_DF = attach_gold_metadata(FINAL_CORPUS_DF, GOLD_DF)
    FINAL_CORPUS_DF["full_new_assumed_binary"] = (
        FINAL_CORPUS_DF["new_gold_binary"].fillna(0).astype(int)
    )
    FINAL_CORPUS_DF["is_unlabeled_assumed_no"] = ~FINAL_CORPUS_DF["matched_gold"]
    FINAL_CORPUS_DF["evaluation_exclude_kyle_only"] = (
        FINAL_CORPUS_DF["kyle_only_row"].fillna(False)
    )
    FINAL_CORPUS_DF["prediction_text"] = FINAL_CORPUS_DF["text_clean"]

    # Ensure every finalized gold row appears exactly once after all curation.
    final_gold_counts = FINAL_CORPUS_DF["Passage_ID"].dropna().value_counts()
    missing_final_gold = sorted(set(GOLD_DF["Passage_ID"]) - set(final_gold_counts.index))
    duplicate_final_gold = final_gold_counts.loc[final_gold_counts.ne(1)]
    if missing_final_gold or not duplicate_final_gold.empty:
        raise AssertionError(
            f"Final corpus gold mapping failed. Missing={missing_final_gold}; "
            f"nonunique={duplicate_final_gold.to_dict()}"
        )

    display(
        FINAL_CORPUS_DF.groupby("document", as_index=False).agg(
            total_passages=("corpus_passage_id", "size"),
            gold_passages=("matched_gold", "sum"),
            assumed_no_passages=("is_unlabeled_assumed_no", "sum"),
            extraction_flagged=("extraction_flags", lambda s: s.fillna("").ne("").sum()),
            fuzzy_leakage_review=("leakage_manual_review", "sum"),
        )
    )

    GOLD_WINDOW_CONSTRUCTION_AUDIT_DF.to_csv(
        AUDIT_DIR / "gold_window_construction_audit.csv", index=False
    )
    DUPLICATE_REMOVAL_AUDIT_DF.to_csv(
        AUDIT_DIR / "duplicate_removal_audit.csv", index=False
    )
    NEAR_DUPLICATE_REVIEW_DF.to_csv(
        AUDIT_DIR / "near_duplicate_manual_review.csv", index=False
    )
    CODEBOOK_EXAMPLES_DF.to_csv(AUDIT_DIR / "codebook_examples_parsed.csv", index=False)
    EXAMPLE_LEAKAGE_AUDIT_DF.to_csv(
        AUDIT_DIR / "codebook_example_leakage_audit.csv", index=False
    )
    FINAL_CORPUS_DF.to_csv(OUTPUT_ROOT / "final_full_pdf_corpus.csv", index=False)
    try:
        FINAL_CORPUS_DF.to_parquet(
            OUTPUT_ROOT / "final_full_pdf_corpus.parquet", index=False
        )
    except Exception as exc:
        print("Parquet export warning:", exc)

## 5. Pre-API review workbook

This workbook is the human gate before model calls. It combines extraction warnings, unresolved near-duplicates, fuzzy example matches, and all gold/source traceability.

In [67]:
def auto_width_worksheet(worksheet, dataframe: pd.DataFrame, max_width: int = 70) -> None:
    for column_index, column in enumerate(dataframe.columns):
        sample = dataframe[column].astype(str).replace("nan", "").head(500)
        width = min(
            max_width,
            max(len(str(column)) + 2, int(sample.map(len).max() if len(sample) else 0) + 2),
        )
        worksheet.set_column(column_index, column_index, width)
    worksheet.freeze_panes(1, 0)
    worksheet.autofilter(0, 0, max(0, len(dataframe)), max(0, len(dataframe.columns) - 1))

def build_manual_review_queue(
    final_corpus: pd.DataFrame,
    page_audit: pd.DataFrame,
    near_duplicates: pd.DataFrame,
    leakage: pd.DataFrame,
    matches: pd.DataFrame,
) -> pd.DataFrame:
    queue_rows: list[dict[str, Any]] = []

    for row in final_corpus.itertuples():
        reasons = []
        if normalize_space(row.extraction_flags):
            reasons.append("extraction_flag")
        if bool(row.leakage_manual_review):
            reasons.append("fuzzy_codebook_example_similarity")
        if bool(row.matched_gold) and normalize_space(row.Review_Issue):
            reasons.append("finalized_gold_review_note")
        if reasons:
            queue_rows.append({
                "review_type": safe_join_flags(reasons),
                "document": row.document,
                "corpus_passage_id": row.corpus_passage_id,
                "Passage_ID": row.Passage_ID,
                "pdf_page_start": row.pdf_page_start,
                "pdf_page_end": row.pdf_page_end,
                "text": row.text_clean,
                "extraction_flags": row.extraction_flags,
                "gold_review_issue": row.Review_Issue,
                "leakage_match_type": row.leakage_match_type,
                "leakage_score": row.leakage_score,
                "source_line_ids": row.source_line_ids,
                "manual_decision": "",
                "reviewer_notes": "",
            })

    if not near_duplicates.empty:
        for row in near_duplicates.itertuples():
            queue_rows.append({
                "review_type": "near_duplicate_pair",
                "document": "",
                "corpus_passage_id": f"{row.left_id} <> {row.right_id}",
                "Passage_ID": "",
                "pdf_page_start": "",
                "pdf_page_end": "",
                "text": "",
                "extraction_flags": "",
                "gold_review_issue": "",
                "leakage_match_type": "",
                "leakage_score": row.similarity,
                "source_line_ids": "",
                "manual_decision": "",
                "reviewer_notes": "",
            })

    flagged_pages = page_audit.loc[
        page_audit["page_flags"].fillna("").ne("")
    ]
    for row in flagged_pages.itertuples():
        queue_rows.append({
            "review_type": "page_level_extraction_flag",
            "document": row.document,
            "corpus_passage_id": "",
            "Passage_ID": "",
            "pdf_page_start": row.pdf_page_number,
            "pdf_page_end": row.pdf_page_number,
            "text": row.pymupdf_text_preview,
            "extraction_flags": row.page_flags,
            "gold_review_issue": "",
            "leakage_match_type": "",
            "leakage_score": "",
            "source_line_ids": "",
            "manual_decision": "",
            "reviewer_notes": "",
        })

    return pd.DataFrame(queue_rows).drop_duplicates().reset_index(drop=True)

def export_pre_api_audit_workbook() -> Path:
    path = AUDIT_DIR / "manual_review_queue.xlsx"
    review_queue = build_manual_review_queue(
        FINAL_CORPUS_DF,
        PAGE_AUDIT_DF,
        NEAR_DUPLICATE_REVIEW_DF,
        EXAMPLE_LEAKAGE_AUDIT_DF,
        GOLD_MATCH_DF,
    )
    sheets = {
        "Manual_Review_Queue": review_queue,
        "Final_Corpus": FINAL_CORPUS_DF,
        "Gold_Match_Audit": GOLD_MATCH_DF,
        "Gold_Candidate_Top": GOLD_RANKED_CANDIDATES_DF,
        "Page_Extraction_Audit": PAGE_AUDIT_DF,
        "Base_Paragraphs": BASE_PARAGRAPHS_DF,
        "Duplicate_Removals": DUPLICATE_REMOVAL_AUDIT_DF,
        "Near_Duplicate_Review": NEAR_DUPLICATE_REVIEW_DF,
        "Example_Leakage": EXAMPLE_LEAKAGE_AUDIT_DF,
        "Removed_Marginal_Lines": REMOVED_LINES_DF,
        "Cross_Page_Merges": CROSS_PAGE_MERGE_AUDIT_DF,
        "Input_Manifest": INPUT_MANIFEST_DF,
    }
    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        for name, dataframe in sheets.items():
            safe_name = re.sub(r"[\[\]:*?/\\]", "_", name)[:31]
            frame = dataframe if dataframe is not None else pd.DataFrame()
            frame.to_excel(writer, index=False, sheet_name=safe_name)
            auto_width_worksheet(writer.sheets[safe_name], frame)
    print("Saved:", path)
    return path

if RUN_EXTRACTION_STAGE:
    PRE_API_AUDIT_WORKBOOK = export_pre_api_audit_workbook()

## 6. Controlled prompt variants

The five variants form a direct baseline plus a 2×2 factorial test of:

- codebook `Examples` absent/present;
- checklist absent/present.

The target passage, model, output schema, reasoning setting, and all other instructions stay fixed.

In [68]:
# -----------------------------
# Prompt construction
# -----------------------------

TARGET_VALUES = [
    "leadership",
    "laws_plans_policies",
    "capabilities",
    "funds_resources",
    "misc_organizational",
    "none",
]

PROMPT_VARIANTS = pd.DataFrame([
    {
        "prompt_variant": "direct_target_only",
        "uses_codebook": False,
        "includes_examples": False,
        "includes_checklist": False,
        "description": "Direct target-passage classification baseline with no codebook.",
    },
    {
        "prompt_variant": "codebook_no_examples_no_checklist",
        "uses_codebook": True,
        "includes_examples": False,
        "includes_checklist": False,
        "description": "Revised codebook excluding Examples; no checklist.",
    },
    {
        "prompt_variant": "codebook_with_examples_no_checklist",
        "uses_codebook": True,
        "includes_examples": True,
        "includes_checklist": False,
        "description": "Revised codebook including Examples; no checklist.",
    },
    {
        "prompt_variant": "codebook_no_examples_with_checklist",
        "uses_codebook": True,
        "includes_examples": False,
        "includes_checklist": True,
        "description": "Revised codebook excluding Examples; checklist included.",
    },
    {
        "prompt_variant": "codebook_with_examples_with_checklist",
        "uses_codebook": True,
        "includes_examples": True,
        "includes_checklist": True,
        "description": "Revised codebook including Examples; checklist included.",
    },
])

SYSTEM_PROMPT = """
You are a careful research annotator studying organizational unlearning in U.S. government
disaster policy. Classify only the supplied target passage. Do not use outside knowledge,
surrounding paragraphs, document title, gold labels, annotator rationales, or earlier model
outputs. Return a calibrated probability that the passage itself meets the active instruction.
Use the requested JSON schema exactly.
""".strip()

DIRECT_BASELINE_INSTRUCTION = """
Decide whether this passage demonstrates organizational unlearning. Use the ordinary research
meaning of unlearning: an institution recognizes that an established assumption, policy,
practice, routine, or technical system is inadequate and deliberately moves away from it,
rather than merely learning something new or making an additive improvement.
""".strip()

CHECKLIST_TEXT = """
Apply this checklist in order:
1. PRIOR ITEM: Does the passage identify an existing assumption, policy, plan, practice,
   routine, governance arrangement, or technical system?
2. INADEQUACY: Does it present that prior item as obsolete, harmful, failed, or misaligned?
3. SUBTRACTION / DISCONTINUITY: Does it call for abandoning, replacing, dismantling, or
   fundamentally rethinking the prior item?
4. EXCLUSIONS: Do not count problem diagnosis alone, ordinary implementation, routine
   updating, additional resources/capacity, a new tool that leaves the underlying logic
   unchanged, or generic "lessons learned" language.
5. TARGET: If and only if Unlearning=Yes, assign the closest target category. Otherwise use
   target_category="none".
A Yes classification normally requires steps 1-3 and must not be explained only by an item
in step 4.
""".strip()

OUTPUT_INSTRUCTION = """
Return:
- unlearning_probability: a number from 0 through 1 representing P(Unlearning=Yes);
- unlearning_label: "Yes" or "No";
- target_category: one of leadership, laws_plans_policies, capabilities,
  funds_resources, misc_organizational, none;
- evidence_quote: the shortest exact quote from the target passage that supports the label,
  or an empty string when no exact evidentiary phrase exists;
- rationale: a concise explanation grounded only in the target passage.

The binary label should ordinarily be Yes when probability is at least 0.50 and No otherwise.
Do not wrap JSON in Markdown.
""".strip()

def render_codebook(codebook: pd.DataFrame, include_examples: bool) -> str:
    columns = [
        "Code",
        "Definition",
        "Detection Logic",
        "Positive Clarification",
        "Negative Clarification",
    ]
    if include_examples:
        columns.insert(3, "Examples")
    sections: list[str] = []
    for _, row in codebook.iterrows():
        code_name = repair_unicode(row.get("Code", ""))
        if not code_name:
            continue
        parts = [f"CODE: {code_name}"]
        for column in columns[1:]:
            value = repair_unicode(row.get(column, ""))
            if value:
                parts.append(f"{column.upper()}:\n{value}")
        sections.append("\n".join(parts))
    return "\n\n---\n\n".join(sections)

def prompt_variant_record(prompt_variant: str) -> dict[str, Any]:
    match = PROMPT_VARIANTS.loc[
        PROMPT_VARIANTS["prompt_variant"].eq(prompt_variant)
    ]
    if len(match) != 1:
        raise KeyError(f"Unknown prompt variant: {prompt_variant}")
    return match.iloc[0].to_dict()

def build_prompt(
    passage_text: str,
    prompt_variant: str,
    codebook: pd.DataFrame = CODEBOOK_DF,
) -> tuple[str, str]:
    variant = prompt_variant_record(prompt_variant)
    sections: list[str] = []
    if variant["uses_codebook"]:
        sections.append(
            "Use the following revised codebook as the governing definition:\n\n"
            + render_codebook(codebook, bool(variant["includes_examples"]))
        )
    else:
        sections.append(DIRECT_BASELINE_INSTRUCTION)
    if variant["includes_checklist"]:
        sections.append(CHECKLIST_TEXT)
    sections.append(OUTPUT_INSTRUCTION)
    sections.append(
        "TARGET PASSAGE START\n"
        + repair_unicode(passage_text)
        + "\nTARGET PASSAGE END"
    )
    return SYSTEM_PROMPT, "\n\n".join(sections)

def build_prompt_manifest(
    corpus: pd.DataFrame,
    variants: pd.DataFrame = PROMPT_VARIANTS,
) -> pd.DataFrame:
    sample_text = corpus.iloc[0]["prediction_text"]
    rows = []
    for variant in variants["prompt_variant"]:
        system, user = build_prompt(sample_text, variant)
        metadata = prompt_variant_record(variant)
        rows.append({
            **metadata,
            "system_prompt_sha256": sha256_text(system),
            "user_template_sha256": sha256_text(
                user.replace(repair_unicode(sample_text), "{{TARGET_PASSAGE}}")
            ),
            "system_prompt_preview": system[:500],
            "user_prompt_preview": user[:1500],
        })
    return pd.DataFrame(rows)

def assert_prompt_factor_parity(corpus: pd.DataFrame) -> None:
    sample = corpus.iloc[0]["prediction_text"]
    rendered_without_examples = render_codebook(CODEBOOK_DF, False)
    rendered_with_examples = render_codebook(CODEBOOK_DF, True)
    if "EXAMPLES:" in rendered_without_examples:
        raise AssertionError("Examples leaked into the no-examples codebook rendering.")
    if "EXAMPLES:" not in rendered_with_examples:
        raise AssertionError("Examples are absent from the with-examples rendering.")

    for variant in PROMPT_VARIANTS.itertuples():
        _, user = build_prompt(sample, variant.prompt_variant)
        if user.count("TARGET PASSAGE START") != 1 or user.count("TARGET PASSAGE END") != 1:
            raise AssertionError(f"Target passage boundary error in {variant.prompt_variant}")
        if bool(variant.includes_checklist) != ("Apply this checklist in order:" in user):
            raise AssertionError(f"Checklist factor mismatch in {variant.prompt_variant}")
        if bool(variant.uses_codebook) != ("CODE:" in user):
            raise AssertionError(f"Codebook factor mismatch in {variant.prompt_variant}")

if RUN_EXTRACTION_STAGE:
    assert_prompt_factor_parity(FINAL_CORPUS_DF)
    PROMPT_MANIFEST_DF = build_prompt_manifest(FINAL_CORPUS_DF)
    display(PROMPT_MANIFEST_DF[[
        "prompt_variant", "uses_codebook", "includes_examples",
        "includes_checklist", "user_template_sha256"
    ]])
    PROMPT_MANIFEST_DF.to_csv(AUDIT_DIR / "prompt_manifest.csv", index=False)

In [69]:
# -----------------------------
# Flat structured-output schema
# -----------------------------

class LLMClassification(BaseModel):
    model_config = ConfigDict(extra="forbid")

    unlearning_probability: float
    unlearning_label: Literal["Yes", "No"]
    target_category: Literal[
        "leadership",
        "laws_plans_policies",
        "capabilities",
        "funds_resources",
        "misc_organizational",
        "none",
    ]
    evidence_quote: str
    rationale: str

UNSUPPORTED_SCHEMA_KEYS = {
    "minimum",
    "maximum",
    "exclusiveMinimum",
    "exclusiveMaximum",
    "multipleOf",
    "default",
    "examples",
    "$comment",
}

def sanitize_json_schema(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            key: sanitize_json_schema(child)
            for key, child in value.items()
            if key not in UNSUPPORTED_SCHEMA_KEYS
        }
    if isinstance(value, list):
        return [sanitize_json_schema(child) for child in value]
    return value

RAW_OUTPUT_SCHEMA = LLMClassification.model_json_schema()
OUTPUT_SCHEMA = sanitize_json_schema(RAW_OUTPUT_SCHEMA)
OUTPUT_SCHEMA_SHA256 = sha256_text(json.dumps(OUTPUT_SCHEMA, sort_keys=True))

def extract_json_object(raw: Any) -> dict[str, Any]:
    if isinstance(raw, BaseModel):
        return raw.model_dump()
    if isinstance(raw, Mapping):
        return dict(raw)
    text = normalize_space(raw)
    if not text:
        raise ValueError("Empty model output.")
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        parsed = json.loads(text[start:end + 1])
        if isinstance(parsed, dict):
            return parsed
    raise ValueError("No valid JSON object found in model output.")

def normalize_probability(value: Any) -> float:
    if isinstance(value, Mapping):
        for key in ["yes", "Yes", "unlearning", "positive"]:
            if key in value:
                value = value[key]
                break
    if isinstance(value, str):
        stripped = value.strip()
        percent = stripped.endswith("%")
        stripped = stripped.rstrip("%").strip()
        number = float(stripped)
        if percent:
            number /= 100.0
    else:
        number = float(value)
    if 1 < number <= 100:
        number /= 100.0
    if not (0.0 <= number <= 1.0):
        raise ValueError(f"Probability outside [0,1]: {value!r}")
    return float(number)

def normalize_model_label(value: Any) -> str:
    binary = normalize_yes_no(value, allow_blank=False)
    return "Yes" if binary == 1 else "No"

def normalize_target(value: Any) -> str:
    if isinstance(value, (list, tuple)):
        value = value[0] if value else "none"
    text = normalize_for_match(value).replace(" ", "_")
    mapping = {
        "leadership": "leadership",
        "laws_plans_policies": "laws_plans_policies",
        "laws_plans_and_policies": "laws_plans_policies",
        "laws_and_policies": "laws_plans_policies",
        "plans_and_policies": "laws_plans_policies",
        "capability": "capabilities",
        "capabilities": "capabilities",
        "funds_resources": "funds_resources",
        "funds_and_resources": "funds_resources",
        "resources": "funds_resources",
        "miscellaneous_organizational": "misc_organizational",
        "misc_organizational": "misc_organizational",
        "organizational": "misc_organizational",
        "none": "none",
        "no_target": "none",
        "not_applicable": "none",
        "n_a": "none",
    }
    if text not in mapping:
        raise ValueError(f"Unrecognized target category: {value!r}")
    return mapping[text]

def parse_and_validate_model_output(
    raw: Any,
    passage_text: str,
) -> tuple[LLMClassification, dict[str, Any]]:
    data = extract_json_object(raw)
    repair_flags: list[str] = []

    # Tolerate known provider-specific key drift without accepting arbitrary structures.
    aliases = {
        "probability": "unlearning_probability",
        "unlearning_prob": "unlearning_probability",
        "prob_unlearning": "unlearning_probability",
        "predicted_probability": "unlearning_probability",
        "label": "unlearning_label",
        "prediction": "unlearning_label",
        "predicted_label": "unlearning_label",
        "target": "target_category",
        "unlearning_target": "target_category",
        "evidence": "evidence_quote",
        "quote": "evidence_quote",
        "reason": "rationale",
        "explanation": "rationale",
    }
    for source, destination in aliases.items():
        if destination not in data and source in data:
            data[destination] = data[source]
            repair_flags.append(f"aliased_{source}_to_{destination}")

    if "unlearning_probability" not in data and "unlearning_probabilities" in data:
        data["unlearning_probability"] = data["unlearning_probabilities"]
        repair_flags.append("extracted_flat_probability_from_plural_field")

    required = {
        "unlearning_probability",
        "unlearning_label",
        "target_category",
        "evidence_quote",
        "rationale",
    }
    missing = required - set(data)
    if missing:
        raise ValueError(f"Missing required output fields: {sorted(missing)}")

    normalized = {
        "unlearning_probability": normalize_probability(data["unlearning_probability"]),
        "unlearning_label": normalize_model_label(data["unlearning_label"]),
        "target_category": normalize_target(data["target_category"]),
        "evidence_quote": normalize_space(data["evidence_quote"]),
        "rationale": normalize_space(data["rationale"]),
    }
    parsed = LLMClassification.model_validate(normalized)

    label_probability_inconsistent = (
        (parsed.unlearning_probability >= DEFAULT_THRESHOLD)
        != (parsed.unlearning_label == "Yes")
    )
    target_label_inconsistent = (
        (parsed.unlearning_label == "No" and parsed.target_category != "none")
        or (parsed.unlearning_label == "Yes" and parsed.target_category == "none")
    )
    evidence_valid = (
        not parsed.evidence_quote
        or normalize_for_match(parsed.evidence_quote) in normalize_for_match(passage_text)
    )
    diagnostics = {
        "parser_repair_flags": safe_join_flags(repair_flags),
        "label_probability_inconsistent": label_probability_inconsistent,
        "target_label_inconsistent": target_label_inconsistent,
        "evidence_quote_valid": evidence_valid,
        "schema_valid": True,
    }
    return parsed, diagnostics

print(json.dumps(OUTPUT_SCHEMA, indent=2)[:2500])
print("Schema SHA256:", OUTPUT_SCHEMA_SHA256)

{
  "additionalProperties": false,
  "properties": {
    "unlearning_probability": {
      "title": "Unlearning Probability",
      "type": "number"
    },
    "unlearning_label": {
      "enum": [
        "Yes",
        "No"
      ],
      "title": "Unlearning Label",
      "type": "string"
    },
    "target_category": {
      "enum": [
        "leadership",
        "laws_plans_policies",
        "capabilities",
        "funds_resources",
        "misc_organizational",
        "none"
      ],
      "title": "Target Category",
      "type": "string"
    },
    "evidence_quote": {
      "title": "Evidence Quote",
      "type": "string"
    },
    "rationale": {
      "title": "Rationale",
      "type": "string"
    }
  },
  "required": [
    "unlearning_probability",
    "unlearning_label",
    "target_category",
    "evidence_quote",
    "rationale"
  ],
  "title": "LLMClassification",
  "type": "object"
}
Schema SHA256: c6a0988f6c06b684247465283499fa8a8708054a5da24142893d410ec3b07d4f

## 7. Provider-isolated structured prediction

The provider layer uses a flat schema and separate adapters for OpenAI, Anthropic, and Google. This directly addresses prior failures:

- no unsupported numeric `minimum`/`maximum` constraints are sent to Anthropic;
- Gemini receives a flat object rather than a nested probability object;
- malformed JSON is logged and retried, never converted to a negative prediction;
- each model/prompt configuration has its own checkpoint and circuit breaker;
- no `temperature` parameter is sent to the current reasoning models.

API keys are read from environment variables (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GEMINI_API_KEY`), a local `.env` file, or matching Google Colab Secrets. They are never written to checkpoints or result files.


In [70]:
# -----------------------------
# Model configurations
# -----------------------------

MODEL_CONFIGS = [
    {
        "run_name": "openai_gpt_5_6_terra_low",
        "provider": "openai",
        "model": os.getenv("OPENAI_MODEL", "gpt-5.6-terra"),
        "reasoning_effort": "low",
        "enabled": True,
        "api_key_env": "OPENAI_API_KEY",
    },
    {
        "run_name": "anthropic_claude_haiku_4_5",
        "provider": "anthropic",
        "model": os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001"),
        "reasoning_effort": "default",
        "enabled": True,
        "api_key_env": "ANTHROPIC_API_KEY",
    },
    {
        "run_name": "google_gemini_3_1_flash_lite_low",
        "provider": "google",
        "model": os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite"),
        "reasoning_effort": "low",
        "enabled": True,
        "api_key_env": "GEMINI_API_KEY",
    },
]

def load_api_keys_from_supported_secret_stores() -> None:
    """Load keys without printing or embedding them in notebook outputs."""
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass

    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return

    for env_name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]:
        if os.getenv(env_name, "").strip():
            continue
        try:
            secret = userdata.get(env_name)
        except Exception:
            secret = None
        if secret:
            os.environ[env_name] = str(secret)

load_api_keys_from_supported_secret_stores()

MODEL_CONFIG_DF = pd.DataFrame(MODEL_CONFIGS)
display(MODEL_CONFIG_DF)

,run_name,provider,model,reasoning_effort,enabled,api_key_env
0,openai_gpt_5_6_terra_low,openai,gpt-5.6-terra,low,True,OPENAI_API_KEY
1,anthropic_claude_haiku_4_5,anthropic,claude-haiku-4-5-20251001,default,True,ANTHROPIC_API_KEY
2,google_gemini_3_1_flash_lite_low,google,gemini-3.1-flash-lite,low,True,GEMINI_API_KEY


In [71]:
# -----------------------------
# Provider adapters
# -----------------------------

@dataclass
class ProviderResponse:
    raw_text: str
    parsed_object: Optional[dict[str, Any]]
    response_id: str
    resolved_model: str
    finish_reason: str
    input_tokens: Optional[int]
    output_tokens: Optional[int]
    total_tokens: Optional[int]
    latency_seconds: float
    provider_metadata_json: str

_CLIENT_CACHE: dict[str, Any] = {}

def require_api_key(config: Mapping[str, Any]) -> str:
    env_name = str(config["api_key_env"])
    value = os.getenv(env_name, "").strip()
    if not value:
        raise RuntimeError(
            f"Missing API key environment variable {env_name}. "
            "Set it before enabling RUN_LLM_CALLS."
        )
    return value

def get_openai_client(config: Mapping[str, Any]):
    cache_key = "openai"
    if cache_key not in _CLIENT_CACHE:
        from openai import OpenAI
        _CLIENT_CACHE[cache_key] = OpenAI(api_key=require_api_key(config))
    return _CLIENT_CACHE[cache_key]

def get_anthropic_client(config: Mapping[str, Any]):
    cache_key = "anthropic"
    if cache_key not in _CLIENT_CACHE:
        import anthropic
        _CLIENT_CACHE[cache_key] = anthropic.Anthropic(
            api_key=require_api_key(config)
        )
    return _CLIENT_CACHE[cache_key]

def get_google_client(config: Mapping[str, Any]):
    cache_key = "google"
    if cache_key not in _CLIENT_CACHE:
        from google import genai
        _CLIENT_CACHE[cache_key] = genai.Client(
            api_key=require_api_key(config)
        )
    return _CLIENT_CACHE[cache_key]

def call_openai(
    config: Mapping[str, Any],
    system_prompt: str,
    user_prompt: str,
) -> ProviderResponse:
    client = get_openai_client(config)
    started = time.perf_counter()

    # Official Python structured-output helper for the Responses API.
    try:
        response = client.responses.parse(
            model=config["model"],
            reasoning={"effort": config.get("reasoning_effort", "low")},
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            text_format=LLMClassification,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        )
        parsed = getattr(response, "output_parsed", None)
        parsed_object = (
            parsed.model_dump() if isinstance(parsed, BaseModel)
            else dict(parsed) if isinstance(parsed, Mapping)
            else None
        )
    except (AttributeError, TypeError):
        # Compatibility fallback for SDKs that expose only responses.create.
        response = client.responses.create(
            model=config["model"],
            reasoning={"effort": config.get("reasoning_effort", "low")},
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            text={
                "format": {
                    "type": "json_schema",
                    "name": "unlearning_classification",
                    "strict": True,
                    "schema": OUTPUT_SCHEMA,
                }
            },
            max_output_tokens=MAX_OUTPUT_TOKENS,
        )
        parsed_object = None

    latency = time.perf_counter() - started
    raw_text = normalize_space(getattr(response, "output_text", ""))
    if not raw_text and parsed_object is not None:
        raw_text = json.dumps(parsed_object, ensure_ascii=False)

    refusal_texts: list[str] = []
    for output in getattr(response, "output", []) or []:
        for item in getattr(output, "content", []) or []:
            if getattr(item, "type", "") == "refusal":
                refusal_texts.append(normalize_space(getattr(item, "refusal", "")))
    if refusal_texts:
        raise RuntimeError("Provider refusal: " + " | ".join(refusal_texts))

    usage = getattr(response, "usage", None)
    return ProviderResponse(
        raw_text=raw_text,
        parsed_object=parsed_object,
        response_id=str(getattr(response, "id", "")),
        resolved_model=str(getattr(response, "model", config["model"])),
        finish_reason=safe_join_flags(
            *[getattr(item, "status", "") for item in getattr(response, "output", []) or []]
        ),
        input_tokens=getattr(usage, "input_tokens", None),
        output_tokens=getattr(usage, "output_tokens", None),
        total_tokens=getattr(usage, "total_tokens", None),
        latency_seconds=latency,
        provider_metadata_json=json.dumps({
            "status": getattr(response, "status", None),
            "incomplete_details": str(getattr(response, "incomplete_details", "")),
        }),
    )

def call_anthropic(
    config: Mapping[str, Any],
    system_prompt: str,
    user_prompt: str,
) -> ProviderResponse:
    client = get_anthropic_client(config)
    started = time.perf_counter()
    response = client.messages.create(
        model=config["model"],
        max_tokens=MAX_OUTPUT_TOKENS,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
        output_config={
            "format": {
                "type": "json_schema",
                "schema": OUTPUT_SCHEMA,
            }
        },
    )
    latency = time.perf_counter() - started
    text_blocks = [
        normalize_space(getattr(block, "text", ""))
        for block in response.content
        if getattr(block, "type", "") == "text"
    ]
    raw_text = "\n".join(text for text in text_blocks if text)
    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", None)
    output_tokens = getattr(usage, "output_tokens", None)
    return ProviderResponse(
        raw_text=raw_text,
        parsed_object=None,
        response_id=str(
            getattr(response, "id", "")
            or getattr(response, "_request_id", "")
        ),
        resolved_model=str(getattr(response, "model", config["model"])),
        finish_reason=str(getattr(response, "stop_reason", "")),
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=(
            input_tokens + output_tokens
            if input_tokens is not None and output_tokens is not None
            else None
        ),
        latency_seconds=latency,
        provider_metadata_json=json.dumps({
            "stop_sequence": getattr(response, "stop_sequence", None),
        }),
    )

def call_google(
    config: Mapping[str, Any],
    system_prompt: str,
    user_prompt: str,
) -> ProviderResponse:
    client = get_google_client(config)
    from google.genai import types

    started = time.perf_counter()
    try:
        generation_config = types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            response_mime_type="application/json",
            response_json_schema=OUTPUT_SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="low"),
        )
        response = client.models.generate_content(
            model=config["model"],
            contents=user_prompt,
            config=generation_config,
        )
    except (TypeError, AttributeError):
        # SDK compatibility fallback. Do not set temperature for Gemini 3.
        response = client.models.generate_content(
            model=config["model"],
            contents=user_prompt,
            config={
                "system_instruction": system_prompt,
                "max_output_tokens": MAX_OUTPUT_TOKENS,
                "response_mime_type": "application/json",
                "response_json_schema": OUTPUT_SCHEMA,
                "thinking_config": {"thinking_level": "low"},
            },
        )
    latency = time.perf_counter() - started

    raw_text = normalize_space(getattr(response, "text", ""))
    usage = getattr(response, "usage_metadata", None)
    finish_reasons = []
    for candidate in getattr(response, "candidates", []) or []:
        finish_reasons.append(str(getattr(candidate, "finish_reason", "")))
    return ProviderResponse(
        raw_text=raw_text,
        parsed_object=None,
        response_id=str(
            getattr(response, "response_id", "")
            or getattr(response, "id", "")
        ),
        resolved_model=config["model"],
        finish_reason=safe_join_flags(finish_reasons),
        input_tokens=getattr(usage, "prompt_token_count", None),
        output_tokens=getattr(usage, "candidates_token_count", None),
        total_tokens=getattr(usage, "total_token_count", None),
        latency_seconds=latency,
        provider_metadata_json=json.dumps({
            "cached_content_token_count": getattr(
                usage, "cached_content_token_count", None
            ),
            "thoughts_token_count": getattr(usage, "thoughts_token_count", None),
        }),
    )

PROVIDER_CALLERS = {
    "openai": call_openai,
    "anthropic": call_anthropic,
    "google": call_google,
}

In [72]:
# -----------------------------
# Error classification and checkpoint helpers
# -----------------------------

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def classify_exception(exc: Exception) -> str:
    text = f"{type(exc).__name__}: {exc}".lower()
    if any(token in text for token in [
        "missing api key",
        "authentication",
        "invalid api key",
        "unauthorized",
        "permission_denied",
        "permission denied",
        "model_not_found",
        "model not found",
        "not have access",
        "unsupported parameter",
        "unknown parameter",
        "invalid_request_error",
        "invalid_argument",
    ]):
        # Schema-specific invalid-request errors are separated below.
        if any(token in text for token in [
            "json schema",
            "json_schema",
            "output_config.format.schema",
            "response_json_schema",
            "schema is invalid",
        ]):
            return "schema_configuration_error"
        return "configuration_or_auth_error"
    if any(token in text for token in [
        "schema_validation",
        "validationerror",
        "no valid json",
        "missing required output",
        "empty model output",
        "probability outside",
        "unrecognized target",
        "cannot normalize yes/no",
    ]):
        return "schema_or_parse_error"
    if "refusal" in text:
        return "refusal"
    if any(token in text for token in [
        "429",
        "rate_limit",
        "rate limit",
        "overloaded",
        "529",
        "500",
        "502",
        "503",
        "504",
        "timeout",
        "timed out",
        "connection",
        "temporarily unavailable",
        "resource_exhausted",
    ]):
        return "transient_api_error"
    return "unknown_api_error"

def stable_json_hash(value: Mapping[str, Any]) -> str:
    return sha256_text(json.dumps(value, sort_keys=True, default=str))

def model_config_hash(config: Mapping[str, Any]) -> str:
    visible = {
        key: value for key, value in config.items()
        if key != "api_key_env"
    }
    return stable_json_hash(visible)

def corpus_dataset_hash(corpus: pd.DataFrame) -> str:
    fields = corpus[[
        "corpus_passage_id",
        "document",
        "prediction_text",
        "text_sha256",
    ]].sort_values("corpus_passage_id")
    return sha256_text(fields.to_csv(index=False))

def make_run_key(
    row: Mapping[str, Any],
    config: Mapping[str, Any],
    prompt_variant: str,
    dataset_hash: str,
) -> str:
    system, user = build_prompt(row["prediction_text"], prompt_variant)
    payload = {
        "corpus_passage_id": row["corpus_passage_id"],
        "text_sha256": row["text_sha256"],
        "dataset_hash": dataset_hash,
        "provider": config["provider"],
        "model": config["model"],
        "model_config_hash": model_config_hash(config),
        "prompt_variant": prompt_variant,
        "system_sha256": sha256_text(system),
        "user_sha256": sha256_text(user),
        "schema_sha256": OUTPUT_SCHEMA_SHA256,
    }
    return stable_json_hash(payload)

def checkpoint_path(config: Mapping[str, Any], prompt_variant: str) -> Path:
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{config['run_name']}__{prompt_variant}")
    return CHECKPOINT_DIR / f"{safe}.jsonl"

def append_jsonl(path: Path, record: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def load_jsonl_latest(path: Path) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"Skipping malformed checkpoint line {path}:{line_number}")
                continue
            run_key = record.get("run_key")
            if run_key:
                latest[run_key] = record
    return latest

In [73]:
# -----------------------------
# Restart-safe prediction runner
# -----------------------------

def run_one_prediction(
    row: Mapping[str, Any],
    config: Mapping[str, Any],
    prompt_variant: str,
    dataset_hash: str,
    stage: str,
) -> dict[str, Any]:
    system_prompt, base_user_prompt = build_prompt(
        row["prediction_text"], prompt_variant
    )
    run_key = make_run_key(row, config, prompt_variant, dataset_hash)
    caller = PROVIDER_CALLERS[config["provider"]]
    attempt_history: list[dict[str, Any]] = []
    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0
    total_latency = 0.0
    last_error_type = ""
    last_error_message = ""
    latest_response: Optional[ProviderResponse] = None

    for attempt in range(1, MAX_API_RETRIES + 1):
        user_prompt = base_user_prompt
        if attempt > 1 and last_error_type == "schema_or_parse_error":
            user_prompt += (
                "\n\nRETRY CORRECTION: Your previous response did not validate. "
                "Return one flat JSON object containing every required field. "
                "unlearning_probability must be a single number from 0 to 1; "
                "do not return a nested probabilities object."
            )
        try:
            response = caller(config, system_prompt, user_prompt)
            latest_response = response
            total_latency += response.latency_seconds
            total_input_tokens += int(response.input_tokens or 0)
            total_output_tokens += int(response.output_tokens or 0)
            total_tokens += int(response.total_tokens or 0)

            parse_source = (
                response.parsed_object
                if response.parsed_object is not None
                else response.raw_text
            )
            parsed, diagnostics = parse_and_validate_model_output(
                parse_source,
                row["prediction_text"],
            )
            attempt_history.append({
                "attempt": attempt,
                "status": "ok",
                "response_id": response.response_id,
                "finish_reason": response.finish_reason,
            })
            return {
                "run_key": run_key,
                "timestamp_utc": utc_now_iso(),
                "stage": stage,
                "status": "ok",
                "error_type": "",
                "error_message": "",
                "provider": config["provider"],
                "run_name": config["run_name"],
                "requested_model": config["model"],
                "resolved_model": response.resolved_model,
                "model_config_sha256": model_config_hash(config),
                "reasoning_effort": config.get("reasoning_effort", ""),
                "prompt_variant": prompt_variant,
                "prompt_variant_sha256": sha256_text(
                    json.dumps(prompt_variant_record(prompt_variant), sort_keys=True)
                ),
                "system_prompt_sha256": sha256_text(system_prompt),
                "user_prompt_sha256": sha256_text(base_user_prompt),
                "schema_sha256": OUTPUT_SCHEMA_SHA256,
                "dataset_sha256": dataset_hash,
                "corpus_passage_id": row["corpus_passage_id"],
                "Passage_ID": row.get("Passage_ID", ""),
                "document": row["document"],
                "pdf_page_start": row["pdf_page_start"],
                "pdf_page_end": row["pdf_page_end"],
                "text_sha256": row["text_sha256"],
                "prediction_text": row["prediction_text"],
                "unlearning_probability": parsed.unlearning_probability,
                "unlearning_label": parsed.unlearning_label,
                "prediction_binary_at_0_5": int(
                    parsed.unlearning_probability >= DEFAULT_THRESHOLD
                ),
                "target_category": parsed.target_category,
                "evidence_quote": parsed.evidence_quote,
                "rationale": parsed.rationale,
                **diagnostics,
                "raw_output": response.raw_text,
                "response_id": response.response_id,
                "finish_reason": response.finish_reason,
                "input_tokens": total_input_tokens or np.nan,
                "output_tokens": total_output_tokens or np.nan,
                "total_tokens": total_tokens or np.nan,
                "latency_seconds": total_latency,
                "attempt_count": attempt,
                "attempt_history_json": json.dumps(attempt_history),
                "provider_metadata_json": response.provider_metadata_json,
            }
        except Exception as exc:
            error_type = classify_exception(exc)
            last_error_type = error_type
            last_error_message = f"{type(exc).__name__}: {exc}"
            attempt_history.append({
                "attempt": attempt,
                "status": error_type,
                "error_message": last_error_message[:1000],
            })

            retryable = (
                error_type == "transient_api_error"
                or (
                    error_type == "schema_or_parse_error"
                    and sum(
                        h["status"] == "schema_or_parse_error"
                        for h in attempt_history
                    ) <= MAX_SCHEMA_RETRIES
                )
            )
            if retryable and attempt < MAX_API_RETRIES:
                delay = BASE_RETRY_SECONDS * (2 ** (attempt - 1))
                delay += random.random() * 0.75
                time.sleep(delay)
                continue
            break

    response = latest_response
    return {
        "run_key": run_key,
        "timestamp_utc": utc_now_iso(),
        "stage": stage,
        "status": last_error_type or "unknown_api_error",
        "error_type": last_error_type or "unknown_api_error",
        "error_message": last_error_message,
        "provider": config["provider"],
        "run_name": config["run_name"],
        "requested_model": config["model"],
        "resolved_model": (
            response.resolved_model if response is not None else config["model"]
        ),
        "model_config_sha256": model_config_hash(config),
        "reasoning_effort": config.get("reasoning_effort", ""),
        "prompt_variant": prompt_variant,
        "prompt_variant_sha256": sha256_text(
            json.dumps(prompt_variant_record(prompt_variant), sort_keys=True)
        ),
        "system_prompt_sha256": sha256_text(system_prompt),
        "user_prompt_sha256": sha256_text(base_user_prompt),
        "schema_sha256": OUTPUT_SCHEMA_SHA256,
        "dataset_sha256": dataset_hash,
        "corpus_passage_id": row["corpus_passage_id"],
        "Passage_ID": row.get("Passage_ID", ""),
        "document": row["document"],
        "pdf_page_start": row["pdf_page_start"],
        "pdf_page_end": row["pdf_page_end"],
        "text_sha256": row["text_sha256"],
        "prediction_text": row["prediction_text"],
        "unlearning_probability": np.nan,
        "unlearning_label": "",
        "prediction_binary_at_0_5": np.nan,
        "target_category": "",
        "evidence_quote": "",
        "rationale": "",
        "parser_repair_flags": "",
        "label_probability_inconsistent": np.nan,
        "target_label_inconsistent": np.nan,
        "evidence_quote_valid": np.nan,
        "schema_valid": False,
        "raw_output": response.raw_text if response is not None else "",
        "response_id": response.response_id if response is not None else "",
        "finish_reason": response.finish_reason if response is not None else "",
        "input_tokens": total_input_tokens or np.nan,
        "output_tokens": total_output_tokens or np.nan,
        "total_tokens": total_tokens or np.nan,
        "latency_seconds": total_latency,
        "attempt_count": len(attempt_history),
        "attempt_history_json": json.dumps(attempt_history),
        "provider_metadata_json": (
            response.provider_metadata_json if response is not None else ""
        ),
    }

def select_preflight_rows(corpus: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    selected_ids: list[str] = []

    def add_first(mask: pd.Series) -> None:
        candidates = corpus.loc[mask & ~corpus["corpus_passage_id"].isin(selected_ids)]
        if not candidates.empty:
            selected_ids.append(str(candidates.iloc[0]["corpus_passage_id"]))

    add_first(corpus["matched_gold"] & corpus["new_gold_binary"].eq(1))
    add_first(corpus["matched_gold"] & corpus["new_gold_binary"].eq(0))
    add_first(~corpus["matched_gold"])
    for passage_id in corpus["corpus_passage_id"]:
        if len(selected_ids) >= n:
            break
        if passage_id not in selected_ids:
            selected_ids.append(str(passage_id))
    return corpus.set_index("corpus_passage_id").loc[selected_ids[:n]].reset_index()

def run_configuration(
    corpus: pd.DataFrame,
    config: Mapping[str, Any],
    prompt_variant: str,
    dataset_hash: str,
    stage: str,
) -> pd.DataFrame:
    path = checkpoint_path(config, prompt_variant)
    existing = load_jsonl_latest(path) if RESUME_FROM_JSONL else {}
    records: list[dict[str, Any]] = []
    consecutive_configuration_errors = 0

    for row in tqdm(
        corpus.to_dict("records"),
        desc=f"{config['run_name']} | {prompt_variant}",
        leave=False,
    ):
        run_key = make_run_key(row, config, prompt_variant, dataset_hash)
        old = existing.get(run_key)
        if old and old.get("status") == "ok":
            records.append(old)
            continue

        record = run_one_prediction(
            row=row,
            config=config,
            prompt_variant=prompt_variant,
            dataset_hash=dataset_hash,
            stage=stage,
        )
        append_jsonl(path, record)
        records.append(record)

        if record["status"] in {
            "configuration_or_auth_error",
            "schema_configuration_error",
        }:
            consecutive_configuration_errors += 1
        else:
            consecutive_configuration_errors = 0

        if consecutive_configuration_errors >= 2:
            print(
                f"Circuit breaker: stopping {config['run_name']} / {prompt_variant} "
                f"after {consecutive_configuration_errors} consecutive configuration errors."
            )
            break
        time.sleep(REQUEST_SLEEP_SECONDS)

    return pd.DataFrame(records)

def run_llm_grid(
    corpus: pd.DataFrame,
    stage: str,
) -> pd.DataFrame:
    dataset_hash = corpus_dataset_hash(FINAL_CORPUS_DF)
    frames: list[pd.DataFrame] = []
    for config in MODEL_CONFIGS:
        if not config.get("enabled", True):
            continue
        for prompt_variant in PROMPT_VARIANTS["prompt_variant"]:
            try:
                frame = run_configuration(
                    corpus=corpus,
                    config=config,
                    prompt_variant=prompt_variant,
                    dataset_hash=dataset_hash,
                    stage=stage,
                )
                if not frame.empty:
                    frames.append(frame)
            except Exception as exc:
                print(
                    f"Configuration failed but other providers will continue: "
                    f"{config['run_name']} / {prompt_variant}: {type(exc).__name__}: {exc}"
                )
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_all_prediction_checkpoints() -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for config in MODEL_CONFIGS:
        for prompt_variant in PROMPT_VARIANTS["prompt_variant"]:
            latest = load_jsonl_latest(checkpoint_path(config, prompt_variant))
            records.extend(latest.values())
    if not records:
        return pd.DataFrame()
    frame = pd.DataFrame(records)
    frame = frame.sort_values("timestamp_utc").drop_duplicates(
        "run_key", keep="last"
    )
    return frame.reset_index(drop=True)

def expected_prediction_grid(corpus: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for config in MODEL_CONFIGS:
        if not config.get("enabled", True):
            continue
        for variant in PROMPT_VARIANTS["prompt_variant"]:
            rows.append({
                "provider": config["provider"],
                "run_name": config["run_name"],
                "requested_model": config["model"],
                "prompt_variant": variant,
                "expected_rows": len(corpus),
            })
    return pd.DataFrame(rows)

def prediction_quality_summary(
    predictions: pd.DataFrame,
    corpus: pd.DataFrame,
) -> pd.DataFrame:
    expected = expected_prediction_grid(corpus)
    if predictions.empty:
        expected["observed_rows"] = 0
        expected["ok_rows"] = 0
        expected["valid_rate"] = 0.0
        expected["missing_rows"] = expected["expected_rows"]
        return expected

    observed = (
        predictions.groupby(
            ["provider", "run_name", "requested_model", "prompt_variant"],
            dropna=False,
        )
        .agg(
            observed_rows=("run_key", "nunique"),
            ok_rows=("status", lambda s: int(s.eq("ok").sum())),
            schema_or_parse_errors=(
                "status", lambda s: int(s.eq("schema_or_parse_error").sum())
            ),
            transient_errors=(
                "status", lambda s: int(s.eq("transient_api_error").sum())
            ),
            configuration_errors=(
                "status",
                lambda s: int(
                    s.isin([
                        "configuration_or_auth_error",
                        "schema_configuration_error",
                    ]).sum()
                ),
            ),
            evidence_quote_invalid=(
                "evidence_quote_valid", lambda s: int(s.eq(False).sum())
            ),
            label_probability_inconsistent=(
                "label_probability_inconsistent", lambda s: int(s.eq(True).sum())
            ),
            input_tokens=("input_tokens", "sum"),
            output_tokens=("output_tokens", "sum"),
            latency_seconds=("latency_seconds", "sum"),
        )
        .reset_index()
    )
    quality = expected.merge(
        observed,
        on=["provider", "run_name", "requested_model", "prompt_variant"],
        how="left",
    )
    for column in [
        "observed_rows",
        "ok_rows",
        "schema_or_parse_errors",
        "transient_errors",
        "configuration_errors",
        "evidence_quote_invalid",
        "label_probability_inconsistent",
    ]:
        quality[column] = quality[column].fillna(0).astype(int)
    quality["missing_rows"] = quality["expected_rows"] - quality["ok_rows"]
    quality["valid_rate"] = quality["ok_rows"] / quality["expected_rows"]
    return quality.sort_values(["provider", "run_name", "prompt_variant"])

if RUN_EXTRACTION_STAGE:
    DATASET_SHA256 = corpus_dataset_hash(FINAL_CORPUS_DF)
    print("Final corpus dataset SHA256:", DATASET_SHA256)
    print("Expected complete calls:", int(expected_prediction_grid(FINAL_CORPUS_DF)["expected_rows"].sum()))

In [74]:
# ============================================================
# GPT TEST ONLY MODE
# Skip PDF extraction/matching and use finalized 76 passages
# ============================================================

GPT_TEST_ONLY_MODE = True

FINAL_CORPUS_DF = GOLD_DF.copy()

# Standardize columns expected by the prediction/evaluation pipeline
FINAL_CORPUS_DF["corpus_passage_id"] = FINAL_CORPUS_DF["Passage_ID"].astype(str)
FINAL_CORPUS_DF["document"] = FINAL_CORPUS_DF["Document"].astype(str)
FINAL_CORPUS_DF["source_file"] = FINAL_CORPUS_DF["Source_File"].fillna("").astype(str)

FINAL_CORPUS_DF["prediction_text"] = (
    FINAL_CORPUS_DF["Text_Content"]
    .fillna("")
    .astype(str)
    .str.strip()
)

FINAL_CORPUS_DF["text_clean"] = FINAL_CORPUS_DF["prediction_text"]

# All 76 rows are finalized human-coded gold passages
FINAL_CORPUS_DF["matched_gold"] = True

# There are no assumed-negative PDF-only paragraphs in GPT Test mode
FINAL_CORPUS_DF["full_new_assumed_binary"] = (
    FINAL_CORPUS_DF["new_gold_binary"].astype(int)
)
FINAL_CORPUS_DF["is_unlabeled_assumed_no"] = False

# Preserve Kyle-only exclusion logic
FINAL_CORPUS_DF["evaluation_exclude_kyle_only"] = (
    FINAL_CORPUS_DF["kyle_only_row"].fillna(False).astype(bool)
)

# PDF provenance columns are intentionally unavailable in GPT Test-only mode
FINAL_CORPUS_DF["pdf_page_start"] = np.nan
FINAL_CORPUS_DF["pdf_page_end"] = np.nan
FINAL_CORPUS_DF["extraction_flags"] = ""
FINAL_CORPUS_DF["source_line_ids"] = ""

# Hash each passage exactly for restart-safe API checkpointing
FINAL_CORPUS_DF["text_sha256"] = FINAL_CORPUS_DF["prediction_text"].map(
    lambda x: sha256_text(normalize_for_match(x))
)

# ------------------------------------------------------------
# QA checks BEFORE any API calls
# ------------------------------------------------------------

assert len(FINAL_CORPUS_DF) == 76, (
    f"Expected 76 finalized passages, found {len(FINAL_CORPUS_DF)}"
)

assert FINAL_CORPUS_DF["Passage_ID"].is_unique, \
    "Passage_ID contains duplicates."

assert FINAL_CORPUS_DF["corpus_passage_id"].is_unique, \
    "corpus_passage_id contains duplicates."

assert FINAL_CORPUS_DF["prediction_text"].str.len().gt(0).all(), \
    "At least one GPT Test passage is blank."

assert FINAL_CORPUS_DF["new_gold_binary"].isin([0, 1]).all()
assert FINAL_CORPUS_DF["old_gold_binary"].isin([0, 1]).all()

# Recheck Kyle-only formula
expected_kyle_only = (
    FINAL_CORPUS_DF["kyle_binary"].notna()
    & FINAL_CORPUS_DF["anmol_binary"].isna()
    & FINAL_CORPUS_DF["prerana_binary"].isna()
)

assert (
    expected_kyle_only
    == FINAL_CORPUS_DF["kyle_only_row"].astype(bool)
).all(), "Kyle-only rows do not match the expected formula."

# Dataset hash used for checkpoint safety
DATASET_SHA256 = corpus_dataset_hash(FINAL_CORPUS_DF)

print("✓ GPT TEST ONLY MODE READY")
print("Final passages:", len(FINAL_CORPUS_DF))
print("Documents:", FINAL_CORPUS_DF["document"].nunique())
print("Dataset SHA256:", DATASET_SHA256)

print("\nNew gold:")
print(FINAL_CORPUS_DF["new_gold_unlearning"].value_counts(dropna=False))

print("\nOld gold:")
print(FINAL_CORPUS_DF["old_gold_unlearning"].value_counts(dropna=False))

print("\nKyle-only passages to exclude in secondary analyses:")
print(int(FINAL_CORPUS_DF["kyle_only_row"].sum()))


# ------------------------------------------------------------
# Compatibility placeholders
# These prevent later result-export cells from looking for
# PDF-extraction audit tables that do not exist in this mode.
# ------------------------------------------------------------

INPUT_MANIFEST_DF = pd.DataFrame([{
    "input_type": "GPT Test only",
    "path": str(WORKBOOK_PATH),
    "rows": len(FINAL_CORPUS_DF),
}])

GOLD_MATCH_DF = pd.DataFrame()
PAGE_AUDIT_DF = pd.DataFrame()
DUPLICATE_REMOVAL_AUDIT_DF = pd.DataFrame()
EXAMPLE_LEAKAGE_AUDIT_DF = pd.DataFrame()
NEAR_DUPLICATE_REVIEW_DF = pd.DataFrame()
MATCH_REVIEW_QUEUE_DF = pd.DataFrame()


# IMPORTANT:
# Downstream notebook cells currently use RUN_EXTRACTION_STAGE
# as a generic "corpus is ready" switch.
#
# We flip it back to True HERE ONLY, after all PDF extraction
# cells have already been skipped.
#
# This does NOT run PDF extraction.
RUN_EXTRACTION_STAGE = True

print("\n✓ Downstream prediction/evaluation pipeline enabled.")

✓ GPT TEST ONLY MODE READY
Final passages: 76
Documents: 5
Dataset SHA256: e62f8a50d1df37141b04d95e3e9e83933ccfa4f0315fed3047f5e0abdc5287ae

New gold:
new_gold_unlearning
Yes    46
No     30
Name: count, dtype: int64

Old gold:
old_gold_unlearning
Yes    54
No     22
Name: count, dtype: int64

Kyle-only passages to exclude in secondary analyses:
8

✓ Downstream prediction/evaluation pipeline enabled.


In [75]:
# API execution gate.
if RUN_LLM_CALLS:
    if not RUN_EXTRACTION_STAGE:
        raise RuntimeError("Load FINAL_CORPUS_DF before running API calls.")

    PREFLIGHT_PREDICTIONS_DF = pd.DataFrame()
    if RUN_PROVIDER_PREFLIGHT:
        PREFLIGHT_CORPUS_DF = select_preflight_rows(
            FINAL_CORPUS_DF,
            PREFLIGHT_ROWS_PER_CONFIGURATION,
        )
        PREFLIGHT_PREDICTIONS_DF = run_llm_grid(
            PREFLIGHT_CORPUS_DF,
            stage="preflight",
        )
        PREFLIGHT_QUALITY_DF = prediction_quality_summary(
            PREFLIGHT_PREDICTIONS_DF,
            PREFLIGHT_CORPUS_DF,
        )
        display(PREFLIGHT_QUALITY_DF)
        if (
            RUN_COMPLETE_LLM_GRID
            and PREFLIGHT_QUALITY_DF["valid_rate"].lt(1.0).any()
        ):
            raise RuntimeError(
                "Preflight coverage is below 100% for at least one model/prompt "
                "configuration. Review checkpoints before running the complete grid."
            )

    if RUN_COMPLETE_LLM_GRID:
        COMPLETE_PREDICTIONS_DF = run_llm_grid(
            FINAL_CORPUS_DF.head(ROW_LIMIT) if ROW_LIMIT else FINAL_CORPUS_DF,
            stage="complete",
        )

PREDICTIONS_DF = load_all_prediction_checkpoints()
PREDICTION_QUALITY_DF = prediction_quality_summary(
    PREDICTIONS_DF,
    FINAL_CORPUS_DF if RUN_EXTRACTION_STAGE else pd.DataFrame(),
)
display(PREDICTION_QUALITY_DF)

openai_gpt_5_6_terra_low | direct_target_only:   0%|          | 0/3 [00:00<?, ?it/s]

openai_gpt_5_6_terra_low | codebook_no_examples_no_checklist:   0%|          | 0/3 [00:00<?, ?it/s]

openai_gpt_5_6_terra_low | codebook_with_examples_no_checklist:   0%|          | 0/3 [00:00<?, ?it/s]

openai_gpt_5_6_terra_low | codebook_no_examples_with_checklist:   0%|          | 0/3 [00:00<?, ?it/s]

openai_gpt_5_6_terra_low | codebook_with_examples_with_checklist:   0%|          | 0/3 [00:00<?, ?it/s]

...

google_gemini_3_1_flash_lite_low | codebook_no_examples_no_checklist:   0%|          | 0/76 [00:00<?, ?it/s]

google_gemini_3_1_flash_lite_low | codebook_with_examples_no_checklist:   0%|          | 0/76 [00:00<?, ?it/s]

google_gemini_3_1_flash_lite_low | codebook_no_examples_with_checklist:   0%|          | 0/76 [00:00<?, ?it/s]

google_gemini_3_1_flash_lite_low | codebook_with_examples_with_checklist:   0%|          | 0/76 [00:00<?, ?it/…

,provider,run_name,requested_model,prompt_variant,expected_rows,observed_rows,ok_rows,schema_or_parse_errors,transient_errors,configuration_errors,evidence_quote_invalid,label_probability_inconsistent,input_tokens,output_tokens,latency_seconds,missing_rows,valid_rate
6,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_no_examples_no_checklist,76,76,76,0,0,0,4,0,225394,14889,229.703640,0,1.000000
8,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_no_examples_with_checklist,76,76,76,0,0,0,6,0,241354,15680,249.902155,0,1.000000
7,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_with_examples_no_checklist,76,76,76,0,0,0,4,0,260962,14348,229.667591,0,1.000000
9,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_with_examples_with_checklist,76,76,76,0,0,0,9,0,276922,15760,239.723627,0,1.000000
5,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,direct_target_only,76,76,76,0,0,0,4,0,63286,12653,194.680276,0,1.000000
11,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_no_checklist,76,76,75,1,0,0,2,0,199502,9695,666.804174,1,0.986842
13,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_with_checklist,76,76,75,1,0,0,3,0,222762,9956,824.428777,1,0.986842
12,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_with_examples_no_checklist,76,76,76,0,0,0,2,0,238985,10095,591.026743,0,1.000000
14,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_with_examples_with_checklist,76,76,74,2,0,0,4,0,264545,9763,821.893622,2,0.973684
10,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,direct_target_only,76,76,76,0,0,0,3,0,31427,7653,493.778059,0,1.000000


## 8. Evaluation targets and coverage-aware metrics

The evaluation table keeps each reference family separate. Coder-specific and pooled-coder analyses use the actual coder label, while new/old gold analyses can be stratified by the priority-selected `Original_Labeler`.

Every metric table includes `expected_n`, `valid_prediction_n`, and `prediction_coverage`. AUROC is left missing, with a flag, when a subgroup contains only one reference class.

In [78]:
# -----------------------------
# Current prediction filtering
# -----------------------------

def expected_current_run_keys(corpus: pd.DataFrame) -> set[str]:
    """Exact run keys for the current corpus, prompts, schema, and model configs."""
    if corpus.empty:
        return set()
    dataset_hash = corpus_dataset_hash(corpus)
    keys: set[str] = set()
    records = corpus.to_dict("records")
    for config in MODEL_CONFIGS:
        if not config.get("enabled", True):
            continue
        for prompt_variant in PROMPT_VARIANTS["prompt_variant"]:
            for row in records:
                keys.add(
                    make_run_key(
                        row=row,
                        config=config,
                        prompt_variant=prompt_variant,
                        dataset_hash=dataset_hash,
                    )
                )
    return keys


def filter_current_predictions(
    predictions: pd.DataFrame,
    corpus: pd.DataFrame,
) -> pd.DataFrame:
    if predictions.empty:
        return predictions.copy()
    required = {
        "run_key", "timestamp_utc", "dataset_sha256",
        "corpus_passage_id", "run_name", "prompt_variant",
    }
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(
            f"Prediction checkpoints are missing required columns: {sorted(missing)}"
        )

    dataset_hash = corpus_dataset_hash(corpus)
    expected_keys = expected_current_run_keys(corpus)
    current = predictions.loc[
        predictions["run_key"].isin(expected_keys)
        & predictions["dataset_sha256"].eq(dataset_hash)
        & predictions["corpus_passage_id"].isin(corpus["corpus_passage_id"])
    ].copy()
    current = current.sort_values("timestamp_utc").drop_duplicates(
        "run_key",
        keep="last",
    )
    if current.duplicated(
        ["run_name", "prompt_variant", "corpus_passage_id"]
    ).any():
        raise AssertionError(
            "Multiple current checkpoints remain for one model/prompt/passage."
        )
    return current.reset_index(drop=True)

if RUN_EXTRACTION_STAGE:
    CURRENT_PREDICTIONS_DF = filter_current_predictions(
        PREDICTIONS_DF,
        FINAL_CORPUS_DF,
    )
    CURRENT_PREDICTION_QUALITY_DF = prediction_quality_summary(
        CURRENT_PREDICTIONS_DF,
        FINAL_CORPUS_DF,
    )
    display(CURRENT_PREDICTION_QUALITY_DF)
else:
    CURRENT_PREDICTIONS_DF = pd.DataFrame()
    CURRENT_PREDICTION_QUALITY_DF = pd.DataFrame()

,provider,run_name,requested_model,prompt_variant,expected_rows,observed_rows,ok_rows,schema_or_parse_errors,transient_errors,configuration_errors,evidence_quote_invalid,label_probability_inconsistent,input_tokens,output_tokens,latency_seconds,missing_rows,valid_rate
6,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_no_examples_no_checklist,76,76,76,0,0,0,4,0,225394,14889,229.703640,0,1.000000
8,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_no_examples_with_checklist,76,76,76,0,0,0,6,0,241354,15680,249.902155,0,1.000000
7,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_with_examples_no_checklist,76,76,76,0,0,0,4,0,260962,14348,229.667591,0,1.000000
9,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_with_examples_with_checklist,76,76,76,0,0,0,9,0,276922,15760,239.723627,0,1.000000
5,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,direct_target_only,76,76,76,0,0,0,4,0,63286,12653,194.680276,0,1.000000
11,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_no_checklist,76,76,75,1,0,0,2,0,199502,9695,666.804174,1,0.986842
13,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_with_checklist,76,76,75,1,0,0,3,0,222762,9956,824.428777,1,0.986842
12,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_with_examples_no_checklist,76,76,76,0,0,0,2,0,238985,10095,591.026743,0,1.000000
14,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_with_examples_with_checklist,76,76,74,2,0,0,4,0,264545,9763,821.893622,2,0.973684
10,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,direct_target_only,76,76,76,0,0,0,3,0,31427,7653,493.778059,0,1.000000


In [79]:
# -----------------------------
# Evaluation target table
# -----------------------------

PRIMARY_SCOPE_SPECS = [
    {
        "evaluation_scope": "all_paragraphs_new_gold_unlabeled_no",
        "target_family": "full_corpus_assumed_negative",
        "target_column": "full_new_assumed_binary",
        "matched_only": False,
    },
    {
        "evaluation_scope": "new_gold_test_only",
        "target_family": "new_gold",
        "target_column": "new_gold_binary",
        "matched_only": True,
    },
    {
        "evaluation_scope": "old_gold_test_only",
        "target_family": "old_gold",
        "target_column": "old_gold_binary",
        "matched_only": True,
    },
]

def primary_target_rows(corpus: pd.DataFrame) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for spec in PRIMARY_SCOPE_SPECS:
        subset = corpus.loc[
            corpus["matched_gold"] if spec["matched_only"] else pd.Series(True, index=corpus.index)
        ]
        for item in subset.to_dict("records"):
            labeler = (
                (normalize_space(item.get("Original_Labeler")) or "unassigned_human_labeler")
                if bool(item.get("matched_gold"))
                else "unlabeled_assumed_no"
            )
            rows.append({
                "target_row_id": (
                    f"{spec['evaluation_scope']}::{item['corpus_passage_id']}::{labeler}"
                ),
                "evaluation_scope": spec["evaluation_scope"],
                "target_family": spec["target_family"],
                "target_labeler": labeler,
                "label_source_type": (
                    "priority_gold_original_labeler"
                    if bool(item.get("matched_gold"))
                    else "assumed_negative"
                ),
                "corpus_passage_id": item["corpus_passage_id"],
                "Passage_ID": item.get("Passage_ID"),
                "document": item["document"],
                "y_true": int(item[spec["target_column"]]),
                "kyle_only_row": bool(item.get("kyle_only_row", False)),
            })
    return rows

def coder_target_rows(corpus: pd.DataFrame) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for coder in ["anmol", "prerana", "kyle"]:
        binary_column = f"{coder}_binary"
        subset = corpus.loc[corpus[binary_column].notna()]
        for item in subset.to_dict("records"):
            rows.append({
                "target_row_id": (
                    f"labeler_{coder}::{item['corpus_passage_id']}::{coder}"
                ),
                "evaluation_scope": f"labeler_{coder}",
                "target_family": "individual_coder",
                "target_labeler": coder,
                "label_source_type": "individual_coder",
                "corpus_passage_id": item["corpus_passage_id"],
                "Passage_ID": item.get("Passage_ID"),
                "document": item["document"],
                "y_true": int(item[binary_column]),
                "kyle_only_row": bool(item.get("kyle_only_row", False)),
            })
            rows.append({
                "target_row_id": (
                    f"pooled_coder_labels::{item['corpus_passage_id']}::{coder}"
                ),
                "evaluation_scope": "pooled_coder_labels",
                "target_family": "pooled_coder_decisions",
                "target_labeler": coder,
                "label_source_type": "individual_coder",
                "corpus_passage_id": item["corpus_passage_id"],
                "Passage_ID": item.get("Passage_ID"),
                "document": item["document"],
                "y_true": int(item[binary_column]),
                "kyle_only_row": bool(item.get("kyle_only_row", False)),
            })
    return rows

def add_drop_kyle_only_scopes(targets: pd.DataFrame) -> pd.DataFrame:
    retained = targets.loc[~targets["kyle_only_row"]].copy()
    retained["evaluation_scope"] = (
        retained["evaluation_scope"] + "__drop_kyle_only"
    )
    retained["target_row_id"] = (
        retained["target_row_id"] + "__drop_kyle_only"
    )
    return pd.concat([targets, retained], ignore_index=True)

def build_evaluation_targets(corpus: pd.DataFrame) -> pd.DataFrame:
    targets = pd.DataFrame(
        primary_target_rows(corpus) + coder_target_rows(corpus)
    )
    targets = add_drop_kyle_only_scopes(targets)
    if targets["target_row_id"].duplicated().any():
        raise AssertionError("target_row_id values are not unique.")
    return targets

if RUN_EXTRACTION_STAGE:
    EVALUATION_TARGETS_DF = build_evaluation_targets(FINAL_CORPUS_DF)
    display(
        EVALUATION_TARGETS_DF.groupby(
            ["evaluation_scope", "target_family"], as_index=False
        ).agg(
            n=("target_row_id", "size"),
            positives=("y_true", "sum"),
            negatives=("y_true", lambda s: int((1 - s).sum())),
            documents=("document", "nunique"),
            labelers=("target_labeler", "nunique"),
        )
    )
    EVALUATION_TARGETS_DF.to_csv(
        AUDIT_DIR / "evaluation_target_manifest.csv", index=False
    )

,evaluation_scope,target_family,n,positives,negatives,documents,labelers
0,all_paragraphs_new_gold_unlabeled_no,full_corpus_assumed_negative,76,46,30,5,4
1,all_paragraphs_new_gold_unlabeled_no__drop_kyle_only,full_corpus_assumed_negative,68,41,27,5,3
2,labeler_anmol,individual_coder,53,39,14,5,1
3,labeler_anmol__drop_kyle_only,individual_coder,53,39,14,5,1
4,labeler_kyle,individual_coder,51,46,5,4,1
5,labeler_kyle__drop_kyle_only,individual_coder,43,41,2,4,1
6,labeler_prerana,individual_coder,18,5,13,2,1
7,labeler_prerana__drop_kyle_only,individual_coder,18,5,13,2,1
8,new_gold_test_only,new_gold,76,46,30,5,4
9,new_gold_test_only__drop_kyle_only,new_gold,68,41,27,5,3


In [80]:
# -----------------------------
# Prediction/target join and metric functions
# -----------------------------

MODEL_ID_COLUMNS = ["provider", "run_name", "requested_model"]
BASE_GROUP_COLUMNS = [
    "prompt_variant",
    "evaluation_scope",
    "target_family",
]

def build_expected_evaluation_grid(
    targets: pd.DataFrame,
) -> pd.DataFrame:
    model_prompt_rows = []
    for config in MODEL_CONFIGS:
        if not config.get("enabled", True):
            continue
        for prompt_variant in PROMPT_VARIANTS["prompt_variant"]:
            model_prompt_rows.append({
                "provider": config["provider"],
                "run_name": config["run_name"],
                "requested_model": config["model"],
                "prompt_variant": prompt_variant,
            })
    model_prompt = pd.DataFrame(model_prompt_rows)
    targets = targets.copy()
    targets["_cross_key"] = 1
    model_prompt["_cross_key"] = 1
    expected = targets.merge(model_prompt, on="_cross_key", how="outer").drop(
        columns="_cross_key"
    )
    return expected

def prepare_evaluation_rows(
    predictions: pd.DataFrame,
    targets: pd.DataFrame,
) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame()
    valid = predictions.loc[predictions["status"].eq("ok")].copy()
    valid["unlearning_probability"] = pd.to_numeric(
        valid["unlearning_probability"], errors="coerce"
    )
    valid = valid.loc[valid["unlearning_probability"].between(0, 1)]
    if valid.duplicated(
        ["run_name", "prompt_variant", "corpus_passage_id"]
    ).any():
        raise AssertionError("Duplicate valid prediction keys remain.")
    evaluated = valid.merge(
        targets,
        on="corpus_passage_id",
        how="inner",
        suffixes=("", "_target"),
        validate="many_to_many",
    )
    if "document_target" in evaluated.columns:
        mismatch = evaluated["document"].ne(evaluated["document_target"])
        if mismatch.any():
            raise AssertionError("Prediction and target document values disagree.")
        evaluated["document"] = evaluated["document_target"]
        evaluated = evaluated.drop(columns=["document_target"])
    return evaluated

def safe_auroc(y_true: Sequence[int], probabilities: Sequence[float]) -> tuple[float, str]:
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    if len(np.unique(y)) < 2:
        return np.nan, "single_reference_class"
    return float(roc_auc_score(y, p)), ""

def calculate_binary_metrics(
    y_true: Sequence[int],
    probabilities: Sequence[float],
    threshold: float,
) -> dict[str, Any]:
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    auroc, auroc_flag = safe_auroc(y, p)
    return {
        "valid_prediction_n": int(len(y)),
        "reference_positive_n": int(y.sum()),
        "reference_negative_n": int((1 - y).sum()),
        "predicted_positive_n": int(pred.sum()),
        "accuracy": float(accuracy_score(y, pred)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "auroc": auroc,
        "auroc_flag": auroc_flag,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "threshold": float(threshold),
    }

ANALYSIS_LEVELS = {
    "model_overall": MODEL_ID_COLUMNS + BASE_GROUP_COLUMNS,
    "model_document": MODEL_ID_COLUMNS + BASE_GROUP_COLUMNS + ["document"],
    "model_labeler": MODEL_ID_COLUMNS + BASE_GROUP_COLUMNS + ["target_labeler"],
    "model_label_source_type": MODEL_ID_COLUMNS + BASE_GROUP_COLUMNS + ["label_source_type"],
    "model_document_labeler": (
        MODEL_ID_COLUMNS + BASE_GROUP_COLUMNS + ["document", "target_labeler"]
    ),
    "pooled_models_overall": BASE_GROUP_COLUMNS,
    "pooled_models_document": BASE_GROUP_COLUMNS + ["document"],
    "pooled_models_labeler": BASE_GROUP_COLUMNS + ["target_labeler"],
    "pooled_models_label_source_type": BASE_GROUP_COLUMNS + ["label_source_type"],
    "pooled_models_document_labeler": (
        BASE_GROUP_COLUMNS + ["document", "target_labeler"]
    ),
}

def evaluate_analysis_level(
    evaluated: pd.DataFrame,
    expected: pd.DataFrame,
    analysis_level: str,
    group_columns: list[str],
    threshold: float = DEFAULT_THRESHOLD,
    threshold_column: Optional[str] = None,
) -> pd.DataFrame:
    actual_rows: list[dict[str, Any]] = []
    if evaluated.empty:
        return pd.DataFrame()

    for keys, group in evaluated.groupby(group_columns, dropna=False, sort=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        metadata = dict(zip(group_columns, keys))
        group_threshold = (
            float(group[threshold_column].iloc[0])
            if threshold_column is not None
            else float(threshold)
        )
        if threshold_column is not None and group[threshold_column].nunique() != 1:
            raise AssertionError(
                f"Multiple thresholds inside one metric group for {analysis_level}."
            )
        metrics = calculate_binary_metrics(
            group["y_true"],
            group["unlearning_probability"],
            group_threshold,
        )
        actual_rows.append({
            "analysis_level": analysis_level,
            **metadata,
            **metrics,
        })
    actual = pd.DataFrame(actual_rows)

    expected_counts = (
        expected.groupby(group_columns, dropna=False)
        .size()
        .reset_index(name="expected_n")
    )
    result = actual.merge(
        expected_counts,
        on=group_columns,
        how="outer",
        validate="one_to_one",
    )
    result["analysis_level"] = analysis_level
    result["valid_prediction_n"] = result["valid_prediction_n"].fillna(0).astype(int)
    result["prediction_coverage"] = (
        result["valid_prediction_n"] / result["expected_n"]
    )
    return result

def evaluate_all_levels(
    evaluated: pd.DataFrame,
    expected: pd.DataFrame,
    threshold: float = DEFAULT_THRESHOLD,
    threshold_column: Optional[str] = None,
) -> pd.DataFrame:
    frames = []
    for level, columns in ANALYSIS_LEVELS.items():
        frame = evaluate_analysis_level(
            evaluated=evaluated,
            expected=expected,
            analysis_level=level,
            group_columns=columns,
            threshold=threshold,
            threshold_column=threshold_column,
        )
        if not frame.empty:
            frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if RUN_EXTRACTION_STAGE:
    EXPECTED_EVALUATION_GRID_DF = build_expected_evaluation_grid(
        EVALUATION_TARGETS_DF
    )
    EVALUATED_PREDICTIONS_DF = prepare_evaluation_rows(
        CURRENT_PREDICTIONS_DF,
        EVALUATION_TARGETS_DF,
    )
    DEFAULT_METRICS_DF = evaluate_all_levels(
        EVALUATED_PREDICTIONS_DF,
        EXPECTED_EVALUATION_GRID_DF,
        threshold=DEFAULT_THRESHOLD,
    )
    if not DEFAULT_METRICS_DF.empty:
        display(
            DEFAULT_METRICS_DF.loc[
                DEFAULT_METRICS_DF["analysis_level"].eq("model_overall"),
                [
                    "provider", "run_name", "requested_model", "prompt_variant",
                    "evaluation_scope", "valid_prediction_n", "expected_n",
                    "prediction_coverage", "accuracy", "recall", "f1", "auroc"
                ],
            ].sort_values(
                ["evaluation_scope", "prompt_variant", "f1"],
                ascending=[True, True, False],
            )
        )

,provider,run_name,requested_model,prompt_variant,evaluation_scope,valid_prediction_n,expected_n,prediction_coverage,accuracy,recall,f1,auroc
0,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_no_examples_no_checklist,all_paragraphs_new_gold_unlabeled_no,76,76,1.000000,0.618421,0.434783,0.579710,0.663406
70,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_no_checklist,all_paragraphs_new_gold_unlabeled_no,75,76,0.986842,0.560000,0.333333,0.476190,0.684444
140,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_no_examples_no_checklist,all_paragraphs_new_gold_unlabeled_no,76,76,1.000000,0.486842,0.173913,0.290909,0.621377
14,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,codebook_no_examples_with_checklist,all_paragraphs_new_gold_unlabeled_no,76,76,1.000000,0.605263,0.391304,0.545455,0.670290
84,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_no_examples_with_checklist,all_paragraphs_new_gold_unlabeled_no,75,76,0.986842,0.546667,0.266667,0.413793,0.721481
...,...,...,...,...,...,...,...,...,...,...,...,...
125,google,google_gemini_3_1_flash_lite_low,gemini-3.1-flash-lite,codebook_with_examples_with_checklist,pooled_coder_labels__drop_kyle_only,110,114,0.964912,0.436364,0.271605,0.415094,0.679012
195,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,codebook_with_examples_with_checklist,pooled_coder_labels__drop_kyle_only,114,114,1.000000,0.368421,0.164706,0.280000,0.759229
69,anthropic,anthropic_claude_haiku_4_5,claude-haiku-4-5-20251001,direct_target_only,pooled_coder_labels__drop_kyle_only,114,114,1.000000,0.570175,0.494118,0.631579,0.663489
209,openai,openai_gpt_5_6_terra_low,gpt-5.6-terra,direct_target_only,pooled_coder_labels__drop_kyle_only,114,114,1.000000,0.447368,0.294118,0.442478,0.581947


## 9. Threshold search and A/B contrasts

Threshold optimization is deliberately simple and is labeled **in-sample exploratory post-processing**. It can improve accuracy, recall, or F1, but it cannot change AUROC because AUROC uses the full probability ranking.

In [81]:
# -----------------------------
# Individual-model threshold optimization
# -----------------------------

THRESHOLD_KEY_COLUMNS = (
    MODEL_ID_COLUMNS + ["prompt_variant", "evaluation_scope", "target_family"]
)

def threshold_search_for_group(
    group: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    curve_rows = []
    for threshold in THRESHOLD_GRID:
        metrics = calculate_binary_metrics(
            group["y_true"],
            group["unlearning_probability"],
            float(threshold),
        )
        curve_rows.append(metrics)
    curve = pd.DataFrame(curve_rows)
    curve["distance_from_0_5"] = (curve["threshold"] - 0.5).abs()
    curve = curve.sort_values(
        ["f1", "accuracy", "recall", "precision", "distance_from_0_5"],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)
    best = curve.iloc[0].to_dict()
    return curve, best

def optimize_model_thresholds(
    evaluated: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if evaluated.empty:
        return pd.DataFrame(), pd.DataFrame()
    curve_frames: list[pd.DataFrame] = []
    best_rows: list[dict[str, Any]] = []

    for keys, group in evaluated.groupby(
        THRESHOLD_KEY_COLUMNS,
        dropna=False,
        sort=False,
    ):
        keys = keys if isinstance(keys, tuple) else (keys,)
        metadata = dict(zip(THRESHOLD_KEY_COLUMNS, keys))
        if group["y_true"].nunique() < 2:
            best_rows.append({
                **metadata,
                "optimized_threshold": DEFAULT_THRESHOLD,
                "optimization_status": "single_reference_class_no_search",
                "optimization_objective": PRIMARY_OPTIMIZATION_METRIC,
                "optimization_method": "in_sample_exploratory",
                **calculate_binary_metrics(
                    group["y_true"],
                    group["unlearning_probability"],
                    DEFAULT_THRESHOLD,
                ),
            })
            continue
        curve, best = threshold_search_for_group(group)
        curve = curve.assign(**metadata)
        curve_frames.append(curve)
        best_rows.append({
            **metadata,
            "optimized_threshold": best["threshold"],
            "optimization_status": "optimized",
            "optimization_objective": PRIMARY_OPTIMIZATION_METRIC,
            "optimization_method": "in_sample_exploratory",
            **{f"optimized_{k}": v for k, v in best.items()},
        })
    return (
        pd.concat(curve_frames, ignore_index=True) if curve_frames else pd.DataFrame(),
        pd.DataFrame(best_rows),
    )

def evaluate_model_specific_thresholds(
    evaluated: pd.DataFrame,
    expected: pd.DataFrame,
    thresholds: pd.DataFrame,
) -> pd.DataFrame:
    if evaluated.empty or thresholds.empty:
        return pd.DataFrame()
    key_cols = THRESHOLD_KEY_COLUMNS
    threshold_map = thresholds[
        key_cols + ["optimized_threshold"]
    ].drop_duplicates(key_cols)
    work = evaluated.merge(
        threshold_map,
        on=key_cols,
        how="inner",
        validate="many_to_one",
    )
    model_levels = {
        key: value
        for key, value in ANALYSIS_LEVELS.items()
        if key.startswith("model_")
    }
    frames = []
    for level, columns in model_levels.items():
        frames.append(
            evaluate_analysis_level(
                work,
                expected,
                analysis_level=level,
                group_columns=columns,
                threshold_column="optimized_threshold",
            )
        )
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not result.empty:
        result["threshold_mode"] = "model_scope_optimized_in_sample"
    return result

if RUN_EXTRACTION_STAGE:
    THRESHOLD_CURVES_DF, BEST_THRESHOLDS_DF = optimize_model_thresholds(
        EVALUATED_PREDICTIONS_DF
    )
    OPTIMIZED_MODEL_METRICS_DF = evaluate_model_specific_thresholds(
        EVALUATED_PREDICTIONS_DF,
        EXPECTED_EVALUATION_GRID_DF,
        BEST_THRESHOLDS_DF,
    )
    if not BEST_THRESHOLDS_DF.empty:
        display(
            BEST_THRESHOLDS_DF.loc[
                BEST_THRESHOLDS_DF["evaluation_scope"].isin([
                    "new_gold_test_only",
                    "old_gold_test_only",
                    "new_gold_test_only__drop_kyle_only",
                    "old_gold_test_only__drop_kyle_only",
                ]),
                [
                    "run_name", "prompt_variant", "evaluation_scope",
                    "optimized_threshold", "optimized_accuracy",
                    "optimized_recall", "optimized_f1", "optimized_auroc"
                ],
            ].sort_values(
                ["evaluation_scope", "run_name", "optimized_f1"],
                ascending=[True, True, False],
            )
        )

,run_name,prompt_variant,evaluation_scope,optimized_threshold,optimized_accuracy,optimized_recall,optimized_f1,optimized_auroc
61,anthropic_claude_haiku_4_5,direct_target_only,new_gold_test_only,0.12,0.605263,1.000000,0.754098,0.662319
73,anthropic_claude_haiku_4_5,codebook_no_examples_no_checklist,new_gold_test_only,0.05,0.605263,1.000000,0.754098,0.663406
85,anthropic_claude_haiku_4_5,codebook_with_examples_no_checklist,new_gold_test_only,0.05,0.605263,1.000000,0.754098,0.676449
97,anthropic_claude_haiku_4_5,codebook_no_examples_with_checklist,new_gold_test_only,0.05,0.605263,1.000000,0.754098,0.670290
109,anthropic_claude_haiku_4_5,codebook_with_examples_with_checklist,new_gold_test_only,0.05,0.605263,1.000000,0.754098,0.666304
145,google_gemini_3_1_flash_lite_low,codebook_with_examples_no_checklist,new_gold_test_only,0.20,0.697368,0.782609,0.757895,0.665217
169,google_gemini_3_1_flash_lite_low,codebook_with_examples_with_checklist,new_gold_test_only,0.15,0.675676,0.840909,0.755102,0.675000
157,google_gemini_3_1_flash_lite_low,codebook_no_examples_with_checklist,new_gold_test_only,0.15,0.653333,0.888889,0.754717,0.721481
133,google_gemini_3_1_flash_lite_low,codebook_no_examples_no_checklist,new_gold_test_only,0.15,0.666667,0.844444,0.752475,0.684444
121,google_gemini_3_1_flash_lite_low,direct_target_only,new_gold_test_only,0.05,0.592105,0.978261,0.743802,0.588043


In [82]:
# -----------------------------
# Controlled A/B contrasts
# -----------------------------

AB_COMPARISONS = [
    {
        "contrast": "examples_effect_without_checklist",
        "treatment": "codebook_with_examples_no_checklist",
        "control": "codebook_no_examples_no_checklist",
    },
    {
        "contrast": "examples_effect_with_checklist",
        "treatment": "codebook_with_examples_with_checklist",
        "control": "codebook_no_examples_with_checklist",
    },
    {
        "contrast": "checklist_effect_without_examples",
        "treatment": "codebook_no_examples_with_checklist",
        "control": "codebook_no_examples_no_checklist",
    },
    {
        "contrast": "checklist_effect_with_examples",
        "treatment": "codebook_with_examples_with_checklist",
        "control": "codebook_with_examples_no_checklist",
    },
    {
        "contrast": "codebook_no_examples_no_checklist_vs_direct",
        "treatment": "codebook_no_examples_no_checklist",
        "control": "direct_target_only",
    },
    {
        "contrast": "codebook_with_examples_no_checklist_vs_direct",
        "treatment": "codebook_with_examples_no_checklist",
        "control": "direct_target_only",
    },
    {
        "contrast": "codebook_no_examples_with_checklist_vs_direct",
        "treatment": "codebook_no_examples_with_checklist",
        "control": "direct_target_only",
    },
    {
        "contrast": "full_codebook_checklist_vs_direct",
        "treatment": "codebook_with_examples_with_checklist",
        "control": "direct_target_only",
    },
]

AB_METRIC_COLUMNS = [
    "accuracy",
    "recall",
    "precision",
    "f1",
    "specificity",
    "auroc",
    "prediction_coverage",
]

def build_ab_contrasts(
    metrics: pd.DataFrame,
    threshold_mode: str,
) -> pd.DataFrame:
    if metrics.empty:
        return pd.DataFrame()
    base = metrics.loc[metrics["analysis_level"].eq("model_overall")].copy()
    join_cols = (
        MODEL_ID_COLUMNS
        + ["evaluation_scope", "target_family"]
    )
    rows = []
    for spec in AB_COMPARISONS:
        treatment = base.loc[
            base["prompt_variant"].eq(spec["treatment"])
        ].copy()
        control = base.loc[
            base["prompt_variant"].eq(spec["control"])
        ].copy()
        merged = treatment.merge(
            control,
            on=join_cols,
            how="inner",
            suffixes=("_treatment", "_control"),
            validate="one_to_one",
        )
        for row in merged.to_dict("records"):
            output = {
                "contrast": spec["contrast"],
                "treatment": spec["treatment"],
                "control": spec["control"],
                "threshold_mode": threshold_mode,
                **{column: row[column] for column in join_cols},
                "treatment_threshold": row.get("threshold_treatment", np.nan),
                "control_threshold": row.get("threshold_control", np.nan),
            }
            for metric in AB_METRIC_COLUMNS:
                output[f"{metric}_treatment"] = row.get(f"{metric}_treatment")
                output[f"{metric}_control"] = row.get(f"{metric}_control")
                treatment_value = row.get(f"{metric}_treatment")
                control_value = row.get(f"{metric}_control")
                output[f"delta_{metric}"] = (
                    treatment_value - control_value
                    if pd.notna(treatment_value) and pd.notna(control_value)
                    else np.nan
                )
            rows.append(output)
    return pd.DataFrame(rows)

if RUN_EXTRACTION_STAGE:
    DEFAULT_AB_CONTRASTS_DF = build_ab_contrasts(
        DEFAULT_METRICS_DF,
        threshold_mode="default_0_5",
    )
    OPTIMIZED_AB_CONTRASTS_DF = build_ab_contrasts(
        OPTIMIZED_MODEL_METRICS_DF,
        threshold_mode="model_scope_optimized_in_sample",
    )
    AB_CONTRASTS_DF = pd.concat(
        [DEFAULT_AB_CONTRASTS_DF, OPTIMIZED_AB_CONTRASTS_DF],
        ignore_index=True,
    )
    if not AB_CONTRASTS_DF.empty:
        display(
            AB_CONTRASTS_DF.loc[
                AB_CONTRASTS_DF["evaluation_scope"].isin([
                    "new_gold_test_only",
                    "new_gold_test_only__drop_kyle_only",
                ]),
                [
                    "run_name", "evaluation_scope", "threshold_mode", "contrast",
                    "delta_accuracy", "delta_recall", "delta_f1", "delta_auroc"
                ],
            ]
        )

,run_name,evaluation_scope,threshold_mode,contrast,delta_accuracy,delta_recall,delta_f1,delta_auroc
8,anthropic_claude_haiku_4_5,new_gold_test_only,default_0_5,examples_effect_without_checklist,0.013158,0.065217,0.041911,0.013043
9,anthropic_claude_haiku_4_5,new_gold_test_only__drop_kyle_only,default_0_5,examples_effect_without_checklist,0.000000,0.048780,0.023691,0.009485
22,google_gemini_3_1_flash_lite_low,new_gold_test_only,default_0_5,examples_effect_without_checklist,-0.033684,-0.007246,-0.021645,-0.019227
23,google_gemini_3_1_flash_lite_low,new_gold_test_only__drop_kyle_only,default_0_5,examples_effect_without_checklist,-0.052678,-0.033537,-0.050575,-0.018451
36,openai_gpt_5_6_terra_low,new_gold_test_only,default_0_5,examples_effect_without_checklist,0.000000,0.000000,0.000000,-0.008333
...,...,...,...,...,...,...,...,...
639,anthropic_claude_haiku_4_5,new_gold_test_only__drop_kyle_only,model_scope_optimized_in_sample,full_codebook_checklist_vs_direct,0.029412,0.000000,0.014062,-0.014453
652,google_gemini_3_1_flash_lite_low,new_gold_test_only,model_scope_optimized_in_sample,full_codebook_checklist_vs_direct,0.083570,-0.137352,0.011300,0.086957
653,google_gemini_3_1_flash_lite_low,new_gold_test_only__drop_kyle_only,model_scope_optimized_in_sample,full_codebook_checklist_vs_direct,0.079768,-0.078174,0.024612,0.094677
666,openai_gpt_5_6_terra_low,new_gold_test_only,model_scope_optimized_in_sample,full_codebook_checklist_vs_direct,0.118421,0.130435,0.101826,0.127174


## 10. Models together: equal-weight and simple weighted ensembles

Only complete cases are used for fair model-weight comparisons. The weighted search requires at least two models to have nonzero weight and reports its in-sample nature explicitly.

In [83]:
# -----------------------------
# Ensemble data preparation
# -----------------------------

ENSEMBLE_REQUIRE_ALL_MODELS = True
ENSEMBLE_OPTIMIZATION_SCOPES = [
    "all_paragraphs_new_gold_unlabeled_no",
    "new_gold_test_only",
    "old_gold_test_only",
    "all_paragraphs_new_gold_unlabeled_no__drop_kyle_only",
    "new_gold_test_only__drop_kyle_only",
    "old_gold_test_only__drop_kyle_only",
]
ENSEMBLE_TOP_SEARCH_ROWS_PER_SCOPE = 25

def enabled_run_names() -> list[str]:
    return [
        config["run_name"]
        for config in MODEL_CONFIGS
        if config.get("enabled", True)
    ]

def build_ensemble_matrix(evaluated: pd.DataFrame) -> pd.DataFrame:
    if evaluated.empty:
        return pd.DataFrame()
    index_columns = [
        "target_row_id",
        "corpus_passage_id",
        "Passage_ID",
        "document",
        "evaluation_scope",
        "target_family",
        "target_labeler",
        "label_source_type",
        "y_true",
        "kyle_only_row",
        "prompt_variant",
    ]
    work = evaluated.copy()
    for column in ["Passage_ID", "target_labeler", "label_source_type"]:
        work[column] = work[column].fillna("").astype(str)
    key_columns = index_columns + ["run_name"]
    if work.duplicated(key_columns).any():
        raise AssertionError("Duplicate model probabilities exist for an ensemble target row.")
    matrix = (
        work.pivot(
            index=index_columns,
            columns="run_name",
            values="unlearning_probability",
        )
        .reset_index()
    )
    matrix.columns.name = None
    for run_name in enabled_run_names():
        if run_name not in matrix.columns:
            matrix[run_name] = np.nan
    matrix["models_available_n"] = matrix[enabled_run_names()].notna().sum(axis=1)
    matrix["complete_model_case"] = (
        matrix["models_available_n"].eq(len(enabled_run_names()))
        if ENSEMBLE_REQUIRE_ALL_MODELS
        else matrix["models_available_n"].ge(2)
    )
    return matrix

def build_ensemble_expected_grid(
    targets: pd.DataFrame,
) -> pd.DataFrame:
    prompts = PROMPT_VARIANTS[["prompt_variant"]].copy()
    targets = targets.copy()
    targets["_cross_key"] = 1
    prompts["_cross_key"] = 1
    return targets.merge(prompts, on="_cross_key").drop(columns="_cross_key")

ENSEMBLE_ANALYSIS_LEVELS = {
    "ensemble_overall": BASE_GROUP_COLUMNS + ["ensemble_name"],
    "ensemble_document": BASE_GROUP_COLUMNS + ["ensemble_name", "document"],
    "ensemble_labeler": BASE_GROUP_COLUMNS + ["ensemble_name", "target_labeler"],
    "ensemble_document_labeler": (
        BASE_GROUP_COLUMNS + ["ensemble_name", "document", "target_labeler"]
    ),
}

def evaluate_ensemble_levels(
    ensemble_rows: pd.DataFrame,
    ensemble_expected: pd.DataFrame,
) -> pd.DataFrame:
    if ensemble_rows.empty:
        return pd.DataFrame()
    frames = []
    expected = ensemble_expected.copy()
    for ensemble_name in ensemble_rows["ensemble_name"].unique():
        expected_named = expected.copy()
        expected_named["ensemble_name"] = ensemble_name
        subset = ensemble_rows.loc[
            ensemble_rows["ensemble_name"].eq(ensemble_name)
        ]
        for level, columns in ENSEMBLE_ANALYSIS_LEVELS.items():
            frames.append(
                evaluate_analysis_level(
                    evaluated=subset.rename(
                        columns={
                            "ensemble_probability": "unlearning_probability"
                        }
                    ),
                    expected=expected_named,
                    analysis_level=level,
                    group_columns=columns,
                    threshold_column="ensemble_threshold",
                )
            )
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def build_equal_weight_ensemble_rows(
    matrix: pd.DataFrame,
) -> pd.DataFrame:
    if matrix.empty:
        return pd.DataFrame()
    work = matrix.loc[matrix["complete_model_case"]].copy()
    model_columns = enabled_run_names()
    if ENSEMBLE_REQUIRE_ALL_MODELS:
        work["ensemble_probability"] = work[model_columns].mean(axis=1)
    else:
        work["ensemble_probability"] = work[model_columns].mean(
            axis=1, skipna=True
        )
    work["ensemble_threshold"] = DEFAULT_THRESHOLD
    work["ensemble_name"] = "equal_weight_models"
    work["ensemble_weights_json"] = json.dumps({
        model: 1 / len(model_columns) for model in model_columns
    })
    work["optimization_scope"] = ""
    work["optimization_method"] = "none_default_0_5"
    return work

if RUN_EXTRACTION_STAGE:
    ENSEMBLE_MATRIX_DF = build_ensemble_matrix(EVALUATED_PREDICTIONS_DF)
    ENSEMBLE_EXPECTED_GRID_DF = build_ensemble_expected_grid(
        EVALUATION_TARGETS_DF
    )
    EQUAL_ENSEMBLE_ROWS_DF = build_equal_weight_ensemble_rows(
        ENSEMBLE_MATRIX_DF
    )
    EQUAL_ENSEMBLE_METRICS_DF = evaluate_ensemble_levels(
        EQUAL_ENSEMBLE_ROWS_DF,
        ENSEMBLE_EXPECTED_GRID_DF,
    )

In [84]:
# -----------------------------
# Weighted ensemble optimization
# -----------------------------

def weight_vectors(model_names: Sequence[str], step: float) -> Iterator[dict[str, float]]:
    units = int(round(1.0 / step))
    n = len(model_names)

    def compositions(total: int, parts: int, prefix: tuple[int, ...] = ()):
        if parts == 1:
            yield prefix + (total,)
            return
        for value in range(total + 1):
            yield from compositions(total - value, parts - 1, prefix + (value,))

    for composition in compositions(units, n):
        weights = np.array(composition, dtype=float) / units
        if np.count_nonzero(weights) < 2:
            continue
        yield dict(zip(model_names, weights))

def vectorized_threshold_metrics(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    thresholds: np.ndarray,
) -> pd.DataFrame:
    pred = probabilities[:, None] >= thresholds[None, :]
    y = y_true.astype(bool)[:, None]
    tp = np.sum(pred & y, axis=0)
    fp = np.sum(pred & ~y, axis=0)
    fn = np.sum(~pred & y, axis=0)
    tn = np.sum(~pred & ~y, axis=0)
    accuracy = (tp + tn) / len(y_true)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) != 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision, dtype=float),
        where=(precision + recall) != 0,
    )
    specificity = np.divide(
        tn, tn + fp, out=np.zeros_like(tn, dtype=float), where=(tn + fp) != 0
    )
    auroc, auroc_flag = safe_auroc(y_true, probabilities)
    return pd.DataFrame({
        "threshold": thresholds,
        "accuracy": accuracy,
        "recall": recall,
        "precision": precision,
        "f1": f1,
        "specificity": specificity,
        "auroc": auroc,
        "auroc_flag": auroc_flag,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    })

def optimize_weighted_ensembles(
    matrix: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if matrix.empty:
        return pd.DataFrame(), pd.DataFrame()
    model_names = enabled_run_names()
    vectors = list(weight_vectors(model_names, ENSEMBLE_WEIGHT_STEP))
    best_rows = []
    top_audit_frames = []

    eligible = matrix.loc[
        matrix["complete_model_case"]
        & matrix["evaluation_scope"].isin(ENSEMBLE_OPTIMIZATION_SCOPES)
    ].copy()

    for (prompt_variant, scope, target_family), group in tqdm(
        eligible.groupby(
            ["prompt_variant", "evaluation_scope", "target_family"],
            sort=False,
        ),
        desc="Weighted ensemble search",
    ):
        y = group["y_true"].to_numpy(dtype=int)
        x = group[model_names].to_numpy(dtype=float)
        if len(group) == 0:
            continue

        candidate_rows = []
        for weights in vectors:
            weight_array = np.array([weights[name] for name in model_names])
            probabilities = x @ weight_array
            curve = vectorized_threshold_metrics(
                y, probabilities, THRESHOLD_GRID.astype(float)
            )
            curve["distance_from_0_5"] = (curve["threshold"] - 0.5).abs()
            curve = curve.sort_values(
                ["f1", "accuracy", "recall", "precision", "distance_from_0_5"],
                ascending=[False, False, False, False, True],
            )
            best_threshold = curve.iloc[0].to_dict()
            equal_distance = float(
                np.abs(weight_array - (1 / len(model_names))).sum()
            )
            candidate_rows.append({
                "prompt_variant": prompt_variant,
                "evaluation_scope": scope,
                "target_family": target_family,
                "n_complete_cases": len(group),
                "weights_json": json.dumps(weights, sort_keys=True),
                "equal_weight_l1_distance": equal_distance,
                **{f"weight_{name}": weights[name] for name in model_names},
                **best_threshold,
            })

        candidates = pd.DataFrame(candidate_rows).sort_values(
            [
                "f1", "accuracy", "recall", "precision",
                "distance_from_0_5", "equal_weight_l1_distance"
            ],
            ascending=[False, False, False, False, True, True],
        ).reset_index(drop=True)
        best = candidates.iloc[0].to_dict()
        best.update({
            "optimization_method": "in_sample_weight_and_threshold_grid",
            "weight_step": ENSEMBLE_WEIGHT_STEP,
        })
        best_rows.append(best)
        candidates.insert(0, "search_rank", np.arange(1, len(candidates) + 1))
        top_audit_frames.append(candidates.head(ENSEMBLE_TOP_SEARCH_ROWS_PER_SCOPE))

    return (
        pd.DataFrame(best_rows),
        pd.concat(top_audit_frames, ignore_index=True)
        if top_audit_frames else pd.DataFrame(),
    )

def apply_best_weighted_ensembles(
    matrix: pd.DataFrame,
    best_weights: pd.DataFrame,
) -> pd.DataFrame:
    if matrix.empty or best_weights.empty:
        return pd.DataFrame()
    model_names = enabled_run_names()
    rows = []
    for best in best_weights.to_dict("records"):
        group = matrix.loc[
            matrix["complete_model_case"]
            & matrix["prompt_variant"].eq(best["prompt_variant"])
            & matrix["evaluation_scope"].eq(best["evaluation_scope"])
            & matrix["target_family"].eq(best["target_family"])
        ].copy()
        weights = json.loads(best["weights_json"])
        group["ensemble_probability"] = sum(
            group[model] * float(weights[model])
            for model in model_names
        )
        group["ensemble_threshold"] = float(best["threshold"])
        group["ensemble_name"] = "weighted_models_optimized"
        group["ensemble_weights_json"] = best["weights_json"]
        group["optimization_scope"] = best["evaluation_scope"]
        group["optimization_method"] = best["optimization_method"]
        rows.append(group)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

if RUN_EXTRACTION_STAGE:
    BEST_ENSEMBLE_WEIGHTS_DF, ENSEMBLE_WEIGHT_SEARCH_AUDIT_DF = (
        optimize_weighted_ensembles(ENSEMBLE_MATRIX_DF)
    )
    WEIGHTED_ENSEMBLE_ROWS_DF = apply_best_weighted_ensembles(
        ENSEMBLE_MATRIX_DF,
        BEST_ENSEMBLE_WEIGHTS_DF,
    )
    WEIGHTED_ENSEMBLE_METRICS_DF = evaluate_ensemble_levels(
        WEIGHTED_ENSEMBLE_ROWS_DF,
        ENSEMBLE_EXPECTED_GRID_DF.loc[
            ENSEMBLE_EXPECTED_GRID_DF["evaluation_scope"].isin(
                ENSEMBLE_OPTIMIZATION_SCOPES
            )
        ],
    )
    ENSEMBLE_METRICS_DF = pd.concat(
        [EQUAL_ENSEMBLE_METRICS_DF, WEIGHTED_ENSEMBLE_METRICS_DF],
        ignore_index=True,
    )
    if not BEST_ENSEMBLE_WEIGHTS_DF.empty:
        display(BEST_ENSEMBLE_WEIGHTS_DF[[
            "prompt_variant", "evaluation_scope", "threshold",
            "accuracy", "recall", "f1", "auroc", "weights_json"
        ]])

Weighted ensemble search:   0%|          | 0/30 [00:00<?, ?it/s]

,prompt_variant,evaluation_scope,threshold,accuracy,recall,f1,auroc,weights_json
0,codebook_no_examples_no_checklist,all_paragraphs_new_gold_unlabeled_no,0.19,0.706667,0.800000,0.765957,0.695926,"{""anthropic_claude_haiku_4_5"": 0.05, ""google_gemini_3_1_flash_lite_low"": 0.9, ""openai_gpt_5_6_terra_low"": 0.05}"
1,codebook_no_examples_with_checklist,all_paragraphs_new_gold_unlabeled_no,0.12,0.706667,0.911111,0.788462,0.747037,"{""anthropic_claude_haiku_4_5"": 0.15, ""google_gemini_3_1_flash_lite_low"": 0.6, ""openai_gpt_5_6_terra_low"": 0.25}"
2,codebook_with_examples_no_checklist,all_paragraphs_new_gold_unlabeled_no,0.19,0.710526,0.804348,0.770833,0.678623,"{""anthropic_claude_haiku_4_5"": 0.15, ""google_gemini_3_1_flash_lite_low"": 0.85, ""openai_gpt_5_6_terra_low"": 0.0}"
3,codebook_with_examples_with_checklist,all_paragraphs_new_gold_unlabeled_no,0.10,0.702703,0.909091,0.784314,0.709091,"{""anthropic_claude_haiku_4_5"": 0.15, ""google_gemini_3_1_flash_lite_low"": 0.5, ""openai_gpt_5_6_terra_low"": 0.35}"
4,direct_target_only,all_paragraphs_new_gold_unlabeled_no,0.12,0.644737,0.956522,0.765217,0.676087,"{""anthropic_claude_haiku_4_5"": 0.75, ""google_gemini_3_1_flash_lite_low"": 0.0, ""openai_gpt_5_6_terra_low"": 0.25}"
5,codebook_no_examples_no_checklist,all_paragraphs_new_gold_unlabeled_no__drop_kyle_only,0.16,0.701493,0.900000,0.782609,0.717593,"{""anthropic_claude_haiku_4_5"": 0.45, ""google_gemini_3_1_flash_lite_low"": 0.45, ""openai_gpt_5_6_terra_low"": 0.1}"
6,codebook_no_examples_with_checklist,all_paragraphs_new_gold_unlabeled_no__drop_kyle_only,0.12,0.731343,0.975000,0.812500,0.789815,"{""anthropic_claude_haiku_4_5"": 0.15, ""google_gemini_3_1_flash_lite_low"": 0.6, ""openai_gpt_5_6_terra_low"": 0.25}"
7,codebook_with_examples_no_checklist,all_paragraphs_new_gold_unlabeled_no__drop_kyle_only,0.19,0.720588,0.853659,0.786517,0.697832,"{""anthropic_claude_haiku_4_5"": 0.15, ""google_gemini_3_1_flash_lite_low"": 0.85, ""openai_gpt_5_6_terra_low"": 0.0}"
8,codebook_with_examples_with_checklist,all_paragraphs_new_gold_unlabeled_no__drop_kyle_only,0.10,0.742424,0.974359,0.817204,0.737417,"{""anthropic_claude_haiku_4_5"": 0.15, ""google_gemini_3_1_flash_lite_low"": 0.5, ""openai_gpt_5_6_terra_low"": 0.35}"
9,direct_target_only,all_paragraphs_new_gold_unlabeled_no__drop_kyle_only,0.12,0.676471,1.000000,0.788462,0.704155,"{""anthropic_claude_haiku_4_5"": 0.75, ""google_gemini_3_1_flash_lite_low"": 0.0, ""openai_gpt_5_6_terra_low"": 0.25}"


## 11. Export final predictions, metrics, and provenance

The results workbook keeps default-threshold, optimized-threshold, ensemble, document, labeler, and combined analyses in separate sheets. Raw checkpoints remain the authoritative record of every request.

In [85]:
# -----------------------------
# Summary helpers
# -----------------------------

def macro_average_model_metrics(metrics: pd.DataFrame) -> pd.DataFrame:
    if metrics.empty:
        return pd.DataFrame()
    model = metrics.loc[metrics["analysis_level"].eq("model_overall")].copy()
    metric_columns = [
        "accuracy", "recall", "precision", "f1",
        "specificity", "auroc", "prediction_coverage"
    ]
    group_columns = [
        "prompt_variant",
        "evaluation_scope",
        "target_family",
    ]
    aggregations = {
        metric: (metric, "mean") for metric in metric_columns
    }
    macro = (
        model.groupby(group_columns, dropna=False)
        .agg(
            model_n=("run_name", "nunique"),
            **{f"macro_{name}": spec for name, spec in aggregations.items()},
        )
        .reset_index()
    )
    macro["analysis_level"] = "macro_average_across_models"
    return macro

def add_prediction_metadata(
    predictions: pd.DataFrame,
    corpus: pd.DataFrame,
) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame()
    metadata_columns = [
        "corpus_passage_id",
        "Passage_ID",
        "document",
        "source_file",
        "pdf_page_start",
        "pdf_page_end",
        "prediction_text",
        "matched_gold",
        "old_gold_unlearning",
        "old_gold_binary",
        "new_gold_unlearning",
        "new_gold_binary",
        "anmol_unlearning",
        "anmol_binary",
        "prerana_unlearning",
        "prerana_binary",
        "kyle_unlearning",
        "kyle_binary",
        "Original_Labeler",
        "kyle_only_row",
        "full_new_assumed_binary",
        "is_unlabeled_assumed_no",
        "extraction_flags",
        "source_line_ids",
    ]
    payload = corpus[metadata_columns].copy()
    duplicate_columns = [
        column for column in payload.columns
        if column in predictions.columns and column != "corpus_passage_id"
    ]
    payload = payload.drop(columns=duplicate_columns)
    return predictions.merge(
        payload,
        on="corpus_passage_id",
        how="left",
        validate="many_to_one",
    )

def results_table_dict() -> dict[str, pd.DataFrame]:
    tables = {
        "Prediction_Quality": CURRENT_PREDICTION_QUALITY_DF,
        "Predictions": PREDICTIONS_WITH_METADATA_DF,
        "Metrics_Default_All": DEFAULT_METRICS_DF,
        "Metrics_Default_Model": DEFAULT_METRICS_DF.loc[
            DEFAULT_METRICS_DF.get("analysis_level", pd.Series(dtype=str)).eq(
                "model_overall"
            )
        ] if not DEFAULT_METRICS_DF.empty else pd.DataFrame(),
        "Metrics_Optimized": OPTIMIZED_MODEL_METRICS_DF,
        "Best_Thresholds": BEST_THRESHOLDS_DF,
        "Threshold_Curves": THRESHOLD_CURVES_DF,
        "AB_Contrasts": AB_CONTRASTS_DF,
        "Ensemble_Metrics": ENSEMBLE_METRICS_DF,
        "Best_Ensemble_Weights": BEST_ENSEMBLE_WEIGHTS_DF,
        "Ensemble_Search_Top": ENSEMBLE_WEIGHT_SEARCH_AUDIT_DF,
        "Macro_Model_Metrics": MACRO_MODEL_METRICS_DF,
        "Evaluation_Targets": EVALUATION_TARGETS_DF,
        "Final_Corpus": FINAL_CORPUS_DF,
        "Prompt_Manifest": PROMPT_MANIFEST_DF,
        "Input_Manifest": INPUT_MANIFEST_DF,
        "Gold_Match_Audit": GOLD_MATCH_DF,
        "Page_Extraction_Audit": PAGE_AUDIT_DF,
        "Duplicate_Audit": DUPLICATE_REMOVAL_AUDIT_DF,
        "Example_Leakage": EXAMPLE_LEAKAGE_AUDIT_DF,
    }
    return tables

def export_results() -> dict[str, Path]:
    outputs: dict[str, Path] = {}

    predictions_csv = RESULTS_DIR / "all_model_predictions.csv"
    PREDICTIONS_WITH_METADATA_DF.to_csv(predictions_csv, index=False)
    outputs["predictions_csv"] = predictions_csv
    try:
        predictions_parquet = RESULTS_DIR / "all_model_predictions.parquet"
        PREDICTIONS_WITH_METADATA_DF.to_parquet(predictions_parquet, index=False)
        outputs["predictions_parquet"] = predictions_parquet
    except Exception as exc:
        print("Prediction Parquet export warning:", exc)

    csv_tables = {
        "metrics_default.csv": DEFAULT_METRICS_DF,
        "metrics_optimized_threshold.csv": OPTIMIZED_MODEL_METRICS_DF,
        "best_thresholds.csv": BEST_THRESHOLDS_DF,
        "threshold_curves.csv": THRESHOLD_CURVES_DF,
        "ab_contrasts.csv": AB_CONTRASTS_DF,
        "ensemble_metrics.csv": ENSEMBLE_METRICS_DF,
        "best_ensemble_weights.csv": BEST_ENSEMBLE_WEIGHTS_DF,
        "prediction_quality.csv": CURRENT_PREDICTION_QUALITY_DF,
    }
    for filename, table in csv_tables.items():
        path = RESULTS_DIR / filename
        table.to_csv(path, index=False)
        outputs[filename] = path

    workbook_path = RESULTS_DIR / "Unlearning_Final_Evaluation_Results.xlsx"
    with pd.ExcelWriter(workbook_path, engine="xlsxwriter") as writer:
        for name, table in results_table_dict().items():
            safe_name = re.sub(r"[\[\]:*?/\\]", "_", name)[:31]
            frame = table if table is not None else pd.DataFrame()
            # Excel has a hard row limit; keep the full threshold curve in CSV if needed.
            if len(frame) > 1_048_000:
                frame = frame.head(1_048_000).copy()
                frame["excel_export_note"] = "Truncated; use CSV for complete table."
            frame.to_excel(writer, index=False, sheet_name=safe_name)
            auto_width_worksheet(writer.sheets[safe_name], frame)
    outputs["results_workbook"] = workbook_path

    manifest = {
        "created_utc": utc_now_iso(),
        "workbook_path": str(WORKBOOK_PATH),
        "workbook_sha256": sha256_file(WORKBOOK_PATH),
        "pdf_inputs": INPUT_MANIFEST_DF.to_dict("records"),
        "dataset_sha256": DATASET_SHA256,
        "output_schema_sha256": OUTPUT_SCHEMA_SHA256,
        "gold_rows": int(len(GOLD_DF)),
        "final_corpus_rows": int(len(FINAL_CORPUS_DF)),
        "matched_gold_rows": int(FINAL_CORPUS_DF["matched_gold"].sum()),
        "assumed_negative_rows": int(
            FINAL_CORPUS_DF["is_unlabeled_assumed_no"].sum()
        ),
        "model_configs": [
            {k: v for k, v in config.items() if k != "api_key_env"}
            for config in MODEL_CONFIGS
        ],
        "prompt_variants": PROMPT_VARIANTS.to_dict("records"),
        "threshold_grid": {
            "minimum": float(THRESHOLD_GRID.min()),
            "maximum": float(THRESHOLD_GRID.max()),
            "step": 0.01,
        },
        "ensemble_weight_step": ENSEMBLE_WEIGHT_STEP,
        "methodological_notes": [
            "Full-corpus unmatched passages are assumed negative and reported separately from human gold.",
            "Threshold and weight optimization are in-sample exploratory.",
            "AUROC is threshold-independent.",
            "Invalid/missing provider responses are excluded and reported through coverage.",
            "Kyle-only removal excludes only rows where Kyle is the sole human labeler.",
        ],
        "outputs": {key: str(value) for key, value in outputs.items()},
    }
    manifest_path = RESULTS_DIR / "run_manifest.json"
    manifest_path.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    outputs["run_manifest"] = manifest_path
    return outputs

if RUN_EXTRACTION_STAGE:
    PREDICTIONS_WITH_METADATA_DF = add_prediction_metadata(
        CURRENT_PREDICTIONS_DF,
        FINAL_CORPUS_DF,
    )
    MACRO_MODEL_METRICS_DF = macro_average_model_metrics(
        DEFAULT_METRICS_DF
    )
    EXPORTED_OUTPUTS = export_results()
    print("Exported:")
    for name, path in EXPORTED_OUTPUTS.items():
        print(f" - {name}: {path}")

NameError: name 'PROMPT_MANIFEST_DF' is not defined

## Interpretation cautions

- **Full-corpus assumed-negative metrics** answer a different question from test-set metrics. They are sensitive to extraction decisions and to the assumption that every unmatched paragraph is `No`.
- **New gold** is the primary finalized reference. **Old gold** is preserved for historical comparison.
- **Labeler metrics** should only use rows that the corresponding person actually labeled; blanks are never converted to `No`.
- **Kyle-only exclusion** removes a row only when Kyle has a label and both Anmol and Prerana are blank. Shared rows remain.
- **Threshold tuning** can change accuracy, recall, precision, specificity, and F1, but not AUROC.
- **Weighted ensembles** and optimized thresholds are exploratory because the same labeled data are used to select and evaluate them. Report default 0.50 results alongside them.

## 12. Non-API self-tests

These tests exercise the finalized workbook invariants, normalization, malformed provider output repair, metric edge cases, parent/child detection, prompt-factor isolation, and a synthetic multi-page PDF with repeated headers, two columns, and a reference section.

In [ ]:
RUN_SELF_TESTS = True

def recursively_find_keys(value: Any) -> set[str]:
    keys: set[str] = set()
    if isinstance(value, dict):
        for key, child in value.items():
            keys.add(key)
            keys |= recursively_find_keys(child)
    elif isinstance(value, list):
        for child in value:
            keys |= recursively_find_keys(child)
    return keys

def create_synthetic_pdf(path: Path) -> Path:
    doc = fitz.open()
    page_rect = fitz.paper_rect("letter")

    for page_index in range(3):
        page = doc.new_page(width=page_rect.width, height=page_rect.height)
        page.insert_text((72, 25), "SYNTHETIC REPEATED HEADER", fontsize=9)
        page.insert_text(
            (page_rect.width / 2 - 5, page_rect.height - 20),
            str(page_index + 1),
            fontsize=9,
        )

        if page_index == 0:
            page.insert_textbox(
                fitz.Rect(72, 80, 540, 165),
                (
                    "This first paragraph explains an established policy and why it failed. "
                    "It ends normally.\n\n"
                    "The agency therefore replaced the obsolete policy with a fundamentally "
                    "different governance arrangement."
                ),
                fontsize=10,
            )
        elif page_index == 1:
            page.insert_textbox(
                fitz.Rect(72, 65, 540, 90),
                "TWO COLUMN TEST SECTION",
                fontsize=11,
            )
            for i in range(6):
                page.insert_textbox(
                    fitz.Rect(72, 105 + i * 45, 285, 137 + i * 45),
                    f"Left column paragraph {i+1} contains enough prose for extraction testing.",
                    fontsize=9,
                )
                page.insert_textbox(
                    fitz.Rect(325, 105 + i * 45, 540, 137 + i * 45),
                    f"Right column paragraph {i+1} contains enough prose for extraction testing.",
                    fontsize=9,
                )
        else:
            page.insert_textbox(
                fitz.Rect(72, 80, 540, 105),
                "References",
                fontsize=12,
            )
            page.insert_textbox(
                fitz.Rect(72, 120, 540, 165),
                (
                    "Smith, J. 2020. A Synthetic Reference Entry. Test Journal 1(1): 1-10.\n\n"
                    "Jones, A. 2021. Another Synthetic Citation. Test Press."
                ),
                fontsize=9,
            )

    path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(path)
    doc.close()
    return path

def run_self_tests() -> pd.DataFrame:
    results: list[dict[str, Any]] = []

    def check(name: str, condition: bool, detail: str = "") -> None:
        if not condition:
            raise AssertionError(f"Self-test failed: {name}. {detail}")
        results.append({"test": name, "status": "PASS", "detail": detail})

    # Finalized workbook invariants
    test_gold, test_codebook = load_finalized_workbook(WORKBOOK_PATH)
    check("finalized_gold_row_count", len(test_gold) == 76)
    check("finalized_gold_passage_ids_unique", test_gold["Passage_ID"].is_unique)
    priority_expected = test_gold.apply(
        lambda row: (
            row["anmol_binary"]
            if pd.notna(row["anmol_binary"])
            else row["prerana_binary"]
            if pd.notna(row["prerana_binary"])
            else row["kyle_binary"]
            if pd.notna(row["kyle_binary"])
            else 0
        ),
        axis=1,
    ).astype(int)
    check(
        "new_gold_priority_rule",
        (priority_expected == test_gold["new_gold_binary"].astype(int)).all(),
        "Priority must be anmol -> prerana -> kyle; rows with no coder default to No.",
    )
    check(
        "kyle_only_formula",
        (
            test_gold["kyle_only_row"]
            == (
                test_gold["kyle_binary"].notna()
                & test_gold["anmol_binary"].isna()
                & test_gold["prerana_binary"].isna()
            )
        ).all(),
    )

    # Normalization and parser repair
    check(
        "unicode_normalization",
        normalize_for_match("Th e offi ce") == normalize_for_match("The office"),
    )
    parsed, diagnostics = parse_and_validate_model_output(
        {
            "unlearning_probabilities": {"Yes": "82%"},
            "label": "yes",
            "target": "laws, plans and policies",
            "evidence": "replaced the obsolete policy",
            "reason": "The prior policy is explicitly replaced.",
        },
        "The agency replaced the obsolete policy.",
    )
    check("probability_percent_and_nested_repair", abs(parsed.unlearning_probability - 0.82) < 1e-12)
    check("label_alias_repair", parsed.unlearning_label == "Yes")
    check("target_alias_repair", parsed.target_category == "laws_plans_policies")
    check("evidence_quote_validation", diagnostics["evidence_quote_valid"] is True)

    # Current checkpoint filtering: stale prompt/schema/model runs must never be evaluated.
    synthetic_corpus_for_keys = pd.DataFrame([{
        "corpus_passage_id": "SYNTH-CURRENT-001",
        "document": "EPA",
        "prediction_text": "The agency replaced the failed prior procedure.",
        "text_sha256": sha256_text(
            normalize_for_match("The agency replaced the failed prior procedure.")
        ),
    }])
    synthetic_dataset_hash = corpus_dataset_hash(synthetic_corpus_for_keys)
    current_config = next(c for c in MODEL_CONFIGS if c.get("enabled", True))
    current_variant = str(PROMPT_VARIANTS.iloc[0]["prompt_variant"])
    expected_key = make_run_key(
        synthetic_corpus_for_keys.iloc[0].to_dict(),
        current_config,
        current_variant,
        synthetic_dataset_hash,
    )
    fake_checkpoints = pd.DataFrame([
        {
            "run_key": expected_key,
            "timestamp_utc": "2026-08-14T00:00:00+00:00",
            "dataset_sha256": synthetic_dataset_hash,
            "corpus_passage_id": "SYNTH-CURRENT-001",
            "run_name": current_config["run_name"],
            "prompt_variant": current_variant,
            "status": "ok",
        },
        {
            "run_key": "stale-run-key",
            "timestamp_utc": "2026-08-14T00:01:00+00:00",
            "dataset_sha256": synthetic_dataset_hash,
            "corpus_passage_id": "SYNTH-CURRENT-001",
            "run_name": current_config["run_name"],
            "prompt_variant": current_variant,
            "status": "ok",
        },
    ])
    filtered_fake = filter_current_predictions(
        fake_checkpoints, synthetic_corpus_for_keys
    )
    check(
        "stale_checkpoint_excluded",
        len(filtered_fake) == 1 and filtered_fake.iloc[0]["run_key"] == expected_key,
    )

    # Metrics and AUROC edge case
    metrics = calculate_binary_metrics(
        [0, 0, 1, 1],
        [0.1, 0.4, 0.6, 0.9],
        0.5,
    )
    check("binary_metrics_perfect_accuracy", metrics["accuracy"] == 1.0)
    check("binary_metrics_perfect_f1", metrics["f1"] == 1.0)
    auroc, flag = safe_auroc([0, 0], [0.1, 0.2])
    check("single_class_auroc_is_nan", np.isnan(auroc) and flag == "single_reference_class")

    # Schema must avoid the constraints that previously broke Anthropic.
    schema_keys = recursively_find_keys(OUTPUT_SCHEMA)
    check(
        "anthropic_numeric_constraints_removed",
        not bool(schema_keys & UNSUPPORTED_SCHEMA_KEYS),
        f"Unexpected keys: {schema_keys & UNSUPPORTED_SCHEMA_KEYS}",
    )

    # Parent-child duplicate detection
    parent_text = (
        "The agency rejected its obsolete emergency plan and replaced it with a "
        "fundamentally different coordinated response system."
    )
    child_text = "The agency rejected its obsolete emergency plan"
    duplicate_test = pd.DataFrame([
        {
            "pre_dedup_corpus_id": "A",
            "document": "Synthetic",
            "analysis_eligible": True,
            "text_compact": compact_for_match(child_text),
        },
        {
            "pre_dedup_corpus_id": "B",
            "document": "Synthetic",
            "analysis_eligible": True,
            "text_compact": compact_for_match(parent_text),
        },
    ])
    old_min = PARENT_CHILD_MIN_SHORT_CHARS
    globals()["PARENT_CHILD_MIN_SHORT_CHARS"] = 25
    try:
        pairs = find_parent_child_pairs(duplicate_test)
    finally:
        globals()["PARENT_CHILD_MIN_SHORT_CHARS"] = old_min
    check("parent_child_duplicate_detection", len(pairs) == 1)

    # Prompt factor isolation
    prompt_test_corpus = pd.DataFrame({
        "prediction_text": ["The agency replaced an obsolete routine."]
    })
    assert_prompt_factor_parity(prompt_test_corpus)
    check("prompt_factor_parity", True)

    # Synthetic PDF extraction
    synthetic_path = OUTPUT_ROOT / "_self_test" / "synthetic_layout_test.pdf"
    create_synthetic_pdf(synthetic_path)
    synthetic = extract_document("Synthetic", synthetic_path)
    removed_text = " ".join(
        synthetic["removed_lines"].get("raw_text", pd.Series(dtype=str)).astype(str)
    )
    check(
        "repeated_header_removed",
        "SYNTHETIC REPEATED HEADER" in removed_text,
    )
    page_two = synthetic["page_audit"].loc[
        synthetic["page_audit"]["pdf_page_number"].eq(2)
    ]
    check(
        "two_column_page_detected",
        not page_two.empty and bool(page_two.iloc[0]["two_column_detected"]),
    )
    reference_rows = synthetic["paragraphs"].loc[
        synthetic["paragraphs"]["exclusion_reason"].str.contains(
            "reference_section", na=False
        )
    ]
    check("reference_section_excluded", len(reference_rows) >= 1)
    check(
        "synthetic_paragraphs_have_source_coordinates",
        synthetic["paragraphs"]["source_line_ids"].fillna("").ne("").all(),
    )

    return pd.DataFrame(results)

if RUN_SELF_TESTS:
    SELF_TEST_RESULTS_DF = run_self_tests()
    display(SELF_TEST_RESULTS_DF)
    SELF_TEST_RESULTS_DF.to_csv(
        AUDIT_DIR / "self_test_results.csv", index=False
    )

## Final pre-run checklist

- [ ] The input manifest lists exactly the intended clean PDFs.
- [ ] No page is flagged `low_text_page_ocr_recommended` without manual review.
- [ ] All 76 gold passages are matched exactly once.
- [ ] The gold-match hard-stop reports zero unmatched and zero ambiguous rows.
- [ ] Exact and parent/child duplicate checks pass.
- [ ] No finalized gold row is removed as a codebook example.
- [ ] Fuzzy example similarities and near-duplicate pairs have been manually reviewed.
- [ ] Prompt hashes and factor settings in `prompt_manifest.csv` are correct.
- [ ] Provider preflight has 100% schema-valid responses for all 15 model/prompt configurations.
- [ ] Full-run prediction coverage is reviewed before interpreting metrics.
- [ ] New gold is treated as primary; old gold and assumed-negative full-corpus results remain clearly labeled.
- [ ] Threshold and weight-optimized results are reported alongside default 0.50 results and identified as exploratory.